# Top 150 SQL Interview Questions & Answers (FAANG / Tier-1 / FinTech / Quant)

> **The Definitive Production & Architectural SQL Guide for Modern Data Scientists, Data Engineers, Analytics Engineers, and Quant Researchers.**

---

### Seniority Tier Breakdown:
* **Junior / Associate Tier (Q1 – Q50):** SQL Fundamentals, Execution Order, Join Mechanics, NULL Semantics, Aggregations, Grouping Sets, Strings, Regex, Date/Time & Subqueries / CTEs.
* **Mid-Level / Senior Tier (Q51 – Q100):** Window Functions (Ranking, Offsets, Dynamic Framing), Gaps & Islands, Sessionization, Semi-Structured JSON/Arrays, Deduplication, UPSERT/Merge, and CDC Patterns.
* **Staff / Lead / Architect / Quant Tier (Q101 – Q150):** Query Optimization & EXPLAIN Plans, Indexing Mechanics, Kimball vs OBT Modeling, Distributed Partitioning & Skew Salting, Concurrency & MVCC / ACID Locking, and Cloud DWH Architecture (DuckDB, Snowflake, BigQuery, Iceberg).

---


In [1]:
# Setup in-memory DuckDB relational engine with Native SQL Studio %%sql Execution
import duckdb
import pandas as pd
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

# Shared in-memory connection
conn = duckdb.connect(':memory:')

def _execute_raw_sql(query):
    query = query.strip()
    statements = [s.strip() for s in query.split(';') if s.strip()]
    last_res = None
    for stmt in statements:
        last_res = conn.execute(stmt)
    try:
        return last_res.df()
    except Exception:
        return "Query Executed Successfully."

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run %%sql queries directly.")


SQL Studio Environment Active! You can now write and run %%sql queries directly.


## =========================================================================
## SENIORITY LEVEL: JUNIOR / ASSOCIATE DATA SCIENTIST / DATA ANALYST
## =========================================================================

### -------------------------------------------------------------------------
### PILLAR 1: SQL FUNDAMENTALS, EXECUTION ORDER & SET OPERATIONS
### -------------------------------------------------------------------------

### Q001: Logical Query Execution Order vs Syntactic Order

#### Verbal Answer:
* **Syntactic vs Logical Execution:** In SQL, clauses are written syntactically as `SELECT`, `FROM`, `WHERE`, `GROUP BY`, `HAVING`, `WINDOW`, `QUALIFY`, `ORDER BY`, `LIMIT`. However, the relational database engine executes them logically in a fundamentally distinct order.

* **Exact Logical Order:**
  1. `FROM` & `JOIN` (determine and cross/join the candidate row sets).
  2. `WHERE` (filter row-level predicates).
  3. `GROUP BY` (aggregate rows into distinct grouping keys).
  4. `HAVING` (filter aggregate group states).
  5. `WINDOW` / `QUALIFY` (evaluate analytical window functions and filter window partitions).
  6. `SELECT` (evaluate expressions, scalar functions, and assign aliases).
  7. `DISTINCT` (eliminate duplicate projected tuples).
  8. `UNION` / `INTERSECT` / `EXCEPT` (apply relational set operations).
  9. `ORDER BY` (sort output tuples).
  10. `LIMIT` / `OFFSET` (slice top/bottom rows).

* **Interview Impact:** Explaining this logical pipeline immediately explains why column aliases defined in `SELECT` cannot be referenced in `WHERE` or `GROUP BY` in ANSI SQL, but can be referenced in `ORDER BY`.

#### Tricky Follow-up:
* **Question:** Why does `WHERE status = 'ACTIVE'` fail when referencing a calculated alias defined inside `SELECT (salary * 1.2) AS new_salary`?

* **Answer:** Because the `WHERE` clause is evaluated at Step 2 to filter source rows before the `SELECT` clause is executed at Step 6. At the time `WHERE` executes, `new_salary` has not yet been computed or registered in the scope.


In [2]:
%%sql
CREATE TABLE employees (
    emp_id INT,
    name VARCHAR,
    department VARCHAR,
    salary DOUBLE
);
INSERT INTO employees VALUES
(1, 'Alice', 'Engineering', 120000),
(2, 'Bob', 'Engineering', 95000),
(3, 'Charlie', 'Marketing', 80000),
(4, 'Diana', 'Marketing', 110000),
(5, 'Evan', 'Sales', 75000);

SELECT 
    department,
    COUNT(emp_id) AS head_count,
    ROUND(AVG(salary), 2) AS avg_dept_salary
FROM employees
WHERE salary >= 80000
GROUP BY department
HAVING COUNT(emp_id) > 1
ORDER BY avg_dept_salary DESC;

department,head_count,avg_dept_salary
Engineering,2,107500.0
Marketing,2,95000.0


### Q002: WHERE vs HAVING: Conceptual and Engine-Level Distinctions

#### Verbal Answer:
* **Row-Level vs Group-Level Filtering:** `WHERE` operates on individual rows before aggregation occurs, while `HAVING` filters aggregated groups after the `GROUP BY` operation has materialized bucket states.

* **Index Utilization:** Predicates inside `WHERE` can leverage B-Tree indexes or zone maps / min-max filters to prune disk blocks and reduce row scans. Predicates in `HAVING` evaluate post-aggregation summaries and cannot prune raw disk pages.

* **Performance Best Practice:** Always push row-level criteria into `WHERE` as early as possible (predicate pushdown) to minimize the number of rows passed to the hash aggregation buffer.

#### Tricky Follow-up:
* **Question:** Can you use `HAVING` without a `GROUP BY` clause?

* **Answer:** Yes. When `HAVING` is used without `GROUP BY`, the entire table is treated as a single aggregate group (e.g., `SELECT AVG(salary) FROM emp HAVING AVG(salary) > 50000`). If the condition evaluates to false, an empty result set is returned.


In [3]:
%%sql
CREATE TABLE transactions (
    tx_id INT,
    user_id INT,
    amount DOUBLE,
    category VARCHAR
);
INSERT INTO transactions VALUES
(1, 101, 50.0, 'Retail'),
(2, 101, 150.0, 'Retail'),
(3, 101, 20.0, 'Fee'),
(4, 102, 300.0, 'Retail'),
(5, 102, 400.0, 'Retail'),
(6, 103, 25.0, 'Retail');

SELECT 
    user_id,
    SUM(amount) AS total_retail_spend,
    COUNT(tx_id) AS tx_count
FROM transactions
WHERE category = 'Retail'
GROUP BY user_id
HAVING SUM(amount) > 150
ORDER BY total_retail_spend DESC;

user_id,total_retail_spend,tx_count
102,700.0,2
101,200.0,2


### Q003: UNION vs UNION ALL: Performance & Duplicate Resolution Mechanics

#### Verbal Answer:
* **Deduplication Overhead:** `UNION` performs a set union that eliminates all duplicate rows across both queries. To do this, the database engine must execute a heavy sorting or hash-based distinct pass across all rows.

* **`UNION ALL` Pass-Through:** `UNION ALL` simply concatenates the two result streams directly without checking for duplicates, consuming zero memory for hash tables and requiring no sorting.

* **Production Rule of Thumb:** Always default to `UNION ALL` unless business logic strictly mandates deduplication and upstream tables are known to contain overlapping keys.

#### Tricky Follow-up:
* **Question:** If two tables are disjoint (e.g., table_2023 and table_2024 with no shared dates), which should you use and why?

* **Answer:** Always use `UNION ALL`. Since the partitions are mutually exclusive by definition, `UNION` introduces a massive, useless hash-deduplication sort over millions of rows that will never find a duplicate.


In [4]:
%%sql
CREATE TABLE q1_leads (email VARCHAR, country VARCHAR);
CREATE TABLE q2_leads (email VARCHAR, country VARCHAR);

INSERT INTO q1_leads VALUES ('a@co.com', 'US'), ('b@co.com', 'UK'), ('c@co.com', 'IN');
INSERT INTO q2_leads VALUES ('b@co.com', 'UK'), ('d@co.com', 'US'), ('e@co.com', 'DE');

SELECT email, country FROM q1_leads
UNION
SELECT email, country FROM q2_leads
ORDER BY email;

SELECT email, country FROM q1_leads
UNION ALL
SELECT email, country FROM q2_leads
ORDER BY email;

email,country
a@co.com,US
b@co.com,UK
b@co.com,UK
c@co.com,IN
d@co.com,US
e@co.com,DE


### Q004: INTERSECT and EXCEPT / MINUS Set Operations

#### Verbal Answer:
* **`INTERSECT`:** Returns only the distinct rows that appear in *both* the left and right query result sets.

* **`EXCEPT` (or `MINUS` in Oracle):** Returns the distinct rows from the left query that do *not* exist in the right query result set.

* **Schema Compatibility:** All set operations (`UNION`, `INTERSECT`, `EXCEPT`) require that both queries have the identical number of columns and compatible data types in corresponding positions. Column names are derived strictly from the first query.

#### Tricky Follow-up:
* **Question:** How does `EXCEPT ALL` differ from `EXCEPT` in engines that support multiset semantics (like PostgreSQL / DuckDB)?

* **Answer:** `EXCEPT` deduplicates rows completely. `EXCEPT ALL` accounts for row multiplicity: if Row X appears 3 times in Query A and 1 time in Query B, `EXCEPT ALL` outputs Row X exactly 2 times (`3 - 1 = 2`).


In [5]:
%%sql
CREATE TABLE beta_testers (user_id INT, plan VARCHAR);
CREATE TABLE paid_subscribers (user_id INT, plan VARCHAR);

INSERT INTO beta_testers VALUES (1, 'Pro'), (2, 'Free'), (3, 'Pro'), (4, 'Enterprise');
INSERT INTO paid_subscribers VALUES (1, 'Pro'), (3, 'Pro'), (5, 'Enterprise');

SELECT user_id, plan FROM beta_testers
INTERSECT
SELECT user_id, plan FROM paid_subscribers;

SELECT user_id, plan FROM beta_testers
EXCEPT
SELECT user_id, plan FROM paid_subscribers;

user_id,plan
2,Free
4,Enterprise


### Q005: CASE WHEN Expressions: Searched vs Simple and Short-Circuit Evaluation

#### Verbal Answer:
* **Searched vs Simple Syntax:**
  - *Simple CASE:* `CASE column WHEN val1 THEN res1 ... END` (equality comparison only).
  - *Searched CASE:* `CASE WHEN condition1 THEN res1 WHEN condition2 THEN res2 ... END` (supports arbitrary boolean expressions, ranges, and `AND`/`OR` combinations).

* **Short-Circuit Semantics:** SQL standard engines evaluate `WHEN` branches in strict top-to-bottom sequence and immediately exit upon finding the first `TRUE` condition.

* **Default Value:** If no `WHEN` branch matches and an `ELSE` clause is omitted, the expression implicitly evaluates to `NULL`.

#### Tricky Follow-up:
* **Question:** Can a `CASE WHEN` protect against division by zero errors in the same query?

* **Answer:** Yes. `CASE WHEN denominator = 0 THEN 0.0 ELSE numerator / denominator END` guarantees that the division branch is never executed when `denominator = 0` due to short-circuiting in standard conforming optimizers (or alternatively `NULLIF(denominator, 0)`).


In [6]:
%%sql
CREATE TABLE tier_data (
    account_id INT,
    revenue DOUBLE,
    churn_flag BOOLEAN
);
INSERT INTO tier_data VALUES
(101, 150000.0, FALSE),
(102, 45000.0, FALSE),
(103, 5000.0, TRUE),
(104, 0.0, FALSE);

SELECT 
    account_id,
    revenue,
    churn_flag,
    CASE 
        WHEN churn_flag = TRUE THEN 'Churned Risk'
        WHEN revenue >= 100000 THEN 'Tier 1 Enterprise'
        WHEN revenue >= 25000 THEN 'Tier 2 Mid-Market'
        ELSE 'Tier 3 SMB'
    END AS customer_segment,
    CASE 
        WHEN revenue > 0 THEN ROUND(10000.0 / revenue, 4)
        ELSE NULL
    END AS safe_unit_ratio
FROM tier_data;

account_id,revenue,churn_flag,customer_segment,safe_unit_ratio
101,150000.0,False,Tier 1 Enterprise,0.0667
102,45000.0,False,Tier 2 Mid-Market,0.2222
103,5000.0,True,Churned Risk,2.0000
104,0.0,False,Tier 3 SMB,NaN


### Q006: COUNT(*), COUNT(1), COUNT(column), and COUNT(DISTINCT column)

#### Verbal Answer:
* **`COUNT(*)` and `COUNT(1)`:** Count every single candidate row produced by the query, including rows where every single column is `NULL`. In modern query optimizers, `COUNT(*)` and `COUNT(1)` compile to the exact same execution plan.

* **`COUNT(column)`:** Counts only rows where the specified column is **NOT NULL**. `NULL` values are silently ignored.

* **`COUNT(DISTINCT column)`:** Computes the cardinality of non-null unique values in the column. Highly CPU and memory intensive on big data because it prevents streaming partial aggregation across distributed nodes.

#### Tricky Follow-up:
* **Question:** What does `COUNT(DISTINCT NULL)` return?

* **Answer:** It returns `0`. Aggregate functions (except `COUNT(*)`) ignore `NULL` values entirely.


In [7]:
%%sql
CREATE TABLE sample_counts (id INT, val VARCHAR);
INSERT INTO sample_counts VALUES
(1, 'Alpha'),
(2, 'Beta'),
(3, 'Alpha'),
(4, NULL),
(5, NULL);

SELECT 
    COUNT(*) AS cnt_star,
    COUNT(1) AS cnt_one,
    COUNT(val) AS cnt_column,
    COUNT(DISTINCT val) AS cnt_distinct
FROM sample_counts;

cnt_star,cnt_one,cnt_column,cnt_distinct
5,5,3,2


### Q007: COALESCE vs IFNULL vs NVL vs ISNULL Across SQL Dialects

#### Verbal Answer:
* **`COALESCE(v1, v2, ..., vn)`:** ANSI SQL standard function that accepts N arguments and returns the very first non-null expression from left to right. Supported across all database engines.

* **Dialect-Specific 2-Argument Functions:**
  - `IFNULL(a, b)`: MySQL, SQLite.
  - `NVL(a, b)`: Oracle, Snowflake.
  - `ISNULL(a, b)`: Microsoft SQL Server.

* **Short-Circuiting Advantage:** `COALESCE` short-circuits. If the first argument is not null, subsequent expressions (including subqueries) are never evaluated.

#### Tricky Follow-up:
* **Question:** What happens if all arguments to `COALESCE(a, b, c)` evaluate to `NULL`?

* **Answer:** The function returns `NULL`. To prevent this, always supply a literal non-null fallback value as the final argument (e.g., `COALESCE(a, b, 0)`).


In [8]:
%%sql
CREATE TABLE contact_info (
    user_id INT,
    work_phone VARCHAR,
    cell_phone VARCHAR,
    home_phone VARCHAR
);
INSERT INTO contact_info VALUES
(1, NULL, '555-0199', '555-0100'),
(2, NULL, NULL, '555-0144'),
(3, '555-0122', '555-0133', NULL),
(4, NULL, NULL, NULL);

SELECT 
    user_id,
    COALESCE(work_phone, cell_phone, home_phone, 'UNAVAILABLE') AS primary_phone
FROM contact_info;

user_id,primary_phone
1,555-0199
2,555-0144
3,555-0122
4,UNAVAILABLE


### Q008: NULLIF: Syntax, Mechanics, and Preventing Zero-Division

#### Verbal Answer:
* **Function Definition:** `NULLIF(expr1, expr2)` compares two expressions. If `expr1 = expr2`, it returns `NULL`. Otherwise, it returns `expr1`.

* **Mathematical Equivalent:** `CASE WHEN expr1 = expr2 THEN NULL ELSE expr1 END`.

* **Primary Production Use Case:** Preventing division by zero errors cleanly. Wrapping the divisor in `NULLIF(divisor, 0)` transforms zero into `NULL`, causing `numerator / NULL` to safely return `NULL` instead of terminating the ETL pipeline with an exception.

#### Tricky Follow-up:
* **Question:** How can you combine `COALESCE` and `NULLIF` to compute a safe percentage that defaults to 0% on 0 impressions?

* **Answer:** `COALESCE(clicks * 1.0 / NULLIF(impressions, 0), 0.0)`. If `impressions = 0`, `NULLIF` yields `NULL`, division yields `NULL`, and `COALESCE` converts it cleanly to `0.0`.


In [9]:
%%sql
CREATE TABLE ad_campaigns (
    campaign_id VARCHAR,
    impressions INT,
    clicks INT
);
INSERT INTO ad_campaigns VALUES
('Camp_A', 10000, 250),
('Camp_B', 0, 0),
('Camp_C', 5000, 120);

SELECT 
    campaign_id,
    impressions,
    clicks,
    -- Safe division without throwing runtime error
    COALESCE(ROUND(clicks * 100.0 / NULLIF(impressions, 0), 2), 0.0) AS ctr_percentage
FROM ad_campaigns;

campaign_id,impressions,clicks,ctr_percentage
Camp_A,10000,250,2.5
Camp_B,0,0,0.0
Camp_C,5000,120,2.4


### Q009: DISTINCT vs GROUP BY: Engine Execution & Deduplication

#### Verbal Answer:
* **Conceptual Purpose:** `SELECT DISTINCT a, b` eliminates duplicate output tuples from the final projection. `GROUP BY a, b` defines partition buckets for aggregate calculations (e.g., `SUM`, `AVG`).

* **Engine Level Execution:** When no aggregate functions are specified (`SELECT a, b ... GROUP BY a, b`), modern optimizers compile both queries into identical physical plans (typically a Hash Aggregate or Sort/Unique operator).

* **Filtering Nuance:** `GROUP BY` allows post-aggregation filtering via `HAVING` and aggregation over child columns (`SUM(c)`), whereas `DISTINCT` applies purely to the projected row set.

#### Tricky Follow-up:
* **Question:** Can `DISTINCT` be applied to only one column in a multi-column `SELECT` statement in standard SQL?

* **Answer:** No in standard SQL (it applies to the whole row tuple). However, PostgreSQL and DuckDB support `DISTINCT ON (col1) col2, col3 ORDER BY col1, col4 DESC`, which returns the first row for each distinct `col1` based on the specified order.


In [10]:
%%sql
CREATE TABLE raw_logs (
    service VARCHAR,
    region VARCHAR,
    latency_ms INT
);
INSERT INTO raw_logs VALUES
('Auth', 'US-East', 45),
('Auth', 'US-East', 50),
('Auth', 'EU-West', 60),
('Billing', 'US-East', 30),
('Billing', 'US-East', 35);

SELECT DISTINCT service, region FROM raw_logs ORDER BY service, region;

SELECT service, region, AVG(latency_ms) AS avg_lat 
FROM raw_logs 
GROUP BY service, region 
ORDER BY service, region;

service,region,avg_lat
Auth,EU-West,60.0
Auth,US-East,47.5
Billing,US-East,32.5


### Q010: Data Type Casting: CAST, TRY_CAST, and Coercion Precedence

#### Verbal Answer:
* **Standard Casting:** `CAST(col AS target_type)` converts values explicitly. If a string contains invalid formatting (e.g. `'N/A'` cast to `INTEGER`), standard `CAST` throws a fatal query runtime error.

* **Safe Casting with `TRY_CAST` / `SAFE_CAST`:** Returns `NULL` instead of failing if the conversion is impossible. Crucial for resilient ingestion of unvalidated raw CSV/JSON pipelines.

* **Implicit Coercion Precedence:** When comparing mixed types (e.g., `VARCHAR` column vs integer literal `123`), the engine converts the type with lower precedence to the type with higher precedence (often casting string to numeric, which can trigger silent full-table scans if the index cannot be used).

#### Tricky Follow-up:
* **Question:** Why is `WHERE cast_date_str = 20240101` dangerous for query indexing on a string column `cast_date_str`?

* **Answer:** The engine implicitly applies `CAST(cast_date_str AS INT)` to every single row in the table, disabling B-Tree index seeks on `cast_date_str` and forcing an expensive full table scan.


In [11]:
%%sql
CREATE TABLE raw_inbound (
    raw_id VARCHAR,
    raw_score VARCHAR
);
INSERT INTO raw_inbound VALUES
('101', '98.5'),
('102', 'corrupt_val'),
('103', '77.2');

SELECT 
    raw_id,
    raw_score,
    TRY_CAST(raw_score AS DOUBLE) AS clean_score,
    COALESCE(TRY_CAST(raw_score AS DOUBLE), 0.0) AS defaulted_score
FROM raw_inbound;

raw_id,raw_score,clean_score,defaulted_score
101,98.5,98.5,98.5
102,corrupt_val,NaN,0.0
103,77.2,77.2,77.2


### -------------------------------------------------------------------------
### PILLAR 2: JOINS, FILTER LOGIC & NULL SEMANTICS
### -------------------------------------------------------------------------

### Q011: Three-Valued Logic (3VL): TRUE, FALSE, and UNKNOWN with NULLs

#### Verbal Answer:
* **Three Truth Values:** In SQL relational calculus, boolean comparisons evaluate to `TRUE`, `FALSE`, or `UNKNOWN` (represented by `NULL`).

* **NULL Comparisons:** `NULL = NULL`, `NULL != NULL`, and `NULL > 5` all evaluate to `UNKNOWN`. Therefore, neither `WHERE col = NULL` nor `WHERE col != NULL` will ever match any rows.

* **Testing for NULL:** Always use the dedicated predicate operators `IS NULL` or `IS NOT NULL`, which evaluate strictly to deterministic boolean `TRUE` or `FALSE`.

#### Tricky Follow-up:
* **Question:** What does `SELECT * FROM table WHERE NOT (status = 'ACTIVE')` return if a row has `status = NULL`?

* **Answer:** It does NOT return the row! `status = 'ACTIVE'` evaluates to `UNKNOWN`. `NOT (UNKNOWN)` evaluates to `UNKNOWN`. The `WHERE` clause requires `TRUE` to retain a row, so the row is filtered out.


In [12]:
%%sql
CREATE TABLE customers (cust_id INT, name VARCHAR, loyalty_tier VARCHAR);
INSERT INTO customers VALUES
(1, 'Alice', 'Gold'),
(2, 'Bob', 'Silver'),
(3, 'Charlie', NULL);

SELECT * FROM customers WHERE loyalty_tier != 'Gold';

SELECT * FROM customers WHERE loyalty_tier != 'Gold' OR loyalty_tier IS NULL;

cust_id,name,loyalty_tier
2,Bob,Silver
3,Charlie,None


### Q012: INNER, LEFT, RIGHT, and FULL OUTER JOIN Mechanics

#### Verbal Answer:
* **INNER JOIN:** Emits rows only where the join predicate evaluates to `TRUE` across both the left and right relations.

* **LEFT OUTER JOIN:** Emits all rows from the left relation. Where matching rows exist on the right, values are populated; where no match exists, all right-table columns are filled with `NULL`.

* **RIGHT OUTER JOIN:** Mirror opposite of LEFT JOIN. (Often rewritten as LEFT JOIN in production pipelines for consistency).

* **FULL OUTER JOIN:** Emits all matching rows plus all unmatched left rows (with NULL right attributes) and all unmatched right rows (with NULL left attributes).

#### Tricky Follow-up:
* **Question:** How can you find rows present in Table A that do not exist in Table B using a LEFT JOIN?

* **Answer:** Perform `SELECT A.* FROM A LEFT JOIN B ON A.id = B.id WHERE B.id IS NULL` (anti-join pattern).


In [13]:
%%sql
CREATE TABLE users (user_id INT, name VARCHAR);
CREATE TABLE orders (order_id INT, user_id INT, amount DOUBLE);

INSERT INTO users VALUES (1, 'Alice'), (2, 'Bob'), (3, 'Charlie');
INSERT INTO orders VALUES (101, 1, 99.0), (102, 1, 49.0), (103, 4, 150.0);

SELECT u.user_id, u.name
FROM users u
LEFT JOIN orders o ON u.user_id = o.user_id
WHERE o.order_id IS NULL;

SELECT u.user_id AS u_id, u.name, o.order_id, o.amount
FROM users u
FULL OUTER JOIN orders o ON u.user_id = o.user_id
ORDER BY COALESCE(u.user_id, o.user_id);

u_id,name,order_id,amount
1,Alice,102,49.0
1,Alice,101,99.0
2,Bob,<NA>,NaN
3,Charlie,<NA>,NaN
<NA>,None,103,150.0


### Q013: CROSS JOIN and Cartesian Products: When and Why

#### Verbal Answer:
* **Cartesian Product Definition:** A `CROSS JOIN` pairs every single row of Table A with every single row of Table B, producing an output size equal to `|A| * |B|` rows.

* **Intentional Production Use Cases:**
  - Generating master calendar / date grids across all products to identify zero-sales days.
  - Generating complete state-action matrices or parameter grids for ML feature pipelines.

* **Accidental Risk:** Omitting a join condition on an `INNER JOIN` (e.g., comma-separated tables in `FROM A, B`) triggers a Cartesian explosion, exhausting memory and stalling clusters.

#### Tricky Follow-up:
* **Question:** How do you generate a complete matrix of all Store IDs and all Dates in a month?

* **Answer:** Generate a unique list of Stores and a generated series of Dates, then `CROSS JOIN` them together. Afterwards, `LEFT JOIN` actual sales data onto this grid.


In [14]:
%%sql
CREATE TABLE stores (store_id VARCHAR);
INSERT INTO stores VALUES ('Store_East'), ('Store_West');

WITH dates AS (
    SELECT CAST(dt AS DATE) AS sale_date
    FROM generate_series(DATE '2024-01-01', DATE '2024-01-03', INTERVAL 1 DAY) AS t(dt)
)
SELECT 
    s.store_id,
    d.sale_date
FROM stores s
CROSS JOIN dates d
ORDER BY s.store_id, d.sale_date;

store_id,sale_date
Store_East,2024-01-01
Store_East,2024-01-02
Store_East,2024-01-03
Store_West,2024-01-01
Store_West,2024-01-02
Store_West,2024-01-03


### Q014: Filtering in ON Clause vs WHERE Clause in Outer Joins

#### Verbal Answer:
* **ON Clause in OUTER JOIN:** Evaluated *during* the join matching phase. If a row in the right table fails the `ON` condition, the left row is still retained in the result set with right columns populated as `NULL`.

* **WHERE Clause in OUTER JOIN:** Evaluated *after* the join has materialized. Any predicate in `WHERE` checking a right-table column for a specific value (e.g., `WHERE right_table.status = 'ACTIVE'`) will eliminate rows where the right column is `NULL`, effectively converting the `LEFT JOIN` into an `INNER JOIN`.

* **Rule:** Table-specific filtering of the preserved table must go in `WHERE`; filtering of the unpreserved table should go in `ON` if you wish to retain all rows of the preserved table.

#### Tricky Follow-up:
* **Question:** What happens to a `LEFT JOIN` if you add `WHERE r.amount > 100`?

* **Answer:** Rows from the left table that had no match in the right table (where `r.amount` became `NULL`) are removed because `NULL > 100` is `UNKNOWN`. It silently degrades into an `INNER JOIN`.


In [15]:
%%sql
CREATE TABLE accounts (acc_id INT, name VARCHAR);
CREATE TABLE subs (sub_id INT, acc_id INT, status VARCHAR);

INSERT INTO accounts VALUES (1, 'Acme Corp'), (2, 'Globex'), (3, 'Initech');
INSERT INTO subs VALUES (10, 1, 'ACTIVE'), (20, 2, 'CANCELLED');

SELECT a.acc_id, a.name, s.sub_id, s.status
FROM accounts a
LEFT JOIN subs s ON a.acc_id = s.acc_id AND s.status = 'ACTIVE';

SELECT a.acc_id, a.name, s.sub_id, s.status
FROM accounts a
LEFT JOIN subs s ON a.acc_id = s.acc_id
WHERE s.status = 'ACTIVE';

acc_id,name,sub_id,status
1,Acme Corp,10,ACTIVE


### Q015: The IN (subquery) with NULL Trap: Why NOT IN Fails

#### Verbal Answer:
* **The `NOT IN` Hazard:** `val NOT IN (1, 2, NULL)` expands logically to: `val != 1 AND val != 2 AND val != NULL`.

* **Failure Mode:** Because `val != NULL` evaluates to `UNKNOWN`, the entire compound boolean expression evaluates to `UNKNOWN` (or `FALSE`). As a result, **zero rows are returned by the entire query**, even if there are hundreds of valid candidate records.

* **Safe Production Standard:** Always use `NOT EXISTS` instead of `NOT IN`, or explicitly filter out `NULL`s in the subquery (`WHERE col IS NOT NULL`).

#### Tricky Follow-up:
* **Question:** Why is `NOT EXISTS` immune to the NULL trap?

* **Answer:** `NOT EXISTS` checks whether the subquery produces at least one row matching the correlation predicate. It evaluates solely based on the presence/absence of candidate tuples, ignoring whether individual attributes inside the tuple are `NULL`.


In [16]:
%%sql
CREATE TABLE main_users (user_id INT);
CREATE TABLE banned_users (user_id INT);

INSERT INTO main_users VALUES (1), (2), (3), (4);
INSERT INTO banned_users VALUES (2), (NULL); -- Contains a NULL;

SELECT * FROM main_users WHERE user_id NOT IN (SELECT user_id FROM banned_users);

SELECT * FROM main_users m 
WHERE NOT EXISTS (
    SELECT 1 FROM banned_users b WHERE b.user_id = m.user_id
);

user_id
1
3
4


### Q016: Self Joins: Hierarchical, Manager-Employee, and Comparative Pairings

#### Verbal Answer:
* **Definition:** A `SELF JOIN` is a regular join where a table is joined to itself under different aliases (e.g., `FROM employees e JOIN employees m ON e.manager_id = m.emp_id`).

* **Common Use Cases:**
  - Resolving single-level organizational hierarchies (Employee -> Manager).
  - Finding pairs of entities matching specific criteria without duplication (`A.id < B.id`).
  - Comparing sequential events within the same entity timeline without window functions.

* **Alias Requirement:** Table aliases are mandatory so the SQL parser can uniquely qualify columns from each instance of the table.

#### Tricky Follow-up:
* **Question:** Why do we write `ON A.user_id < B.user_id` instead of `ON A.user_id != B.user_id` when finding all unique pairs of users in the same city?

* **Answer:** Using `!=` produces duplicate mirror pairs `(Alice, Bob)` and `(Bob, Alice)`. Using `<` ensures each pair is generated exactly once in canonical sorted order.


In [17]:
%%sql
CREATE TABLE staff (
    emp_id INT,
    name VARCHAR,
    manager_id INT
);
INSERT INTO staff VALUES
(1, 'Sarah (CEO)', NULL),
(2, 'Alex (VP)', 1),
(3, 'Brian (VP)', 1),
(4, 'Clara (Dev)', 2),
(5, 'David (Dev)', 2);

SELECT 
    e.emp_id,
    e.name AS employee_name,
    COALESCE(m.name, 'TOP EXECUTIVE') AS direct_manager
FROM staff e
LEFT JOIN staff m ON e.manager_id = m.emp_id
ORDER BY e.emp_id;

emp_id,employee_name,direct_manager
1,Sarah (CEO),TOP EXECUTIVE
2,Alex (VP),Sarah (CEO)
3,Brian (VP),Sarah (CEO)
4,Clara (Dev),Alex (VP)
5,David (Dev),Alex (VP)


### Q017: Non-Equi Joins: Range Intervals and Timestamp Overlaps

#### Verbal Answer:
* **Non-Equi Join Definition:** Any join where the predicate uses comparison operators other than equality (e.g., `<`, `<=`, `>`, `>=`, `BETWEEN`, `LIKE`).

* **Core Analytical Applications:**
  - Point-in-time dimension lookups: Joining an event timestamp to a slowly changing rate table valid between `start_date` and `end_date`.
  - Joining continuous metrics to discretized score bands or tax brackets.

* **Engine Performance Consideration:** Traditional Hash Joins only work on equi-joins (`=`). Non-equi joins typically force Nested Loop Joins or Band Joins, which require range-aware index structures or sorting.

#### Tricky Follow-up:
* **Question:** How does DuckDB or modern columnar engines optimize `A.ts BETWEEN B.start_ts AND B.end_ts`?

* **Answer:** They use dedicated Range Joins (Inequality Join / IEJoin) or Sort-Merge Band Joins that achieve `O(N log N)` complexity rather than the naive `O(N * M)` Cartesian scan.


In [18]:
%%sql
CREATE TABLE tax_brackets (
    tier VARCHAR,
    min_income DOUBLE,
    max_income DOUBLE,
    tax_rate DOUBLE
);
CREATE TABLE earners (
    earner_id INT,
    income DOUBLE
);

INSERT INTO tax_brackets VALUES
('Bracket 1', 0, 50000, 0.10),
('Bracket 2', 50001, 100000, 0.20),
('Bracket 3', 100001, 99999999, 0.35);

INSERT INTO earners VALUES
(101, 42000),
(102, 75000),
(103, 185000);

SELECT 
    e.earner_id,
    e.income,
    b.tier,
    b.tax_rate,
    ROUND(e.income * b.tax_rate, 2) AS estimated_tax
FROM earners e
JOIN tax_brackets b 
    ON e.income >= b.min_income AND e.income <= b.max_income
ORDER BY e.earner_id;

earner_id,income,tier,tax_rate,estimated_tax
101,42000.0,Bracket 1,0.10,4200.0
102,75000.0,Bracket 2,0.20,15000.0
103,185000.0,Bracket 3,0.35,64750.0


### Q018: Multi-Key Joins and Composite Key Collision Prevention

#### Verbal Answer:
* **Composite Join Keys:** When entities are uniquely identified by multiple attributes (e.g., `tenant_id` + `user_id`, or `country_code` + `postal_code`), joins must match on all constituent key columns.

* **Data Leakage Hazard:** Omitting one part of a composite key (e.g., joining multi-tenant SaaS tables on `user_id` alone without `tenant_id`) causes massive data cross-contamination and unintended row multiplications.

* **NULL Composite Keys:** In ANSI SQL, if any part of a composite key contains `NULL`, `ON A.k1 = B.k1 AND A.k2 = B.k2` evaluates to `UNKNOWN`. Modern engines offer `IS NOT DISTINCT FROM` or `<=>` for null-safe composite joins.

#### Tricky Follow-up:
* **Question:** What is the ANSI standard syntax to perform a null-safe equality join in modern SQL?

* **Answer:** `ON A.key IS NOT DISTINCT FROM B.key`. This returns `TRUE` if both are identical values OR if both are `NULL`.


In [19]:
%%sql
CREATE TABLE saas_users (tenant_id VARCHAR, user_id INT, email VARCHAR);
CREATE TABLE saas_billing (tenant_id VARCHAR, user_id INT, plan VARCHAR);

INSERT INTO saas_users VALUES ('ORG_A', 1, 'a@a.com'), ('ORG_B', 1, 'b@b.com');
INSERT INTO saas_billing VALUES ('ORG_A', 1, 'Enterprise'), ('ORG_B', 1, 'Starter');

SELECT 
    u.tenant_id,
    u.user_id,
    u.email,
    b.plan
FROM saas_users u
JOIN saas_billing b 
    ON u.tenant_id = b.tenant_id 
   AND u.user_id = b.user_id;

tenant_id,user_id,email,plan
ORG_A,1,a@a.com,Enterprise
ORG_B,1,b@b.com,Starter


### Q019: SEMI-JOIN vs ANTI-JOIN: Definition and Query Optimization

#### Verbal Answer:
* **Semi-Join (`EXISTS` / `IN`):** Returns rows from the left table that have at least one match in the right table. Unlike an `INNER JOIN`, it **never duplicates** left-side rows regardless of how many matches exist in the right table.

* **Anti-Join (`NOT EXISTS` / `NOT IN`):** Returns rows from the left table that have zero matches in the right table.

* **Optimizer Efficiency:** The query engine can stop scanning the right-side table as soon as the first matching record is discovered (short-circuiting).

#### Tricky Follow-up:
* **Question:** If an `INNER JOIN` produces 10 rows for a user with 10 orders, how many rows does an `EXISTS` semi-join produce?

* **Answer:** Exactly 1 row. Semi-joins check strictly for existence and preserve the cardinality of the left relation.


In [20]:
%%sql
CREATE TABLE clients (client_id INT, client_name VARCHAR);
CREATE TABLE support_tickets (ticket_id INT, client_id INT, priority VARCHAR);

INSERT INTO clients VALUES (1, 'Acme'), (2, 'BetaCo'), (3, 'CyberCorp');
INSERT INTO support_tickets VALUES (101, 1, 'HIGH'), (102, 1, 'LOW'), (103, 1, 'MED');

SELECT c.client_id, c.client_name
FROM clients c
WHERE EXISTS (
    SELECT 1 FROM support_tickets t WHERE t.client_id = c.client_id
);

SELECT c.client_id, c.client_name
FROM clients c
WHERE NOT EXISTS (
    SELECT 1 FROM support_tickets t WHERE t.client_id = c.client_id
);

client_id,client_name
2,BetaCo
3,CyberCorp


### Q020: LIKE, ILIKE, Wildcards, and Escape Characters

#### Verbal Answer:
* **Wildcards:**
  - `%`: Matches zero or more arbitrary characters.
  - `_`: Matches exactly one single character.

* **Case Sensitivity:** `LIKE` is case-sensitive in ANSI SQL and PostgreSQL. `ILIKE` is case-insensitive.

* **Escape Character Handling:** To search for literal percent `%` or underscore `_` signs, declare an `ESCAPE` clause (e.g., `WHERE discount LIKE '%10\%%' ESCAPE ''`).

#### Tricky Follow-up:
* **Question:** Why does `WHERE name LIKE '%smith'` prevent traditional B-Tree index range scans?

* **Answer:** B-Trees are sorted lexicographically from left to right. A leading wildcard (`%smith`) prevents the engine from locating the starting prefix in the index, forcing a full index or table scan.


In [21]:
%%sql
CREATE TABLE products (sku VARCHAR, title VARCHAR, promo_tag VARCHAR);
INSERT INTO products VALUES
('SKU_01', 'Pro Camera 4K', '10%_OFF'),
('SKU_02', 'Mini Drone', 'NO_DISCOUNT'),
('SKU_03', 'Pro Tripod 100', '25%_OFF'),
('SKU_04', 'Action Cam', '5%_OFF');

SELECT 
    sku,
    title,
    promo_tag
FROM products
WHERE title ILIKE '%pro%'
  AND promo_tag LIKE '%!%_OFF' ESCAPE '!';

sku,title,promo_tag
SKU_01,Pro Camera 4K,10%_OFF
SKU_03,Pro Tripod 100,25%_OFF


### -------------------------------------------------------------------------
### PILLAR 3: AGGREGATIONS, GROUPING SETS & PIVOT TRANSFORMATIONS
### -------------------------------------------------------------------------

### Q021: Standard Aggregations: SUM, AVG, MIN, MAX, and NULL Ignorance

#### Verbal Answer:
* **NULL Handling in Aggregates:** All standard aggregate functions (`SUM`, `AVG`, `MIN`, `MAX`, `STDDEV`) ignore `NULL` values during calculation.

* **The `AVG` Calculation Nuance:** `AVG(col)` divides the sum of non-null values by the count of non-null rows (`SUM(col) / COUNT(col)`). It does **not** divide by total rows. If `NULL` represents 0 in your domain, you must write `AVG(COALESCE(col, 0))`.

* **All-NULL Groups:** `SUM`, `AVG`, `MIN`, `MAX` on an entirely null dataset return `NULL`, whereas `COUNT(*)` returns `0`.

#### Tricky Follow-up:
* **Question:** Given rows [10, 20, NULL], what does `AVG(val)` return versus `AVG(COALESCE(val, 0))`?

* **Answer:** `AVG(val)` returns `(10 + 20) / 2 = 15.0`. `AVG(COALESCE(val, 0))` returns `(10 + 20 + 0) / 3 = 10.0`.


In [22]:
%%sql
CREATE TABLE scores (student VARCHAR, score DOUBLE);
INSERT INTO scores VALUES ('A', 100), ('B', 50), ('C', NULL);

SELECT 
    SUM(score) AS total_score,
    COUNT(score) AS non_null_count,
    COUNT(*) AS total_rows,
    AVG(score) AS standard_avg,
    AVG(COALESCE(score, 0)) AS coalesced_avg
FROM scores;

total_score,non_null_count,total_rows,standard_avg,coalesced_avg
150.0,2,3,75.0,50.0


### Q022: GROUP BY with Multiple Columns and Cardinality Slicing

#### Verbal Answer:
* **Multi-Column Bucketing:** `GROUP BY col1, col2` creates unique combinations of `(col1, col2)` values. The number of output rows equals the cardinality of the Cartesian product of distinct values present.

* **All Non-Aggregated Columns Rule:** Every column present in the `SELECT` list that is not wrapped inside an aggregate function **must** be explicitly declared in the `GROUP BY` clause in standard ANSI SQL.

* **Order of Columns:** The order of columns listed inside `GROUP BY` does not change the mathematical output, but can impact physical hash-table bucket organization.

#### Tricky Follow-up:
* **Question:** Why does MySQL historically allow selecting non-aggregated columns missing from `GROUP BY` (pre-5.7 `ONLY_FULL_GROUP_BY` mode)?

* **Answer:** MySQL returned an arbitrary, non-deterministic row value from the group. Modern SQL engines and SQL standards strictly forbid this due to data inaccuracy.


In [23]:
%%sql
CREATE TABLE regional_sales (
    region VARCHAR,
    channel VARCHAR,
    revenue DOUBLE
);
INSERT INTO regional_sales VALUES
('North', 'Online', 5000),
('North', 'Retail', 3000),
('North', 'Online', 4500),
('South', 'Online', 2000),
('South', 'Retail', 6000);

SELECT 
    region,
    channel,
    SUM(revenue) AS total_revenue,
    COUNT(*) AS tx_count
FROM regional_sales
GROUP BY region, channel
ORDER BY region, channel;

region,channel,total_revenue,tx_count
North,Online,9500.0,2
North,Retail,3000.0,1
South,Online,2000.0,1
South,Retail,6000.0,1


### Q023: GROUPING SETS: Aggregating Across Multiple Granularities in a Single Scan

#### Verbal Answer:
* **The Problem:** When building executive dashboards requiring totals by `(Region, Product)`, totals by `(Region)`, totals by `(Product)`, and Grand Total, running multiple queries with `UNION ALL` scans the base table 4 separate times.

* **The Solution:** `GROUPING SETS ((Region, Product), (Region), (Product), ())` computes all 4 dimensional aggregations in a single optimized pass over the dataset.

* **NULL Representation:** Missing dimensions in the sub-totals are returned as `NULL`.

#### Tricky Follow-up:
* **Question:** How can you distinguish whether a `NULL` in the output represents a subtotal roll-up versus an actual `NULL` in the raw data?

* **Answer:** Use the standard `GROUPING(col)` function. It returns `1` if the column was aggregated away in the subtotal/grand-total and `0` if it represents actual row data.


In [24]:
%%sql
CREATE TABLE sales_cube (
    region VARCHAR,
    product VARCHAR,
    amount DOUBLE
);
INSERT INTO sales_cube VALUES
('East', 'Apples', 100),
('East', 'Oranges', 150),
('West', 'Apples', 200);

SELECT 
    region,
    product,
    GROUPING(region) AS is_region_subtotal,
    GROUPING(product) AS is_product_subtotal,
    SUM(amount) AS total_sales
FROM sales_cube
GROUP BY GROUPING SETS (
    (region, product),
    (region),
    (product),
    ()
)
ORDER BY is_region_subtotal, is_product_subtotal, region, product;

region,product,is_region_subtotal,is_product_subtotal,total_sales
East,Apples,0,0,100.0
East,Oranges,0,0,150.0
West,Apples,0,0,200.0
East,None,0,1,250.0
West,None,0,1,200.0
None,Apples,1,0,300.0
None,Oranges,1,0,150.0
None,None,1,1,450.0


### Q024: ROLLUP: Hierarchical Subtotals and Grand Totals

#### Verbal Answer:
* **Hierarchical Aggregation:** `GROUP BY ROLLUP(Year, Quarter, Month)` generates progressive hierarchical subtotals from right to left:
  - `(Year, Quarter, Month)`
  - `(Year, Quarter)`
  - `(Year)`
  - `()` (Grand Total).

* **Use Case:** Perfect for strict tree hierarchies (e.g., Continent -> Country -> City or Year -> Quarter -> Month) where intermediate non-hierarchical combinations (e.g., Month alone) are meaningless.

* **Total Combinations:** For $N$ columns, `ROLLUP` produces exactly $N + 1$ grouping levels.

#### Tricky Follow-up:
* **Question:** How many grouping levels does `ROLLUP(a, b, c)` produce versus `CUBE(a, b, c)`?

* **Answer:** `ROLLUP(a, b, c)` produces 4 grouping levels (`3 + 1`). `CUBE(a, b, c)` produces all $2^3 = 8$ possible power set combinations.


In [25]:
%%sql
CREATE TABLE time_sales (
    yr INT,
    qtr VARCHAR,
    revenue DOUBLE
);
INSERT INTO time_sales VALUES
(2023, 'Q1', 1000),
(2023, 'Q2', 1500),
(2024, 'Q1', 2000),
(2024, 'Q2', 2500);

SELECT 
    yr,
    qtr,
    SUM(revenue) AS total_revenue
FROM time_sales
GROUP BY ROLLUP(yr, qtr)
ORDER BY yr NULLS LAST, qtr NULLS LAST;

yr,qtr,total_revenue
2023,Q1,1000.0
2023,Q2,1500.0
2023,None,2500.0
2024,Q1,2000.0
2024,Q2,2500.0
2024,None,4500.0
<NA>,None,7000.0


### Q025: CUBE: Power-Set Multidimensional Aggregations

#### Verbal Answer:
* **Multidimensional Cross-Tabulation:** `GROUP BY CUBE(a, b)` generates all $2^N$ possible permutations of grouping sets:
  - `(a, b)`
  - `(a)`
  - `(b)`
  - `()`

* **Analytical Application:** Powers OLAP slicing and dicing, allowing Business Intelligence (BI) caches to answer any arbitrary slice query instantly without re-querying the base fact table.

#### Tricky Follow-up:
* **Question:** What is the primary risk of running `CUBE` on a table with 8 dimensions?

* **Answer:** The Cartesian dimension explosion: $2^8 = 256$ separate grouping aggregations will be computed, causing massive CPU load, memory exhaustion, and huge output data explosion.


In [26]:
%%sql
CREATE TABLE channel_sales (
    dept VARCHAR,
    gender VARCHAR,
    spend DOUBLE
);
INSERT INTO channel_sales VALUES
('Sports', 'M', 500),
('Sports', 'F', 700),
('Beauty', 'F', 1200);

SELECT 
    dept,
    gender,
    SUM(spend) AS total_spend
FROM channel_sales
GROUP BY CUBE(dept, gender)
ORDER BY dept NULLS LAST, gender NULLS LAST;

dept,gender,total_spend
Beauty,F,1200.0
Beauty,None,1200.0
Sports,F,700.0
Sports,M,500.0
Sports,None,1200.0
None,F,1900.0
None,M,500.0
None,None,2400.0


### Q026: Conditional Aggregations: The CASE WHEN Inside Aggregate Pattern

#### Verbal Answer:
* **The Pivot Pattern:** Embedding `CASE WHEN` inside aggregate functions (`SUM(CASE WHEN ... THEN 1 ELSE 0 END)`) allows transforming row-level categorical values into columns (pivoting) in ANSI standard SQL.

* **Filtering Specific Buckets:** Enables calculating multiple conditional metrics in a single pass without joining multiple subqueries.

* **FILTER Clause (Modern Standard):** Modern engines (PostgreSQL, DuckDB, SQLite 3.30+) support the cleaner ANSI standard syntax: `SUM(amount) FILTER (WHERE status = 'PAID')`.

#### Tricky Follow-up:
* **Question:** Why is `COUNT(CASE WHEN status = 'PAID' THEN 1 END)` equivalent to `SUM(CASE WHEN status = 'PAID' THEN 1 ELSE 0 END)`?

* **Answer:** Because the `CASE WHEN` without `ELSE` returns `NULL` when the condition is false. Since `COUNT(column)` ignores `NULL`s, it counts only the `PAID` rows.


In [27]:
%%sql
CREATE TABLE orders_feed (
    user_id INT,
    status VARCHAR,
    amount DOUBLE
);
INSERT INTO orders_feed VALUES
(1, 'COMPLETED', 100),
(1, 'REFUNDED', 30),
(2, 'COMPLETED', 200),
(2, 'COMPLETED', 150),
(3, 'REFUNDED', 50);

SELECT 
    user_id,
    -- ANSI CASE WHEN pattern
    SUM(CASE WHEN status = 'COMPLETED' THEN amount ELSE 0 END) AS completed_spend,
    SUM(CASE WHEN status = 'REFUNDED' THEN amount ELSE 0 END) AS refunded_spend,
    -- Modern FILTER clause
    COUNT(*) FILTER (WHERE status = 'COMPLETED') AS completed_count
FROM orders_feed
GROUP BY user_id
ORDER BY user_id;

user_id,completed_spend,refunded_spend,completed_count
1,100.0,30.0,1
2,350.0,0.0,2
3,0.0,50.0,0


### Q027: PIVOT and UNPIVOT: Syntax and Native Engine Support

#### Verbal Answer:
* **PIVOT:** Rotates row values into column headers, aggregating values in the process (long-to-wide transformation).

* **UNPIVOT:** Rotates column headers into row values (wide-to-long normalization).

* **Engine Landscape:** Supported natively in Snowflake, BigQuery, DuckDB, Oracle, and MS SQL Server. In PostgreSQL or older engines, pivoting is accomplished via conditional aggregation (`CASE WHEN`) or `crosstab()`.

#### Tricky Follow-up:
* **Question:** Why is unpivoted (normalized/long) format generally preferred for analytical data warehouses over wide format?

* **Answer:** Long format allows dynamic addition of new categories without executing `ALTER TABLE` DDL schema migrations, and optimizes columnar compression and vector filtering across partitions.


In [28]:
%%sql
CREATE TABLE monthly_revenue (
    product VARCHAR,
    month VARCHAR,
    revenue DOUBLE
);
INSERT INTO monthly_revenue VALUES
('Laptop', 'Jan', 10000),
('Laptop', 'Feb', 12000),
('Phone', 'Jan', 8000),
('Phone', 'Feb', 9500);

PIVOT monthly_revenue 
ON month 
USING SUM(revenue) 
GROUP BY product;

product,Feb,Jan
Phone,9500.0,8000.0
Laptop,12000.0,10000.0


### Q028: STRING_AGG, LISTAGG, and ARRAY_AGG: Group Concatenation

#### Verbal Answer:
* **Aggregation to Collections:** Instead of reducing rows to a scalar number, these functions collapse multiple row values within a group into a delimited string or a native array/list.

* **Engine Naming Standards:**
  - `STRING_AGG(col, delimiter [ORDER BY ...])`: PostgreSQL, BigQuery, DuckDB, SQL Server.
  - `LISTAGG(col, delimiter) WITHIN GROUP (ORDER BY ...)`: Oracle, Snowflake, Redshift.
  - `ARRAY_AGG(col [ORDER BY ...])`: PostgreSQL, BigQuery, Snowflake, DuckDB (returns native array).

* **Ordering:** Always specify `ORDER BY` inside the aggregate function to ensure deterministic concatenation order.

#### Tricky Follow-up:
* **Question:** What happens if a group contains `NULL` values when calling `STRING_AGG(name, ', ')`?

* **Answer:** `STRING_AGG` silently ignores `NULL` entries without appending extra delimiters.


In [29]:
%%sql
CREATE TABLE team_members (
    team_name VARCHAR,
    member_name VARCHAR,
    role_order INT
);
INSERT INTO team_members VALUES
('Alpha', 'Alice (Lead)', 1),
('Alpha', 'Bob', 2),
('Alpha', 'Charlie', 3),
('Beta', 'David (Lead)', 1),
('Beta', 'Eve', 2);

SELECT 
    team_name,
    STRING_AGG(member_name, ' -> ' ORDER BY role_order ASC) AS team_hierarchy,
    ARRAY_AGG(member_name ORDER BY role_order ASC) AS member_list
FROM team_members
GROUP BY team_name;

team_name,team_hierarchy,member_list
Alpha,Alice (Lead) -> Bob -> Charlie,"[Alice (Lead), Bob, Charlie]"
Beta,David (Lead) -> Eve,"[David (Lead), Eve]"


### Q029: Approximate Aggregations: HyperLogLog (HLL) and Approximate Percentiles

#### Verbal Answer:
* **The Exact Count Distinct Bottleneck:** On massive tables with billions of rows, `COUNT(DISTINCT user_id)` requires transferring all unique IDs across the network to a single node to build a hash table, causing OOM errors.

* **HyperLogLog (HLL):** Probabilistic algorithm that estimates distinct counts with standard error around 1-2% using fixed memory (e.g. 1 KB) regardless of cardinality (`APPROX_COUNT_DISTINCT` in Spark/Snowflake/DuckDB).

* **T-Digest / GK for Percentiles:** Functions like `APPROX_PERCENTILE` or `MEDIAN` estimate quantiles with minimal memory without requiring a full sort of terabytes of data.

#### Tricky Follow-up:
* **Question:** Can you pre-aggregate HyperLogLog sketches across daily batches and merge them later?

* **Answer:** Yes! HLL sketches have the mathematical property of **mergeability**. Daily HLL sketches can be combined using `HLL_COMBINE()` to find the exact 30-day distinct active users without scanning raw logs.


In [30]:
%%sql
CREATE TABLE raw_events AS 
SELECT 
    (random() * 10000)::INT AS user_id,
    (random() * 500)::DOUBLE AS page_load_ms
FROM generate_series(1, 100000);

SELECT 
    COUNT(DISTINCT user_id) AS exact_distinct_users,
    APPROX_COUNT_DISTINCT(user_id) AS approx_hll_distinct_users,
    ROUND(APPROX_QUANTILE(page_load_ms, 0.95), 2) AS p95_latency_ms
FROM raw_events;

exact_distinct_users,approx_hll_distinct_users,p95_latency_ms
10000,8571,475.04


### Q030: Statistical Aggregations: STDDEV, VARIANCE, CORR, and COVAR

#### Verbal Answer:
* **Sample vs Population Statistics:**
  - `STDDEV_SAMP(x)` / `VAR_SAMP(x)`: Divides sum of squared errors by $N - 1$ (unbiased sample estimator).
  - `STDDEV_POP(x)` / `VAR_POP(x)`: Divides by $N$ (entire population).

* **Bivariate Correlation:** `CORR(y, x)` computes Pearson's correlation coefficient $r \in [-1, 1]$ directly in the database engine in a single scan.

* **Database Acceleration:** Pushing statistical moments into the database engine computes summary metrics across billions of rows in parallel before transferring data to Python.

#### Tricky Follow-up:
* **Question:** What does `STDDEV_SAMP(val)` return if a group contains only 1 row?

* **Answer:** It returns `NULL` because the denominator $N - 1 = 1 - 1 = 0$, preventing a division-by-zero runtime error.


In [31]:
%%sql
CREATE TABLE metrics (
    ad_spend DOUBLE,
    conversions DOUBLE
);
INSERT INTO metrics VALUES
(100, 10),
(200, 22),
(300, 29),
(400, 42),
(500, 51);

SELECT 
    ROUND(AVG(ad_spend), 2) AS avg_spend,
    ROUND(STDDEV_SAMP(ad_spend), 2) AS std_spend,
    ROUND(VAR_SAMP(ad_spend), 2) AS var_spend,
    ROUND(CORR(conversions, ad_spend), 4) AS pearson_correlation
FROM metrics;

avg_spend,std_spend,var_spend,pearson_correlation
300.0,158.11,25000.0,0.9969


### -------------------------------------------------------------------------
### PILLAR 4: STRING MANIPULATION, REGULAR EXPRESSIONS & DATETIME/TIMEZONES
### -------------------------------------------------------------------------

### Q031: String Slicing and Extraction: SUBSTRING, LEFT, RIGHT, and SPLIT_PART

#### Verbal Answer:
* **1-Based Indexing:** Standard SQL string functions (`SUBSTRING`, `POSITION`, `INSTR`) are **1-indexed**, meaning the first character is at index `1`, not `0`.

* **Core Functions:**
  - `SUBSTRING(str FROM start FOR length)` or `SUBSTR(str, start, len)`.
  - `LEFT(str, n)` / `RIGHT(str, n)`.
  - `SPLIT_PART(str, delimiter, index)`: Extracts the Nth token from a delimited string (extremely efficient alternative to regex).

#### Tricky Follow-up:
* **Question:** How can you extract the domain name from an email address without regular expressions?

* **Answer:** Use `SPLIT_PART(email, '@', 2)` or `SUBSTRING(email FROM POSITION('@' IN email) + 1)`.


In [32]:
%%sql
CREATE TABLE user_emails (user_id INT, full_email VARCHAR);
INSERT INTO user_emails VALUES
(1, 'alice.smith@google.com'),
(2, 'bob_jones@amazon.co.uk'),
(3, 'carol123@apple.com');

SELECT 
    user_id,
    full_email,
    SPLIT_PART(full_email, '@', 1) AS username,
    SPLIT_PART(full_email, '@', 2) AS domain_name,
    LEFT(SPLIT_PART(full_email, '@', 1), 3) AS user_prefix
FROM user_emails;

user_id,full_email,username,domain_name,user_prefix
1,alice.smith@google.com,alice.smith,google.com,ali
2,bob_jones@amazon.co.uk,bob_jones,amazon.co.uk,bob
3,carol123@apple.com,carol123,apple.com,car


### Q032: Regular Expressions in SQL: REGEXP_MATCHES, REGEXP_REPLACE, and REGEXP_EXTRACT

#### Verbal Answer:
* **Pattern Matching:** Standard `LIKE` is limited to `%` and `_`. Regular expression functions allow matching arbitrary regex grammars (POSIX or PCRE).

* **Key Functions:**
  - `REGEXP_MATCHES(str, pattern)`: Boolean predicate checking if pattern exists.
  - `REGEXP_EXTRACT(str, pattern [, group])`: Extracts the matching capture group substring.
  - `REGEXP_REPLACE(str, pattern, replacement)`: Replaces all occurrences of regex match.

* **Performance:** Regex evaluation is CPU heavy. If simple string functions like `SPLIT_PART` or `LIKE` suffice, avoid invoking the regex engine.

#### Tricky Follow-up:
* **Question:** How can you mask all credit card numbers in a log message to show only the last 4 digits?

* **Answer:** Use `REGEXP_REPLACE(raw_log, '\d{4}-\d{4}-\d{4}-(\d{4})', 'XXXX-XXXX-XXXX-\1')`.


In [33]:
%%sql
CREATE TABLE server_logs (log_id INT, raw_message VARCHAR);
INSERT INTO server_logs VALUES
(1, 'User registered with IP: 192.168.1.50 at port 8080'),
(2, 'Error: Failed payment for CC: 4111-2222-3333-4444'),
(3, 'Ping received from IP: 10.0.0.1 at port 443');

SELECT 
    log_id,
    -- Extract IP address
    REGEXP_EXTRACT(raw_message, '([0-9]{1,3}\.[0-9]{1,3}\.[0-9]{1,3}\.[0-9]{1,3})', 1) AS extracted_ip,
    -- Mask sensitive card data
    REGEXP_REPLACE(raw_message, '[0-9]{4}-[0-9]{4}-[0-9]{4}-([0-9]{4})', 'XXXX-XXXX-XXXX-\1') AS sanitized_message
FROM server_logs;

log_id,extracted_ip,sanitized_message
1,192.168.1.50,User registered with IP: 192.168.1.50 at port 8080
2,,Error: Failed payment for CC: XXXX-XXXX-XXXX-4444
3,10.0.0.1,Ping received from IP: 10.0.0.1 at port 443


### Q033: Date Truncation: DATE_TRUNC vs EXTRACT / DATE_PART

#### Verbal Answer:
* **`DATE_TRUNC('granularity', timestamp)`:** Rounds down a timestamp to the beginning of the specified interval (e.g., `'day'`, `'month'`, `'year'`, `'hour'`). Returns a valid **DATE or TIMESTAMP** object.

* **`EXTRACT(part FROM timestamp)` / `DATE_PART`:** Pulls out an isolated integer scalar attribute (e.g., `EXTRACT(MONTH FROM dt)` returns `1` through `12`).

* **Grouping Trap:** To aggregate monthly revenue across multiple years, `GROUP BY DATE_TRUNC('month', dt)` separates `Jan 2023` from `Jan 2024`. `GROUP BY EXTRACT(MONTH FROM dt)` merges all Januarys across all years together!

#### Tricky Follow-up:
* **Question:** When is `EXTRACT(DOW FROM dt)` preferred over `DATE_TRUNC`?

* **Answer:** When analyzing seasonal intra-week behavior (e.g., comparing average sales on Mondays vs Fridays across 5 years of historical data).


In [34]:
%%sql
CREATE TABLE web_events (event_id INT, ts TIMESTAMP);
INSERT INTO web_events VALUES
(1, TIMESTAMP '2023-01-15 14:23:45'),
(2, TIMESTAMP '2023-01-28 09:12:00'),
(3, TIMESTAMP '2024-01-10 18:40:11');

SELECT 
    event_id,
    ts,
    DATE_TRUNC('month', ts) AS truncated_month,
    EXTRACT(MONTH FROM ts) AS isolated_month_num,
    EXTRACT(YEAR FROM ts) AS isolated_year_num,
    EXTRACT(DAYOFWEEK FROM ts) AS day_of_week
FROM web_events;

event_id,ts,truncated_month,isolated_month_num,isolated_year_num,day_of_week
1,2023-01-15 14:23:45,2023-01-01,1,2023,0
2,2023-01-28 09:12:00,2023-01-01,1,2023,6
3,2024-01-10 18:40:11,2024-01-01,1,2024,3


### Q034: Date Arithmetic: Intervals, DATEDIFF, and AGE

#### Verbal Answer:
* **Interval Arithmetic:** ANSI standard SQL allows adding and subtracting interval types directly (e.g., `current_timestamp + INTERVAL 7 DAY` or `ts - INTERVAL 3 HOUR`).

* **Difference Calculation:**
  - `DATEDIFF('unit', start_date, end_date)`: Supported in Snowflake, BigQuery, SQL Server, DuckDB.
  - Direct subtraction: `end_date - start_date` in PostgreSQL returns an integer number of days or interval.

#### Tricky Follow-up:
* **Question:** How do you calculate a customer's age in full years given their `birth_date`?

* **Answer:** Use `DATEDIFF('year', birth_date, CURRENT_DATE)` or `DATE_PART('year', AGE(CURRENT_DATE, birth_date))`.


In [35]:
%%sql
CREATE TABLE subscriptions (
    sub_id INT,
    start_date DATE,
    end_date DATE
);
INSERT INTO subscriptions VALUES
(1, DATE '2023-01-01', DATE '2023-04-15'),
(2, DATE '2023-06-01', DATE '2023-06-10'),
(3, DATE '2023-01-01', NULL);

SELECT 
    sub_id,
    start_date,
    COALESCE(end_date, CURRENT_DATE) AS effective_end_date,
    DATEDIFF('day', start_date, COALESCE(end_date, CURRENT_DATE)) AS duration_days,
    start_date + INTERVAL 30 DAY AS renewal_notice_date
FROM subscriptions;

sub_id,start_date,effective_end_date,duration_days,renewal_notice_date
1,2023-01-01,2023-04-15,104,2023-01-31
2,2023-06-01,2023-06-10,9,2023-07-01
3,2023-01-01,2026-10-02,1370,2023-01-31


### Q035: Timezones: TIMESTAMP WITHOUT TIME ZONE vs TIMESTAMP WITH TIME ZONE (TIMESTAMPTZ)

#### Verbal Answer:
* **TIMESTAMP WITHOUT TIME ZONE:** Stores a raw year-month-day hour:min:sec tuple with no timezone context (wall-clock time). Comparing wall-clock times from users in London and Tokyo produces incorrect interval math.

* **TIMESTAMPTZ (TIMESTAMP WITH TIME ZONE):** Normalizes and stores all timestamps internally in UTC. When retrieved, the client interface converts the UTC value to the session's local timezone.

* **Production Rule:** Store all backend database events in UTC (`TIMESTAMPTZ`), and convert to local timezones only at query presentation time using `AT TIME ZONE`.

#### Tricky Follow-up:
* **Question:** How does `ts AT TIME ZONE 'America/New_York'` behave on a UTC timestamp during Daylight Saving Time transitions?

* **Answer:** The engine automatically consults the IANA Olson timezone database to offset the timestamp by either UTC-5 (EST) or UTC-4 (EDT) depending on the exact date.


In [36]:
%%sql
CREATE TABLE flight_events (
    flight_id VARCHAR,
    utc_departure TIMESTAMPTZ
);
INSERT INTO flight_events VALUES
('FL_101', TIMESTAMPTZ '2024-06-01 12:00:00+00'),
('FL_102', TIMESTAMPTZ '2024-12-01 12:00:00+00');

SELECT 
    flight_id,
    utc_departure,
    -- Converted to New York local time (detects EDT vs EST)
    timezone('America/New_York', utc_departure) AS ny_local_departure,
    -- Converted to Tokyo local time (UTC+9)
    timezone('Asia/Tokyo', utc_departure) AS tokyo_local_departure
FROM flight_events;

flight_id,utc_departure,ny_local_departure,tokyo_local_departure
FL_101,2024-06-01 17:30:00+05:30,2024-06-01 08:00:00,2024-06-01 21:00:00
FL_102,2024-12-01 17:30:00+05:30,2024-12-01 07:00:00,2024-12-01 21:00:00


### Q036: Formatting & Parsing: TO_CHAR, STRPTIME, and STRFTIME

#### Verbal Answer:
* **Parsing Strings to Dates (`STRPTIME` / `TO_DATE`):** Converts raw unformatted text (e.g., `'01/31/2024'`) into a typed `DATE` or `TIMESTAMP` using format tokens (`%m/%d/%Y`).

* **Formatting Dates to Strings (`STRFTIME` / `TO_CHAR`):** Converts typed dates into custom display formats (e.g., `'2024-W05'` or `'Jan 31, 2024'`).

* **Performance Caution:** Formatting functions convert binary numeric dates into strings, which should only be done in final presentation layers.

#### Tricky Follow-up:
* **Question:** What is the difference between `%y` and `%Y` in standard `strftime` tokens?

* **Answer:** `%y` produces a two-digit year (e.g., `24`), while `%Y` produces a four-digit year (e.g., `2024`).


In [37]:
%%sql
CREATE TABLE raw_date_strings (raw_text VARCHAR);
INSERT INTO raw_date_strings VALUES
('31-10-2023 23:59:50'),
('15-04-2024 12:30:00');

SELECT 
    raw_text,
    -- Parse string into Timestamp
    strptime(raw_text, '%d-%m-%Y %H:%M:%S') AS parsed_ts,
    -- Format timestamp into human readable string
    strftime(strptime(raw_text, '%d-%m-%Y %H:%M:%S'), '%B %d, %Y (%A)') AS formatted_display
FROM raw_date_strings;

raw_text,parsed_ts,formatted_display
31-10-2023 23:59:50,2023-10-31 23:59:50,"October 31, 2023 (Tuesday)"
15-04-2024 12:30:00,2024-04-15 12:30:00,"April 15, 2024 (Monday)"


### Q037: Date Series Generation: Continuous Calendar Scaffolding

#### Verbal Answer:
* **The Sparse Data Problem:** If an e-commerce store has zero sales on Tuesday, a simple `GROUP BY sale_date` completely omits Tuesday from the output, breaking time-series chart axes and ML lags.

* **Calendar Scaffolding:** Generate a continuous synthetic sequence of all dates in the range using `generate_series(start_date, end_date, interval)` (PostgreSQL, DuckDB) or a Recursive CTE.

* **Infilling with LEFT JOIN:** `LEFT JOIN` the sparse metrics onto the continuous calendar scaffolding and use `COALESCE(sales, 0)` to guarantee continuous time bins.

#### Tricky Follow-up:
* **Question:** How can you implement calendar series generation in engines lacking `generate_series` (like Snowflake or MySQL 8.0)?

* **Answer:** Use a Recursive CTE starting with `min_date` and incrementing by `INTERVAL 1 DAY` until `max_date`, or join against a pre-built physical `Dim_Date` table.


In [38]:
%%sql
CREATE TABLE actual_orders (order_date DATE, revenue DOUBLE);
INSERT INTO actual_orders VALUES
(DATE '2024-01-01', 500.0),
(DATE '2024-01-04', 300.0); -- Missing Jan 2 and Jan 3;

WITH calendar AS (
    SELECT CAST(dt AS DATE) AS cal_date
    FROM generate_series(DATE '2024-01-01', DATE '2024-01-04', INTERVAL 1 DAY) AS t(dt)
)
SELECT 
    c.cal_date,
    COALESCE(o.revenue, 0.0) AS daily_revenue
FROM calendar c
LEFT JOIN actual_orders o ON c.cal_date = o.order_date
ORDER BY c.cal_date;

cal_date,daily_revenue
2024-01-01,500.0
2024-01-02,0.0
2024-01-03,0.0
2024-01-04,300.0


### Q038: Fuzzy Matching and String Distance: Levenshtein, Jaro-Winkler, and Soundex

#### Verbal Answer:
* **Entity Resolution / Deduplication:** Raw customer inputs often contain typos (e.g. `'Jonh Smith'` vs `'John Smith'`).

* **Distance Metrics:**
  - `LEVENSHTEIN(s1, s2)`: Counts minimum single-character edits (insertions, deletions, substitutions) to transform `s1` into `s2`.
  - `JARO_WINKLER(s1, s2)`: Computes normalized similarity score $\in [0, 1]$, giving higher weighting to matching prefixes.
  - `SOUNDEX(s)`: Encodes words by phonetic sound in English.

#### Tricky Follow-up:
* **Question:** What is the computational complexity of computing Levenshtein distance across two tables of size $N$ and $M$?

* **Answer:** It requires an $O(N \times M)$ Cartesian comparison, where each comparison is $O(L_1 \times L_2)$ string dynamic programming. On large tables, blocking techniques (e.g., matching soundex or first 3 letters first) are required to avoid cluster timeouts.


In [39]:
%%sql
CREATE TABLE messy_names (raw_input VARCHAR);
INSERT INTO messy_names VALUES
('Khurram'),
('Khuram'),
('Alexander'),
('Alexandar'),
('Google');

SELECT 
    raw_input,
    levenshtein(raw_input, 'Alexander') AS dist_to_alexander,
    ROUND(jaro_winkler_similarity(raw_input, 'Alexander'), 4) AS jaro_score
FROM messy_names
ORDER BY jaro_score DESC;

raw_input,dist_to_alexander,jaro_score
Alexander,0,1.0000
Alexandar,1,0.9556
Google,8,0.5185
Khuram,8,0.4259
Khurram,9,0.4180


### Q039: LPAD, RPAD, and Formatted Zero-Padded Key Construction

#### Verbal Answer:
* **Fixed-Width Keys:** Legacy systems and financial interchange formats require fixed-length strings (e.g., 10-character account IDs with leading zeros like `'0000123456'`).

* **Syntax:** `LPAD(str, target_length, pad_character)` pads the left side; `RPAD` pads the right side.

* **Hash/Composite Key Normalization:** Zero-padding numeric components ensures that alphanumeric sorting matches numeric sorting (`'02'` < `'10'`, whereas `'10'` < `'2'`).

#### Tricky Follow-up:
* **Question:** What happens if the source string is already longer than `target_length` in `LPAD(str, len, pad)`?

* **Answer:** The string is truncated on the right to fit the specified `target_length`.


In [40]:
%%sql
CREATE TABLE raw_accounts (raw_id INT, country VARCHAR);
INSERT INTO raw_accounts VALUES
(42, 'US'),
(1005, 'GB'),
(899901, 'DE');

SELECT 
    raw_id,
    LPAD(CAST(raw_id AS VARCHAR), 8, '0') AS padded_acc_id,
    RPAD(country, 5, '.') AS padded_country,
    CONCAT(country, '_', LPAD(CAST(raw_id AS VARCHAR), 8, '0')) AS canonical_composite_key
FROM raw_accounts;

raw_id,padded_acc_id,padded_country,canonical_composite_key
42,00000042,US...,US_00000042
1005,00001005,GB...,GB_00001005
899901,00899901,DE...,DE_00899901


### Q040: Handling Fiscal Calendars and 4-4-5 Accounting Periods

#### Verbal Answer:
* **Fiscal vs Calendar Years:** Many enterprises operate on fiscal calendars (e.g., Apple's fiscal year starts in October, government in July, retail 4-4-5 calendars).

* **SQL Implementation:** Offsetting the date before truncation (e.g. `DATE_TRUNC('year', dt + INTERVAL 3 MONTH)`) or computing fiscal quarter via modular arithmetic on the month.

* **Dimensional Modeling Best Practice:** For complex 4-4-5 or 5-4-4 retail retail calendars, never rely on arithmetic formulas; join directly to a dedicated `Dim_Date` dimension table with pre-calculated fiscal metadata.

#### Tricky Follow-up:
* **Question:** How do you calculate Fiscal Quarter if Fiscal Year starts on October 1st?

* **Answer:** `EXTRACT(QUARTER FROM dt + INTERVAL 3 MONTH)` or `CASE WHEN month >= 10 THEN 1 WHEN month >= 7 THEN 4 WHEN month >= 4 THEN 3 ELSE 2 END`.


In [41]:
%%sql
CREATE TABLE transactions_fy (tx_id INT, tx_date DATE, amount DOUBLE);
INSERT INTO transactions_fy VALUES
(1, DATE '2023-09-15', 100), -- Q4 in FY starting Oct
(2, DATE '2023-10-05', 200), -- Q1 FY24
(3, DATE '2024-01-20', 300); -- Q2 FY24;

SELECT 
    tx_id,
    tx_date,
    amount,
    -- Fiscal Year (Starting Oct 1)
    EXTRACT(YEAR FROM tx_date + INTERVAL 3 MONTH) AS fiscal_year,
    -- Fiscal Quarter
    EXTRACT(QUARTER FROM tx_date + INTERVAL 3 MONTH) AS fiscal_quarter
FROM transactions_fy;

tx_id,tx_date,amount,fiscal_year,fiscal_quarter
1,2023-09-15,100.0,2023,4
2,2023-10-05,200.0,2024,1
3,2024-01-20,300.0,2024,2


### -------------------------------------------------------------------------
### PILLAR 5: SUBQUERIES, CTES & RECURSIVE QUERIES
### -------------------------------------------------------------------------

### Q041: Scalar vs Correlated Subqueries: Architecture and Performance

#### Verbal Answer:
* **Scalar Subquery:** Returns exactly one single value (1 row, 1 column). Can be used anywhere a constant expression is valid (e.g., `SELECT salary - (SELECT AVG(salary) FROM emp)`). Evaluated once for the entire query.

* **Correlated Subquery:** References columns from the outer query table (e.g., `WHERE salary > (SELECT AVG(e2.salary) FROM emp e2 WHERE e2.dept = e1.dept)`).

* **Execution Danger:** Naive engines execute correlated subqueries row-by-row (nested loop $O(N 	imes M)$). Modern cost-based optimizers rewrite correlated subqueries into hash joins (decorrelation).

#### Tricky Follow-up:
* **Question:** What runtime error occurs if a subquery in a `SELECT` expression returns 2 rows?

* **Answer:** The engine throws a fatal runtime exception: `'more than one row returned by a subquery used as an expression'`.


In [42]:
%%sql
CREATE TABLE emp_salaries (
    emp_id INT,
    name VARCHAR,
    dept VARCHAR,
    salary DOUBLE
);
INSERT INTO emp_salaries VALUES
(1, 'Alice', 'Tech', 120000),
(2, 'Bob', 'Tech', 90000),
(3, 'Charlie', 'Sales', 110000),
(4, 'Diana', 'Sales', 70000);

SELECT 
    e.emp_id,
    e.name,
    e.dept,
    e.salary,
    ROUND((SELECT AVG(e2.salary) FROM emp_salaries e2 WHERE e2.dept = e.dept), 2) AS dept_avg_salary
FROM emp_salaries e
WHERE e.salary > (
    SELECT AVG(e2.salary) 
    FROM emp_salaries e2 
    WHERE e2.dept = e.dept
);

emp_id,name,dept,salary,dept_avg_salary
1,Alice,Tech,120000.0,105000.0
3,Charlie,Sales,110000.0,90000.0


### Q042: Common Table Expressions (CTEs): Readability and Modularity

#### Verbal Answer:
* **Definition:** A CTE (`WITH cte_name AS (...)`) defines a named temporary result set that exists only within the execution scope of a single SQL statement.

* **Advantages over Nested Subqueries:**
  - Eliminates deeply nested, unreadable subquery pyramids.
  - Enables top-down, linear narrative query structuring.
  - Allows self-referencing and recursive graph traversals.

* **Optimizer Inlining:** In modern engines (DuckDB, Snowflake, BigQuery, PostgreSQL 12+), non-recursive CTEs are inlined and cost-optimized just like subqueries.

#### Tricky Follow-up:
* **Question:** Can multiple CTEs reference preceding CTEs defined earlier in the same `WITH` block?

* **Answer:** Yes. CTEs are evaluated sequentially, so `CTE_2` can query `CTE_1`, and `CTE_3` can query both `CTE_1` and `CTE_2`.


In [43]:
%%sql
CREATE TABLE raw_orders (
    order_id INT,
    user_id INT,
    amount DOUBLE,
    status VARCHAR
);
INSERT INTO raw_orders VALUES
(1, 101, 250.0, 'COMPLETED'),
(2, 101, 150.0, 'COMPLETED'),
(3, 102, 50.0, 'CANCELLED'),
(4, 103, 500.0, 'COMPLETED');

WITH valid_orders AS (
    SELECT user_id, amount
    FROM raw_orders
    WHERE status = 'COMPLETED'
),
user_aggregates AS (
    SELECT 
        user_id,
        SUM(amount) AS total_spend,
        COUNT(*) AS completed_orders
    FROM valid_orders
    GROUP BY user_id
)
SELECT 
    user_id,
    total_spend,
    completed_orders,
    CASE WHEN total_spend >= 400 THEN 'VIP' ELSE 'Standard' END AS user_segment
FROM user_aggregates
ORDER BY total_spend DESC;

user_id,total_spend,completed_orders,user_segment
103,500.0,1,VIP
101,400.0,2,VIP


### Q043: CTE Materialization: Materialized vs Inlined CTEs and Optimization Fences

#### Verbal Answer:
* **Inlined CTE (Default in Modern SQL):** The engine treats the CTE as a virtual view, inlining its definition into the main query and pushing down outer `WHERE` filters directly into the CTE scan.

* **Materialized CTE (`WITH cte AS MATERIALIZED (...)`):** Forces the engine to compute and materialize the CTE into a temporary in-memory/disk buffer once, preventing re-evaluation if referenced multiple times.

* **Optimization Fence Warning:** Materialization creates an optimization fence, preventing predicate pushdown. If outer query has `WHERE id = 5`, a materialized CTE still scans the entire table before filtering.

#### Tricky Follow-up:
* **Question:** When is explicit `MATERIALIZED` beneficial in PostgreSQL?

* **Answer:** When a CTE performs an expensive calculation (like a complex regex parse or external call) and is referenced 3+ times in different branches of a `UNION` or `JOIN`.


In [44]:
%%sql
CREATE TABLE heavy_metrics AS 
SELECT i AS id, (i * 2.5) AS score 
FROM generate_series(1, 1000) t(i);

WITH summary_stats AS MATERIALIZED (
    SELECT 
        AVG(score) AS mean_score,
        STDDEV(score) AS std_score
    FROM heavy_metrics
)
SELECT 
    h.id,
    h.score,
    ROUND((h.score - s.mean_score) / s.std_score, 2) AS z_score
FROM heavy_metrics h
CROSS JOIN summary_stats s
WHERE h.id <= 5;

id,score,z_score
1,2.5,-1.73
2,5.0,-1.73
3,7.5,-1.72
4,10.0,-1.72
5,12.5,-1.72


### Q044: Recursive CTEs: Graph Traversal and Organizational Hierarchy

#### Verbal Answer:
* **Recursive CTE Structure:**
  1. **Anchor Member:** Initial query that returns the base root nodes (e.g., CEO where `manager_id IS NULL`).
  2. `UNION ALL`: Connects the anchor to the recursive step.
  3. **Recursive Member:** Query that joins the CTE to the base table to fetch child records, referencing the CTE itself.
  4. **Termination Condition:** Recursion halts automatically when the recursive member returns 0 rows.

* **Use Cases:** Org charts, Bill of Materials (BOM), category trees, flight path itinerary searching.

#### Tricky Follow-up:
* **Question:** What happens if there is a cycle in the data (e.g., A manages B and B manages A)?

* **Answer:** The recursive query runs in an infinite loop until it hits the engine's `MAXRECURSION` limit or memory exhaustion. Cycle detection via an array of visited IDs (`ARRAY[id]`) is required to break cycles.


In [45]:
%%sql
CREATE TABLE org_chart (
    emp_id INT,
    name VARCHAR,
    manager_id INT
);
INSERT INTO org_chart VALUES
(1, 'CEO Alice', NULL),
(2, 'VP Bob', 1),
(3, 'VP Charlie', 1),
(4, 'Director Diana', 2),
(5, 'Engineer Evan', 4);

WITH RECURSIVE Hierarchy AS (
    -- Anchor member: CEO
    SELECT 
        emp_id, 
        name, 
        manager_id, 
        1 AS depth_level,
        CAST(name AS VARCHAR) AS path_string
    FROM org_chart
    WHERE manager_id IS NULL
    
    UNION ALL
    
    -- Recursive member: Direct reports
    SELECT 
        e.emp_id, 
        e.name, 
        e.manager_id, 
        h.depth_level + 1,
        h.path_string || ' -> ' || e.name
    FROM org_chart e
    JOIN Hierarchy h ON e.manager_id = h.emp_id
)
SELECT emp_id, name, depth_level, path_string
FROM Hierarchy
ORDER BY depth_level, emp_id;

emp_id,name,depth_level,path_string
1,CEO Alice,1,CEO Alice
2,VP Bob,2,CEO Alice -> VP Bob
3,VP Charlie,2,CEO Alice -> VP Charlie
4,Director Diana,3,CEO Alice -> VP Bob -> Director Diana
5,Engineer Evan,4,CEO Alice -> VP Bob -> Director Diana -> Engineer Evan


### Q045: Recursive CTEs: Shortest Path and Graph Routing

#### Verbal Answer:
* **Graph Traversal in SQL:** Directed acyclic graphs (DAGs) and weighted graphs can be queried using recursive CTEs by accumulating path history and path distance.

* **Tracking Visited Nodes:** Storing visited nodes in an array/string prevents cycles and allows computing path lineage.

* **Shortest Path Finding:** Aggregate over candidate paths (`MIN(total_distance)`) grouped by `(origin, destination)`.

#### Tricky Follow-up:
* **Question:** How do you enforce recursion depth limits to prevent runaway queries?

* **Answer:** Include `WHERE depth < MAX_LIMIT` in the recursive member, or set engine parameters like `SET max_recursion_depth = 50`.


In [46]:
%%sql
CREATE TABLE flights (
    origin VARCHAR,
    dest VARCHAR,
    cost INT
);
INSERT INTO flights VALUES
('NYC', 'CHI', 150),
('CHI', 'SFO', 200),
('NYC', 'MIA', 100),
('MIA', 'SFO', 400),
('NYC', 'SFO', 500);

WITH RECURSIVE FlightPaths AS (
    SELECT 
        origin, 
        dest, 
        cost AS total_cost, 
        1 AS stops,
        CAST(origin || ' -> ' || dest AS VARCHAR) AS route
    FROM flights
    WHERE origin = 'NYC'
    
    UNION ALL
    
    SELECT 
        p.origin, 
        f.dest, 
        p.total_cost + f.cost, 
        p.stops + 1,
        p.route || ' -> ' || f.dest
    FROM FlightPaths p
    JOIN flights f ON p.dest = f.origin
    WHERE p.stops < 4 AND p.route NOT LIKE '%' || f.dest || '%'
)
SELECT route, total_cost, stops
FROM FlightPaths
WHERE dest = 'SFO'
ORDER BY total_cost ASC;

route,total_cost,stops
NYC -> CHI -> SFO,350,2
NYC -> SFO,500,1
NYC -> MIA -> SFO,500,2


### Q046: EXISTS vs IN: Performance, Null Safety, and Optimizer Rewrites

#### Verbal Answer:
* **`EXISTS`:** Evaluates a boolean subquery and stops scanning as soon as the first matching tuple is found (semi-join short circuit). Immune to inner `NULL` values.

* **`IN`:** Evaluates whether a scalar equals any element in the subquery result set. If the subquery contains `NULL`, `NOT IN` fails completely.

* **Modern Optimizer Behavior:** On non-nullable columns, modern cost-based query optimizers rewrite both `IN` and `EXISTS` into identical physical Semi-Join execution plans.

#### Tricky Follow-up:
* **Question:** Why is `SELECT 1` standard inside `WHERE EXISTS (SELECT 1 FROM ...)`?

* **Answer:** Because `EXISTS` tests purely for row existence. The projected column list in the `SELECT` clause of the subquery is completely ignored by the engine optimizer.


In [47]:
%%sql
CREATE TABLE stores_list (store_id INT, city VARCHAR);
CREATE TABLE daily_sales (sale_id INT, store_id INT, amount DOUBLE);

INSERT INTO stores_list VALUES (1, 'Austin'), (2, 'Dallas'), (3, 'Houston');
INSERT INTO daily_sales VALUES (10, 1, 500.0), (11, 1, 300.0), (12, 3, 1200.0);

SELECT s.store_id, s.city
FROM stores_list s
WHERE EXISTS (
    SELECT 1 
    FROM daily_sales d 
    WHERE d.store_id = s.store_id AND d.amount > 1000
);

store_id,city
3,Houston


### Q047: ANY / SOME and ALL Quantified Comparison Operators

#### Verbal Answer:
* **`x > ANY (subquery)`:** Evaluates to `TRUE` if `x` is greater than **at least one** value returned by the subquery (equivalent to `x > MIN(...)`).

* **`x > ALL (subquery)`:** Evaluates to `TRUE` if `x` is greater than **every single** value returned by the subquery (equivalent to `x > MAX(...)`).

* **`= ANY (subquery)`:** Exactly equivalent to `x IN (subquery)`.

#### Tricky Follow-up:
* **Question:** What does `x > ALL (empty subquery)` evaluate to?

* **Answer:** It evaluates to `TRUE` vacuously, according to the mathematical definition of universal quantification over an empty set.


In [48]:
%%sql
CREATE TABLE junior_devs (name VARCHAR, salary DOUBLE);
CREATE TABLE senior_devs (name VARCHAR, salary DOUBLE);

INSERT INTO junior_devs VALUES ('Alice', 85000), ('Bob', 95000);
INSERT INTO senior_devs VALUES ('Charlie', 90000), ('Diana', 130000);

Count
2


### Q048: Subqueries in FROM Clauses (Derived Tables) and Alias Rules

#### Verbal Answer:
* **Derived Tables:** A subquery inside the `FROM` clause acts as a transient virtual table for the duration of the query.

* **Mandatory Aliasing:** Standard ANSI SQL strictly mandates that all derived tables in `FROM` must be assigned an explicit alias (e.g., `FROM (SELECT ...) AS dt`).

* **Column Visibility:** Columns projected by the derived table become the valid attribute scope for outer `WHERE`, `GROUP BY`, and `SELECT` clauses.

#### Tricky Follow-up:
* **Question:** Can an outer query filter on a column of the underlying table that was omitted in the derived table's `SELECT` list?

* **Answer:** No. Derived tables create a strict scope boundary. Only explicitly projected columns exist in the derived relation.


In [49]:
%%sql
CREATE TABLE site_traffic (
    session_id INT,
    country VARCHAR,
    page_views INT
);
INSERT INTO site_traffic VALUES
(1, 'US', 5),
(2, 'US', 12),
(3, 'UK', 3),
(4, 'UK', 8),
(5, 'DE', 20);

SELECT 
    country_summary.country,
    country_summary.avg_views,
    CASE WHEN country_summary.avg_views >= 10 THEN 'High Engagement' ELSE 'Normal' END AS engagement_tier
FROM (
    SELECT 
        country,
        ROUND(AVG(page_views), 1) AS avg_views
    FROM site_traffic
    GROUP BY country
) AS country_summary
WHERE country_summary.avg_views > 5;

country,avg_views,engagement_tier
US,8.5,Normal
UK,5.5,Normal
DE,20.0,High Engagement


### Q049: LATERAL Joins and CROSS APPLY: Row-by-Row Correlated Joins

#### Verbal Answer:
* **`LATERAL` Join (PostgreSQL, DuckDB) / `CROSS APPLY` (SQL Server, Oracle):** Allows a subquery in the `FROM` or `JOIN` clause to reference columns from preceding tables in the join order.

* **Standard Join Limitation:** Standard joins evaluate subqueries independently in isolation. `JOIN (SELECT ... WHERE outer_table.id = ...)` throws an unknown identifier error without `LATERAL`.

* **Key Analytical Powers:**
  - Computing Top-N items per category dynamically.
  - Expanding arrays/JSON records row-by-row into columns.
  - Reusing complex calculated columns across multiple output expressions.

#### Tricky Follow-up:
* **Question:** What is the difference between `CROSS JOIN LATERAL` and `LEFT JOIN LATERAL`?

* **Answer:** If the lateral subquery returns 0 rows, `CROSS JOIN LATERAL` drops the outer row entirely (inner join behavior), while `LEFT JOIN LATERAL` preserves the outer row with `NULL`s for lateral columns.


In [50]:
%%sql
CREATE TABLE departments_lead (dept_id INT, dept_name VARCHAR);
CREATE TABLE dept_employees (emp_id INT, dept_id INT, salary DOUBLE);

INSERT INTO departments_lead VALUES (1, 'Engineering'), (2, 'Marketing');
INSERT INTO dept_employees VALUES
(101, 1, 150000), (102, 1, 140000), (103, 1, 110000),
(201, 2, 90000),  (202, 2, 85000),  (203, 2, 70000);

SELECT 
    d.dept_name,
    top_earners.emp_id,
    top_earners.salary
FROM departments_lead d
CROSS JOIN LATERAL (
    SELECT e.emp_id, e.salary
    FROM dept_employees e
    WHERE e.dept_id = d.dept_id
    ORDER BY e.salary DESC
    LIMIT 2
) AS top_earners
ORDER BY d.dept_name, top_earners.salary DESC;

dept_name,emp_id,salary
Engineering,101,150000.0
Engineering,102,140000.0
Marketing,201,90000.0
Marketing,202,85000.0


### Q050: CTE vs Temporary Tables vs Views: Selection Matrix

#### Verbal Answer:
* **Views (`CREATE VIEW`):** Virtual saved query definitions stored permanently in metadata catalog. Re-evaluated every time they are queried. Zero storage cost.

* **Temporary Tables (`CREATE TEMP TABLE`):** Materialized physical tables stored in temp storage for the duration of the database session. Supports creating custom B-Tree indexes and statistics.

* **CTEs (`WITH ...`):** Scoped strictly to a single query execution. No catalog footprint.

* **Decision Framework:**
  - Single query modularity -> CTE.
  - Multi-step ETL pipeline referencing intermediate dataset 5+ times with indexing needs -> Temp Table.
  - Reusable business logic across many team queries -> View.

#### Tricky Follow-up:
* **Question:** Why can creating an index on a Temp Table dramatically outperform a recursive CTE on massive multi-step graph joins?

* **Answer:** Because Temp Tables can be indexed on join keys and analyzed for distribution statistics, allowing the cost-based optimizer to choose optimal Hash or Merge join plans instead of unindexed scans.


In [51]:
%%sql
CREATE TABLE raw_txs AS 
SELECT i AS tx_id, (i * 10) AS val 
FROM generate_series(1, 100) t(i);

CREATE TEMPORARY TABLE filtered_txs AS
SELECT * FROM raw_txs WHERE val >= 500;

SELECT 
    COUNT(*) AS high_val_count,
    AVG(val) AS avg_high_val
FROM filtered_txs;

high_val_count,avg_high_val
51,750.0


## =========================================================================
## SENIORITY LEVEL: MID-LEVEL / SENIOR DATA SCIENTIST / ANALYTICS ENGINEER
## =========================================================================

### -------------------------------------------------------------------------
### PILLAR 6: STANDARD WINDOW FUNCTIONS: RANKING & OFFSETS
### -------------------------------------------------------------------------

### Q051: ROW_NUMBER vs RANK vs DENSE_RANK: Tie-Breaking Mechanics

#### Verbal Answer:
* **`ROW_NUMBER()`:** Assigns a unique sequential integer ($1, 2, 3, 4, ...$) to each row within a partition. Ties are broken non-deterministically unless the `ORDER BY` clause has unique tie-breaker columns.

* **`RANK()`:** Assigns identical rank numbers to tied values, but skips subsequent rank numbers by the number of tied elements ($1, 2, 2, 4, ...$).

* **`DENSE_RANK()`:** Assigns identical rank numbers to tied values, and increments consecutively without skipping ($1, 2, 2, 3, ...$).

* **Interview Gold Standard:** When asked for the "Nth highest salary", always use `DENSE_RANK()`. Using `ROW_NUMBER()` or `LIMIT 1 OFFSET N-1` fails when multiple employees share identical top salaries.

#### Tricky Follow-up:
* **Question:** Given salaries `[100k, 90k, 90k, 80k]`, what are the outputs of `ROW_NUMBER()`, `RANK()`, and `DENSE_RANK()` for the 80k row?

* **Answer:** `ROW_NUMBER()` outputs `4`. `RANK()` outputs `4` (skipping rank 3). `DENSE_RANK()` outputs `3`.


In [52]:
%%sql
CREATE TABLE comp_data (
    emp_name VARCHAR,
    dept VARCHAR,
    salary DOUBLE
);
INSERT INTO comp_data VALUES
('Alice', 'Tech', 100000),
('Bob', 'Tech', 90000),
('Charlie', 'Tech', 90000),
('David', 'Tech', 80000),
('Eve', 'Sales', 95000),
('Frank', 'Sales', 70000);

SELECT 
    dept,
    emp_name,
    salary,
    ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) AS row_num,
    RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS rnk,
    DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS dense_rnk
FROM comp_data
ORDER BY dept, salary DESC;

dept,emp_name,salary,row_num,rnk,dense_rnk
Sales,Eve,95000.0,1,1,1
Sales,Frank,70000.0,2,2,2
Tech,Alice,100000.0,1,1,1
Tech,Bob,90000.0,2,2,2
Tech,Charlie,90000.0,3,2,2
Tech,David,80000.0,4,4,3


### Q052: LAG and LEAD: Offsets, Lookbacks, and Period-over-Period Changes

#### Verbal Answer:
* **`LAG(column, offset, default_value)`:** Accesses data from a preceding row at a specified physical offset within the partition (defaults to offset 1 and default `NULL`).

* **`LEAD(column, offset, default_value)`:** Accesses data from a subsequent row ahead in the partition.

* **Core Analytical Applications:**
  - Day-over-day (DoD) or Month-over-Month (MoM) revenue growth: `(revenue - LAG(revenue, 1) OVER (ORDER BY dt)) / LAG(revenue, 1) OVER (ORDER BY dt)`.
  - Calculating duration between user clicks / pageviews.

#### Tricky Follow-up:
* **Question:** What happens if you omit the `default_value` argument and `LAG` evaluates beyond the partition boundary?

* **Answer:** It returns `NULL`. Specifying `LAG(val, 1, 0.0)` returns `0.0` instead of `NULL` for the first record in each partition.


In [53]:
%%sql
CREATE TABLE monthly_mrr (
    month_dt DATE,
    mrr DOUBLE
);
INSERT INTO monthly_mrr VALUES
(DATE '2024-01-01', 50000),
(DATE '2024-02-01', 55000),
(DATE '2024-03-01', 53000),
(DATE '2024-04-01', 62000);

SELECT 
    month_dt,
    mrr,
    LAG(mrr, 1) OVER (ORDER BY month_dt) AS prev_month_mrr,
    LEAD(mrr, 1) OVER (ORDER BY month_dt) AS next_month_mrr,
    ROUND((mrr - LAG(mrr, 1) OVER (ORDER BY month_dt)) * 100.0 / 
          NULLIF(LAG(mrr, 1) OVER (ORDER BY month_dt), 0), 2) AS mom_growth_pct
FROM monthly_mrr
ORDER BY month_dt;

month_dt,mrr,prev_month_mrr,next_month_mrr,mom_growth_pct
2024-01-01,50000.0,NaN,55000.0,NaN
2024-02-01,55000.0,50000.0,53000.0,10.00
2024-03-01,53000.0,55000.0,62000.0,-3.64
2024-04-01,62000.0,53000.0,NaN,16.98


### Q053: FIRST_VALUE, LAST_VALUE, and NTH_VALUE Mechanics

#### Verbal Answer:
* **`FIRST_VALUE(col)`:** Evaluates the value of `col` at the very first row of the window frame.

* **`LAST_VALUE(col)`:** Evaluates the value of `col` at the last row of the window frame.

* **The `LAST_VALUE` Default Frame Trap:** The default window frame when `ORDER BY` is present is `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`. Therefore, `LAST_VALUE(col)` returns the current row's value rather than the partition's final value unless `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` is explicitly specified!

#### Tricky Follow-up:
* **Question:** Why does `LAST_VALUE(salary) OVER (ORDER BY hire_date)` return the employee's own salary instead of the newest hire's salary?

* **Answer:** Because the default frame stops at `CURRENT ROW`. To reach the true end of the partition, you must add `ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING` (or `UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`).


In [54]:
%%sql
CREATE TABLE student_grades (
    student VARCHAR,
    subject VARCHAR,
    score INT
);
INSERT INTO student_grades VALUES
('Alice', 'Math', 95),
('Bob', 'Math', 82),
('Charlie', 'Math', 70),
('Diana', 'Math', 60);

SELECT 
    student,
    score,
    FIRST_VALUE(student) OVER (ORDER BY score DESC) AS top_student,
    -- Without explicit frame, LAST_VALUE only looks up to CURRENT ROW
    LAST_VALUE(student) OVER (
        ORDER BY score DESC 
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS lowest_student,
    NTH_VALUE(student, 2) OVER (
        ORDER BY score DESC 
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS runner_up_student
FROM student_grades
ORDER BY score DESC;

student,score,top_student,lowest_student,runner_up_student
Alice,95,Alice,Diana,Bob
Bob,82,Alice,Diana,Bob
Charlie,70,Alice,Diana,Bob
Diana,60,Alice,Diana,Bob


### Q054: NTILE: Equal-Sized Bucket Segmentation and Quantile Bucketing

#### Verbal Answer:
* **`NTILE(n)`:** Divides the ordered rows in a partition into $n$ approximately equal-sized integer buckets ($1$ to $n$).

* **Remainder Distribution:** If the row count is not evenly divisible by $n$, the extra rows are distributed one-by-one to the lowest bucket numbers first (e.g., 10 rows into 4 buckets produces bucket sizes: 3, 3, 2, 2).

* **Common Use Cases:**
  - Assigning customer RFM (Recency, Frequency, Monetary) quintiles ($1-5$).
  - Decile and Quartile performance reporting.

#### Tricky Follow-up:
* **Question:** How does `NTILE(4)` handle a table with fewer than 4 rows (e.g. 2 rows)?

* **Answer:** It assigns bucket 1 to the first row and bucket 2 to the second row. Buckets 3 and 4 remain empty.


In [55]:
%%sql
CREATE TABLE spend_distribution (
    customer_id INT,
    annual_spend DOUBLE
);
INSERT INTO spend_distribution VALUES
(1, 12000), (2, 8500), (3, 7200), (4, 5100), 
(5, 4300), (6, 2100), (7, 1500), (8, 500);

SELECT 
    customer_id,
    annual_spend,
    NTILE(4) OVER (ORDER BY annual_spend DESC) AS spend_quartile,
    NTILE(2) OVER (ORDER BY annual_spend DESC) AS spend_tier_half
FROM spend_distribution
ORDER BY annual_spend DESC;

customer_id,annual_spend,spend_quartile,spend_tier_half
1,12000.0,1,1
2,8500.0,1,1
3,7200.0,2,1
4,5100.0,2,1
5,4300.0,3,2
6,2100.0,3,2
7,1500.0,4,2
8,500.0,4,2


### Q055: PERCENT_RANK vs CUME_DIST: Cumulative Distribution Calculations

#### Verbal Answer:
* **`PERCENT_RANK()`:** Relative rank of a row evaluated as: $\text{PERCENT\_RANK} = \frac{\text{RANK} - 1}{N - 1}$, where $N$ is total rows in partition. Output ranges strictly from $0.0$ to $1.0$.

* **`CUME_DIST()` (Cumulative Distribution):** Proportion of rows with values less than or equal to the current row's value: $\text{CUME\_DIST} = \frac{\text{Number of rows with value} \le \text{current value}}{N}$. Output ranges strictly from $> 0.0$ to $1.0$.

* **Statistical Distinction:** `PERCENT_RANK` represents relative percentile position (first row is always 0.0), while `CUME_DIST` represents the empirical cumulative distribution function (ECDF).

#### Tricky Follow-up:
* **Question:** For 4 rows `[10, 20, 30, 40]`, what are `PERCENT_RANK` and `CUME_DIST` for the first and last rows?

* **Answer:** First row (10): `PERCENT_RANK = (1-1)/3 = 0.0`, `CUME_DIST = 1/4 = 0.25`. Last row (40): `PERCENT_RANK = (4-1)/3 = 1.0`, `CUME_DIST = 4/4 = 1.0`.


In [56]:
%%sql
CREATE TABLE exam_results (
    candidate VARCHAR,
    score INT
);
INSERT INTO exam_results VALUES
('Alice', 95),
('Bob', 80),
('Charlie', 80),
('David', 50);

SELECT 
    candidate,
    score,
    RANK() OVER (ORDER BY score ASC) AS rnk,
    ROUND(PERCENT_RANK() OVER (ORDER BY score ASC), 4) AS pct_rank,
    ROUND(CUME_DIST() OVER (ORDER BY score ASC), 4) AS cume_distribution
FROM exam_results
ORDER BY score ASC;

candidate,score,rnk,pct_rank,cume_distribution
David,50,1,0.0000,0.25
Bob,80,2,0.3333,0.75
Charlie,80,2,0.3333,0.75
Alice,95,4,1.0000,1.00


### Q056: Running Totals and Cumulative Sums: Partitioned and Ordered

#### Verbal Answer:
* **Cumulative Aggregations:** Adding an `OVER (PARTITION BY ... ORDER BY ...)` clause to `SUM(col)` or `AVG(col)` computes a running cumulative metric per partition.

* **Order of Execution:** The database sorts the rows within each partition according to the `ORDER BY` clause, then streams rows through the aggregate accumulator.

* **Framing Impact:** By default, adding `ORDER BY` changes the window frame to `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`, which aggregates all previous rows up to the current row.

#### Tricky Follow-up:
* **Question:** What happens to `SUM(amount) OVER (ORDER BY amount)` if two rows have the exact same `amount` value?

* **Answer:** With `RANGE` (the default), tied rows are evaluated together, so both tied rows will display the identical combined running sum. Use `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` to force strict row-by-row accumulation.


In [57]:
%%sql
CREATE TABLE ledger (
    tx_id INT,
    account_id VARCHAR,
    tx_date DATE,
    amount DOUBLE
);
INSERT INTO ledger VALUES
(1, 'ACC_1', DATE '2024-01-01', 100),
(2, 'ACC_1', DATE '2024-01-02', 250),
(3, 'ACC_1', DATE '2024-01-03', -50),
(4, 'ACC_2', DATE '2024-01-01', 500),
(5, 'ACC_2', DATE '2024-01-02', 150);

SELECT 
    account_id,
    tx_date,
    amount,
    SUM(amount) OVER (
        PARTITION BY account_id 
        ORDER BY tx_date 
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_balance,
    AVG(amount) OVER (
        PARTITION BY account_id 
        ORDER BY tx_date 
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_avg_tx
FROM ledger
ORDER BY account_id, tx_date;

account_id,tx_date,amount,running_balance,running_avg_tx
ACC_1,2024-01-01,100.0,100.0,100.0
ACC_1,2024-01-02,250.0,350.0,175.0
ACC_1,2024-01-03,-50.0,300.0,100.0
ACC_2,2024-01-01,500.0,500.0,500.0
ACC_2,2024-01-02,150.0,650.0,325.0


### Q057: Top-N Rows Per Group Pattern: Ranking vs QUALIFY vs CTEs

#### Verbal Answer:
* **The Classic SQL Interview Challenge:** "Find the top 2 highest-earning employees in every department."

* **Why WHERE Fails Directly:** Window functions cannot be placed directly inside a `WHERE` clause (`WHERE ROW_NUMBER() <= 2` is illegal) because `WHERE` executes before `WINDOW`.

* **The 3 Solution Patterns:**
  1. **CTE / Subquery:** Wrap the window function in a CTE, then filter `WHERE rn <= 2`.
  2. **QUALIFY Clause:** In Snowflake, BigQuery, DuckDB, Teradata: filter window results directly using `QUALIFY ROW_NUMBER() OVER (...) <= 2`.
  3. **LATERAL Join:** Use `CROSS JOIN LATERAL (SELECT ... LIMIT 2)`.

#### Tricky Follow-up:
* **Question:** Why is the `QUALIFY` clause considered one of the most productive additions to modern SQL?

* **Answer:** It eliminates the need for boiler-plate subqueries and CTE wrappers just to filter on window function ranks or flags.


In [58]:
%%sql
CREATE TABLE staff_salaries (
    dept VARCHAR,
    emp_name VARCHAR,
    salary INT
);
INSERT INTO staff_salaries VALUES
('Engineering', 'Alice', 150000),
('Engineering', 'Bob', 140000),
('Engineering', 'Charlie', 120000),
('Sales', 'Diana', 110000),
('Sales', 'Evan', 105000),
('Sales', 'Frank', 90000);

SELECT 
    dept,
    emp_name,
    salary,
    DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS rank_in_dept
FROM staff_salaries
QUALIFY DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) <= 2
ORDER BY dept, salary DESC;

dept,emp_name,salary,rank_in_dept
Engineering,Alice,150000,1
Engineering,Bob,140000,2
Sales,Diana,110000,1
Sales,Evan,105000,2


### Q058: Named WINDOW Clause: DRY Principles for Multiple Window Expressions

#### Verbal Answer:
* **The Problem:** Queries computing `ROW_NUMBER()`, `SUM()`, `AVG()`, `LAG()` over the identical `PARTITION BY account_id ORDER BY tx_date` repeat the exact same window specification multiple times.

* **Named WINDOW Clause:** Declares a reusable window definition at the end of the query: `WINDOW w AS (PARTITION BY account_id ORDER BY tx_date)`.

* **Benefits:** Reduces SQL duplication (DRY principle), simplifies refactoring, and signals to the optimizer that multiple functions share the exact same partition sort buffer.

#### Tricky Follow-up:
* **Question:** Can you extend a named window specification with custom framing inside the `SELECT` expression?

* **Answer:** Yes! You can write `SELECT SUM(x) OVER (w ROWS BETWEEN 3 PRECEDING AND CURRENT ROW) ... WINDOW w AS (PARTITION BY dept ORDER BY dt)`.


In [59]:
%%sql
CREATE TABLE daily_metrics (
    sensor_id INT,
    reading_dt DATE,
    val DOUBLE
);
INSERT INTO daily_metrics VALUES
(1, DATE '2024-01-01', 10.0),
(1, DATE '2024-01-02', 15.0),
(1, DATE '2024-01-03', 12.0),
(2, DATE '2024-01-01', 20.0),
(2, DATE '2024-01-02', 25.0);

SELECT 
    sensor_id,
    reading_dt,
    val,
    ROW_NUMBER() OVER w AS seq_num,
    LAG(val, 1) OVER w AS prev_val,
    SUM(val) OVER w AS cum_val,
    AVG(val) OVER w AS cum_avg
FROM daily_metrics
WINDOW w AS (PARTITION BY sensor_id ORDER BY reading_dt)
ORDER BY sensor_id, reading_dt;

sensor_id,reading_dt,val,seq_num,prev_val,cum_val,cum_avg
1,2024-01-01,10.0,1,NaN,10.0,10.000000
1,2024-01-02,15.0,2,10.0,25.0,12.500000
1,2024-01-03,12.0,3,15.0,37.0,12.333333
2,2024-01-01,20.0,1,NaN,20.0,20.000000
2,2024-01-02,25.0,2,20.0,45.0,22.500000


### Q059: Window Function Performance: Partition Sorting and Memory Spills

#### Verbal Answer:
* **Partitioning & Sorting Cost:** Evaluating `OVER (PARTITION BY A ORDER BY B)` requires partitioning all rows by `A` (hash partition) and sorting each partition by `B`.

* **Multiple Incompatible Windows:** If a query contains `OVER (PARTITION BY A ORDER BY B)` and `OVER (PARTITION BY X ORDER BY Y)`, the query engine must perform **two distinct sort-and-partition operations**, shuffling the entire dataset across cluster memory twice.

* **Memory Spilling:** If the partition exceeds working memory (`work_mem`), intermediate tuples spill to temporary disk storage, dramatically degrading performance.

#### Tricky Follow-up:
* **Question:** How can you optimize a query with 5 different window functions for maximum execution speed?

* **Answer:** Align their `PARTITION BY` and `ORDER BY` keys to share the exact same sort order whenever analytically possible. This allows the engine to compute all 5 functions in a single stream pass over one sort buffer.


In [60]:
%%sql
CREATE TABLE perf_events AS 
SELECT 
    (i % 100) AS device_id,
    i AS event_id,
    (random() * 100)::DOUBLE AS metric
FROM generate_series(1, 10000) t(i);

EXPLAIN 
SELECT 
    device_id,
    event_id,
    metric,
    AVG(metric) OVER (PARTITION BY device_id ORDER BY event_id) AS running_avg
FROM perf_events;

explain_key,explain_value
physical_plan,"┌───────────────────────────┐\n│ PROJECTION │\n│ ──────────────────── │\n│ #0 │\n│ #1 │\n│ #2 │\n│ #3 │\n│ │\n│ ~10,000 rows │\n└─────────────┬─────────────┘\n┌─────────────┴─────────────┐\n│ WINDOW │\n│ ──────────────────── │\n│ Projections: │\n│ avg(metric) OVER │\n│ (PARTITION BY device_id │\n│ ORDER BY event_id ASC │\n│ NULLS LAST) │\n└─────────────┬─────────────┘\n┌─────────────┴─────────────┐\n│ SEQ_SCAN │\n│ ──────────────────── │\n│ Table: │\n│ memory.main.perf_events │\n│ │\n│ Type: Sequential Scan │\n│ │\n│ Projections: │\n│ device_id │\n│ event_id │\n│ metric │\n│ │\n│ ~10,000 rows │\n└───────────────────────────┘\n"


### Q060: Handling NULLs in ORDER BY Clauses: NULLS FIRST vs NULLS LAST

#### Verbal Answer:
* **Default NULL Sorting Behavior:**
  - In PostgreSQL / Oracle / Snowflake: In `ORDER BY col ASC`, `NULL`s sort **last** (`NULLS LAST`). In `DESC`, `NULL`s sort **first** (`NULLS FIRST`).
  - In MySQL / SQL Server: `NULL`s are treated as smaller than all non-null values (sort first in `ASC`).

* **Explicit Specification:** Always write `ORDER BY col ASC NULLS FIRST` or `ORDER BY col DESC NULLS LAST` explicitly to guarantee identical deterministic behavior across all database engines.

#### Tricky Follow-up:
* **Question:** Why is `ORDER BY discount_pct DESC NULLS LAST` critical when ranking the highest discounts?

* **Answer:** Without `NULLS LAST`, default `DESC` sorting places `NULL`s at Rank 1, causing missing data to masquerade as the top discounts.


In [61]:
%%sql
CREATE TABLE deals (deal_name VARCHAR, discount_pct DOUBLE);
INSERT INTO deals VALUES
('Deal A', 0.20),
('Deal B', NULL),
('Deal C', 0.50),
('Deal D', 0.10);

SELECT 
    deal_name,
    discount_pct,
    ROW_NUMBER() OVER (ORDER BY discount_pct DESC NULLS LAST) AS discount_rank
FROM deals
ORDER BY discount_rank;

deal_name,discount_pct,discount_rank
Deal C,0.5,1
Deal A,0.2,2
Deal D,0.1,3
Deal B,NaN,4


### -------------------------------------------------------------------------
### PILLAR 7: ADVANCED WINDOW FRAMING: ROWS, RANGE & DYNAMIC WINDOWS
### -------------------------------------------------------------------------

### Q061: ROWS vs RANGE vs GROUPS: Exact Framing Specifications

#### Verbal Answer:
* **`ROWS`:** Specifies window boundaries based on physical row counts (e.g., `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` counts exactly 3 physical rows).

* **`RANGE`:** Specifies window boundaries based on logical value offsets relative to the current row's order key (e.g., `RANGE BETWEEN INTERVAL 7 DAY PRECEDING AND CURRENT ROW`).

* **`GROUPS`:** Specifies boundaries based on groups of tied peer rows (e.g., `GROUPS BETWEEN 1 PRECEDING AND CURRENT ROW`).

* **Default Frame Trap:** When `ORDER BY` is specified without an explicit frame clause, ANSI standard SQL defaults to: `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`.

#### Tricky Follow-up:
* **Question:** Why is `ROWS BETWEEN ...` significantly faster than `RANGE BETWEEN ...` for simple running totals in PostgreSQL?

* **Answer:** `ROWS` only requires advancing a physical pointer. `RANGE` requires inspecting row values to check for duplicates / peer groups on every single row step.


In [62]:
%%sql
CREATE TABLE daily_stock (
    trade_date DATE,
    price DOUBLE
);
INSERT INTO daily_stock VALUES
(DATE '2024-01-01', 100),
(DATE '2024-01-02', 105),
(DATE '2024-01-03', 110),
(DATE '2024-01-04', 120),
(DATE '2024-01-05', 115);

SELECT 
    trade_date,
    price,
    -- 3-day moving average using physical ROWS
    AVG(price) OVER (
        ORDER BY trade_date 
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS ma_3day_rows
FROM daily_stock
ORDER BY trade_date;

trade_date,price,ma_3day_rows
2024-01-01,100.0,100.000000
2024-01-02,105.0,102.500000
2024-01-03,110.0,105.000000
2024-01-04,120.0,111.666667
2024-01-05,115.0,115.000000


### Q062: Rolling 7-Day / 30-Day Moving Averages on Irregular Dates

#### Verbal Answer:
* **The Irregular Time-Series Problem:** If transactions occur irregularly (e.g., Jan 1, Jan 2, Jan 10), `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` takes 7 physical rows spanning weeks, corrupting a 7-day moving average.

* **RANGE with Interval Framing:** `RANGE BETWEEN INTERVAL 6 DAY PRECEDING AND CURRENT ROW` dynamically calculates the time boundary based on actual calendar days (`trade_date - 6 days`), regardless of how many or how few transactions occurred.

* **Support:** Native in DuckDB, PostgreSQL, BigQuery, Snowflake, and modern ANSI SQL.

#### Tricky Follow-up:
* **Question:** What happens if no transactions occurred in the preceding 6 days when using `RANGE BETWEEN INTERVAL 6 DAY PRECEDING`?

* **Answer:** The frame encompasses only the current day's row, returning that single day's value as the average without error.


In [63]:
%%sql
CREATE TABLE irregular_events (
    event_time TIMESTAMP,
    revenue DOUBLE
);
INSERT INTO irregular_events VALUES
(TIMESTAMP '2024-01-01 10:00:00', 100.0),
(TIMESTAMP '2024-01-02 11:00:00', 150.0),
(TIMESTAMP '2024-01-02 15:00:00', 50.0),
(TIMESTAMP '2024-01-10 09:00:00', 300.0); -- Gap between Jan 2 and Jan 10;

SELECT 
    event_time,
    revenue,
    SUM(revenue) OVER (
        ORDER BY event_time 
        RANGE BETWEEN INTERVAL 2 DAY PRECEDING AND CURRENT ROW
    ) AS rolling_48hr_revenue
FROM irregular_events
ORDER BY event_time;

event_time,revenue,rolling_48hr_revenue
2024-01-01 10:00:00,100.0,100.0
2024-01-02 11:00:00,150.0,250.0
2024-01-02 15:00:00,50.0,300.0
2024-01-10 09:00:00,300.0,300.0


### Q063: Centered Moving Averages and Symmetric Framing

#### Verbal Answer:
* **Centered Windows:** In signal processing, financial smoothing, and time-series trend decomposition, a centered moving average smooths data using both past and future observations.

* **Framing Syntax:** `ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING` (a 3-point centered window) or `ROWS BETWEEN 3 PRECEDING AND 3 FOLLOWING` (a 7-point centered window).

* **Boundary Edge Handling:** At the first row, only `CURRENT ROW` and `1 FOLLOWING` exist (2 rows total). The engine dynamically adjusts the denominator to 2 rather than dividing by 3.

#### Tricky Follow-up:
* **Question:** Why are centered moving averages prohibited in real-time online feature stores for fraud detection?

* **Answer:** Because `1 FOLLOWING` requires future data that has not yet occurred at event decision time (data leakage / lookahead bias).


In [64]:
%%sql
CREATE TABLE sensor_readings (
    step_id INT,
    val DOUBLE
);
INSERT INTO sensor_readings VALUES
(1, 10.0), (2, 20.0), (3, 100.0), (4, 22.0), (5, 12.0);

SELECT 
    step_id,
    val,
    ROUND(AVG(val) OVER (
        ORDER BY step_id 
        ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
    ), 2) AS centered_smoothed_val
FROM sensor_readings
ORDER BY step_id;

step_id,val,centered_smoothed_val
1,10.0,15.00
2,20.0,43.33
3,100.0,47.33
4,22.0,44.67
5,12.0,17.00


### Q064: Cumulative Distinct Counts in Window Functions: The Workaround

#### Verbal Answer:
* **The ANSI SQL Limitation:** `COUNT(DISTINCT user_id) OVER (ORDER BY event_date)` is **not supported** in most SQL engines (PostgreSQL, SQLite, SQL Server, DuckDB) because maintaining an incremental distinct set across a moving frame is computationally prohibitive.

* **Workaround 1: First Occurrence Flag:** Calculate `ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_date) AS rn`. If `rn = 1`, emit 1 else 0. Then compute `SUM(flag) OVER (ORDER BY event_date)`.

* **Workaround 2: HyperLogLog:** In modern data warehouses (Snowflake, BigQuery), use `HLL_ACCUMULATE` with window aggregation.

#### Tricky Follow-up:
* **Question:** Why is the First Occurrence Flag pattern exact and 100% compliant across all SQL dialects?

* **Answer:** Because every user is counted exactly once on the date they first appeared. The running sum of these '1's gives the exact cumulative count of unique users over time.


In [65]:
%%sql
CREATE TABLE user_logins (
    event_dt DATE,
    user_id INT
);
INSERT INTO user_logins VALUES
(DATE '2024-01-01', 101),
(DATE '2024-01-01', 102),
(DATE '2024-01-02', 101), -- Repeat login
(DATE '2024-01-02', 103),
(DATE '2024-01-03', 104);

WITH first_seen AS (
    SELECT 
        event_dt,
        user_id,
        CASE 
            WHEN ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_dt) = 1 THEN 1 
            ELSE 0 
        END AS is_new_user
    FROM user_logins
)
SELECT 
    event_dt,
    user_id,
    is_new_user,
    SUM(is_new_user) OVER (ORDER BY event_dt, user_id) AS cumulative_unique_users
FROM first_seen
ORDER BY event_dt, user_id;

event_dt,user_id,is_new_user,cumulative_unique_users
2024-01-01,101,1,1.0
2024-01-01,102,1,2.0
2024-01-02,101,0,2.0
2024-01-02,103,1,3.0
2024-01-03,104,1,4.0


### Q065: EXCLUDE Clause in Window Framing: CURRENT ROW, GROUP, TIES, NO OTHERS

#### Verbal Answer:
* **Modern SQL Standard EXCLUDE Clause:** Allows excluding specific rows from within the evaluated window frame:
  - `EXCLUDE NO OTHERS` (default: exclude nothing).
  - `EXCLUDE CURRENT ROW` (removes current row from calculation).
  - `EXCLUDE GROUP` (removes current row and all its peer tied rows).
  - `EXCLUDE TIES` (retains current row, but removes its tied peer rows).

* **Use Case:** Computing leave-one-out averages (average of all peer items in the department *excluding* the employee themselves).

#### Tricky Follow-up:
* **Question:** How did developers compute leave-one-out average before the `EXCLUDE CURRENT ROW` clause?

* **Answer:** Via formula arithmetic: `(SUM(salary) OVER (PARTITION BY dept) - salary) / (COUNT(*) OVER (PARTITION BY dept) - 1)`.


In [66]:
%%sql
CREATE TABLE peer_salaries (
    dept VARCHAR,
    emp VARCHAR,
    salary DOUBLE
);
INSERT INTO peer_salaries VALUES
('Tech', 'Alice', 100),
('Tech', 'Bob', 80),
('Tech', 'Charlie', 60);

SELECT 
    dept,
    emp,
    salary,
    AVG(salary) OVER (
        PARTITION BY dept 
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING 
        EXCLUDE CURRENT ROW
    ) AS avg_peer_salary
FROM peer_salaries;

dept,emp,salary,avg_peer_salary
Tech,Alice,100.0,70.0
Tech,Bob,80.0,80.0
Tech,Charlie,60.0,90.0


### Q066: Forward-Filling Missing Values (LOCF: Last Observation Carried Forward)

#### Verbal Answer:
* **The LOCF Problem:** Sensor feeds, financial prices, and telemetry frequently contain `NULL`s when no state change occurs. We need to forward-fill each `NULL` with the most recent valid non-null value.

* **Method 1: `IGNORE NULLS` (Modern SQL):** `LAST_VALUE(col IGNORE NULLS) OVER (ORDER BY ts ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`.

* **Method 2: Group Partitioning Trick (Classic SQL):** Create a group identifier using `COUNT(col) OVER (ORDER BY ts)`, then use `FIRST_VALUE(col) OVER (PARTITION BY grp_id)`.

#### Tricky Follow-up:
* **Question:** Which major cloud data warehouses support `LAST_VALUE(...) IGNORE NULLS`?

* **Answer:** Snowflake, BigQuery, Databricks Spark SQL, DuckDB, and Oracle natively support `IGNORE NULLS`.


In [67]:
%%sql
CREATE TABLE telemetry (
    ts TIMESTAMP,
    temperature DOUBLE
);
INSERT INTO telemetry VALUES
(TIMESTAMP '2024-01-01 10:00:00', 21.5),
(TIMESTAMP '2024-01-01 10:01:00', NULL),
(TIMESTAMP '2024-01-01 10:02:00', NULL),
(TIMESTAMP '2024-01-01 10:03:00', 22.8),
(TIMESTAMP '2024-01-01 10:04:00', NULL);

SELECT 
    ts,
    temperature AS raw_temp,
    LAST_VALUE(temperature IGNORE NULLS) OVER (
        ORDER BY ts 
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS ffilled_temp
FROM telemetry
ORDER BY ts;

ts,raw_temp,ffilled_temp
2024-01-01 10:00:00,21.5,21.5
2024-01-01 10:01:00,NaN,21.5
2024-01-01 10:02:00,NaN,21.5
2024-01-01 10:03:00,22.8,22.8
2024-01-01 10:04:00,NaN,22.8


### Q067: Running Maxima and Drawdown Calculations in Quantitative Finance

#### Verbal Answer:
* **Peak-to-Date / High Water Mark:** In financial risk management, maximum cumulative price achieved up to time $t$: $\text{HWM}_t = \max_{k \le t} (P_k)$.

* **SQL Implementation:** `MAX(price) OVER (ORDER BY dt ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`.

* **Drawdown Percentage:** The relative loss from the peak: $\text{Drawdown} = \frac{P_t - \text{HWM}_t}{\text{HWM}_t}$. Maximum Drawdown (MDD) is the minimum of this series.

#### Tricky Follow-up:
* **Question:** How do you calculate Maximum Drawdown (MDD) across the entire asset history in SQL?

* **Answer:** Compute the running drawdown series in a CTE, then query `SELECT MIN(drawdown_pct) AS max_drawdown FROM cte`.


In [68]:
%%sql
CREATE TABLE portfolio_nav (
    nav_date DATE,
    nav DOUBLE
);
INSERT INTO portfolio_nav VALUES
(DATE '2024-01-01', 100.0),
(DATE '2024-01-02', 120.0), -- New Peak
(DATE '2024-01-03', 105.0), -- Drawdown from 120
(DATE '2024-01-04', 90.0),  -- Drawdown: (90 - 120)/120 = -25%
(DATE '2024-01-05', 130.0); -- New Peak;

WITH watermarks AS (
    SELECT 
        nav_date,
        nav,
        MAX(nav) OVER (ORDER BY nav_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS high_water_mark
    FROM portfolio_nav
)
SELECT 
    nav_date,
    nav,
    high_water_mark,
    ROUND((nav - high_water_mark) * 100.0 / high_water_mark, 2) AS drawdown_pct
FROM watermarks
ORDER BY nav_date;

nav_date,nav,high_water_mark,drawdown_pct
2024-01-01,100.0,100.0,0.0
2024-01-02,120.0,120.0,0.0
2024-01-03,105.0,120.0,-12.5
2024-01-04,90.0,120.0,-25.0
2024-01-05,130.0,130.0,0.0


### Q068: Ratio to Report: Proportion of Total and Partition Contributions

#### Verbal Answer:
* **Definition:** Computing the percentage of total that each individual row contributes to its parent group or the entire organization.

* **SQL Pattern:** Dividing the row's metric by the unpartitioned / partitioned window sum: `col * 100.0 / SUM(col) OVER (PARTITION BY dept)`.

* **Advantage over GROUP BY Subquery:** Calculates row-level attributes alongside department totals without joining an intermediate aggregate table.

#### Tricky Follow-up:
* **Question:** How can you compute both department share and global company share in the same `SELECT` clause?

* **Answer:** By using two distinct window specifications: `col / SUM(col) OVER (PARTITION BY dept)` and `col / SUM(col) OVER ()`.


In [69]:
%%sql
CREATE TABLE revenue_items (
    region VARCHAR,
    product VARCHAR,
    sales DOUBLE
);
INSERT INTO revenue_items VALUES
('North', 'Laptops', 5000),
('North', 'Accessories', 1000),
('South', 'Laptops', 3000),
('South', 'Accessories', 1000);

SELECT 
    region,
    product,
    sales,
    SUM(sales) OVER (PARTITION BY region) AS region_total,
    ROUND(sales * 100.0 / SUM(sales) OVER (PARTITION BY region), 2) AS pct_of_region,
    ROUND(sales * 100.0 / SUM(sales) OVER (), 2) AS pct_of_company_global
FROM revenue_items
ORDER BY region, sales DESC;

region,product,sales,region_total,pct_of_region,pct_of_company_global
North,Laptops,5000.0,6000.0,83.33,50.0
North,Accessories,1000.0,6000.0,16.67,10.0
South,Laptops,3000.0,4000.0,75.00,30.0
South,Accessories,1000.0,4000.0,25.00,10.0


### Q069: Streaks and Consecutive Event Counters

#### Verbal Answer:
* **The Streak Problem:** Tracking how many consecutive days a user has logged in, or how many consecutive games a sports team has won.

* **Algorithm:**
  1. Identify reset boundaries where the streak breaks (`CASE WHEN status != LAG(status) THEN 1 ELSE 0 END`).
  2. Compute running sum of reset flags to assign a unique `streak_group_id`.
  3. Compute `ROW_NUMBER() OVER (PARTITION BY user_id, streak_group_id ORDER BY event_date)` to generate the consecutive streak counter.

#### Tricky Follow-up:
* **Question:** How do you find the maximum winning streak length in a team's match history?

* **Answer:** Group by `(team, streak_group_id)` where `status = 'WIN'`, compute `COUNT(*) AS streak_len`, and take `MAX(streak_len)`.


In [70]:
%%sql
CREATE TABLE game_results (
    match_id INT,
    team VARCHAR,
    result VARCHAR
);
INSERT INTO game_results VALUES
(1, 'Warriors', 'WIN'),
(2, 'Warriors', 'WIN'),
(3, 'Warriors', 'WIN'),
(4, 'Warriors', 'LOSS'),
(5, 'Warriors', 'WIN'),
(6, 'Warriors', 'WIN');

WITH prev_tagged AS (
    SELECT 
        match_id,
        team,
        result,
        LAG(result) OVER (PARTITION BY team ORDER BY match_id) AS prev_result
    FROM game_results
),
streak_ident AS (
    SELECT 
        match_id,
        team,
        result,
        SUM(CASE WHEN result = prev_result THEN 0 ELSE 1 END) 
            OVER (PARTITION BY team ORDER BY match_id) AS streak_group_id
    FROM prev_tagged
)
SELECT 
    match_id,
    team,
    result,
    ROW_NUMBER() OVER (PARTITION BY team, streak_group_id ORDER BY match_id) AS consecutive_streak_count
FROM streak_ident
ORDER BY match_id;

match_id,team,result,consecutive_streak_count
1,Warriors,WIN,1
2,Warriors,WIN,2
3,Warriors,WIN,3
4,Warriors,LOSS,1
5,Warriors,WIN,1
6,Warriors,WIN,2


### Q070: Weighted Moving Averages (WMA) and Exponential Smoothing in SQL

#### Verbal Answer:
* **Weighted Moving Average:** Assigns higher weights to recent observations (e.g., 3-day WMA with weights 3, 2, 1).

* **SQL Calculation:** `(price * 3 + LAG(price, 1) * 2 + LAG(price, 2) * 1) / 6.0`.

* **Recursive CTE for EMA:** Exponential Moving Average ($EMA_t = lpha P_t + (1 - lpha) EMA_{t-1}$) can be evaluated recursively across sequential step IDs.

#### Tricky Follow-up:
* **Question:** Why is WMA preferred over Simple Moving Average (SMA) in rapid algorithmic trading?

* **Answer:** WMA reduces lag by assigning the greatest weight to the most recent price action, reacting much faster to sudden market regime changes.


In [71]:
%%sql
CREATE TABLE tick_data (
    step INT,
    price DOUBLE
);
INSERT INTO tick_data VALUES
(1, 100.0), (2, 102.0), (3, 105.0), (4, 110.0), (5, 108.0);

SELECT 
    step,
    price,
    -- 3-Step Weighted Moving Average (weights: 3 for t, 2 for t-1, 1 for t-2)
    ROUND(
        (price * 3.0 + 
         COALESCE(LAG(price, 1) OVER (ORDER BY step), price) * 2.0 + 
         COALESCE(LAG(price, 2) OVER (ORDER BY step), price) * 1.0) / 6.0, 
        2
    ) AS wma_3step
FROM tick_data
ORDER BY step;

step,price,wma_3step
1,100.0,100.00
2,102.0,101.33
3,105.0,103.17
4,110.0,107.00
5,108.0,108.17


### -------------------------------------------------------------------------
### PILLAR 8: GAPS AND ISLANDS, SESSIONIZATION & EVENT SEQUENCES
### -------------------------------------------------------------------------

### Q071: The Classic Gaps and Islands Problem: ROW_NUMBER Difference Technique

#### Verbal Answer:
* **Definition:** "Islands" are contiguous sequences of consecutive values (e.g., dates: Jan 1, 2, 3). "Gaps" are missing intervals between islands.

* **The Magic ROW_NUMBER() Difference Algorithm:**
  1. For consecutive integer values (or consecutive dates), compute: `val - ROW_NUMBER() OVER (ORDER BY val)`.
  2. Because both `val` and `ROW_NUMBER()` increment by exactly 1 in a contiguous sequence, their difference (`val - rn`) remains **constant** throughout the entire island!
  3. `GROUP BY (val - rn)` to aggregate each distinct island, taking `MIN(val) AS island_start` and `MAX(val) AS island_end`.

#### Tricky Follow-up:
* **Question:** Why is this difference constant during consecutive streaks?

* **Answer:** Let dates be $D_0+1, D_0+2, D_0+3$ and rank be $1, 2, 3$. The difference $(D_0+k) - k = D_0$, which is completely invariant for all items in that unbroken streak.


In [72]:
%%sql
CREATE TABLE active_days (
    user_id INT,
    active_date DATE
);
INSERT INTO active_days VALUES
(1, DATE '2024-01-01'),
(1, DATE '2024-01-02'),
(1, DATE '2024-01-03'), -- Island 1: Jan 1 to Jan 3 (3 days)
(1, DATE '2024-01-07'),
(1, DATE '2024-01-08'); -- Island 2: Jan 7 to Jan 8 (2 days);

WITH tagged_islands AS (
    SELECT 
        user_id,
        active_date,
        active_date - (ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY active_date) * INTERVAL 1 DAY) AS island_group_key
    FROM active_days
)
SELECT 
    user_id,
    MIN(active_date) AS streak_start,
    MAX(active_date) AS streak_end,
    COUNT(*) AS streak_length_days
FROM tagged_islands
GROUP BY user_id, island_group_key
ORDER BY streak_start;

user_id,streak_start,streak_end,streak_length_days
1,2024-01-01,2024-01-03,3
1,2024-01-07,2024-01-08,2


### Q072: Detecting Missing Value Intervals (Gaps Identification)

#### Verbal Answer:
* **Finding Missing Ranges:** Identify where numbers or dates in a sequence were dropped or skipped.

* **Algorithm using LEAD:**
  1. Compare each row's key with the next row's key: `LEAD(id, 1) OVER (ORDER BY id)`.
  2. If `next_id - current_id > 1`, a gap exists!
  3. The missing range is from `current_id + 1` to `next_id - 1`.

#### Tricky Follow-up:
* **Question:** How do you detect if the very first ID in a table is missing (e.g. sequence starts at 5 instead of 1)?

* **Answer:** Check `MIN(id)` separately against the expected starting anchor (1), or `FULL OUTER JOIN` against a generated sequence series.


In [73]:
%%sql
CREATE TABLE invoice_seq (inv_num INT);
INSERT INTO invoice_seq VALUES (101), (102), (103), (107), (108), (112);

WITH next_steps AS (
    SELECT 
        inv_num,
        LEAD(inv_num, 1) OVER (ORDER BY inv_num) AS next_inv_num
    FROM invoice_seq
)
SELECT 
    inv_num + 1 AS gap_start,
    next_inv_num - 1 AS gap_end,
    (next_inv_num - inv_num - 1) AS missing_count
FROM next_steps
WHERE next_inv_num - inv_num > 1;

gap_start,gap_end,missing_count
104,106,3
109,111,3


### Q073: Web Event Sessionization (30-Minute Inactivity Window)

#### Verbal Answer:
* **The Business Problem:** Group clickstream events into discrete user sessions, where a new session begins if the user is inactive for > 30 minutes.

* **The 2-Step SQL Pattern:**
  1. **Session Boundary Detection:** Compare event timestamp to previous event: `CASE WHEN ts - LAG(ts) > INTERVAL 30 MINUTE OR LAG(ts) IS NULL THEN 1 ELSE 0 END AS is_new_session`.
  2. **Session ID Generation:** Compute running cumulative sum of boundary flags: `SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY ts) AS session_id`.

#### Tricky Follow-up:
* **Question:** Why is `SUM(flag) OVER (...)` preferred over a recursive CTE for sessionization?

* **Answer:** `SUM(flag) OVER` runs in a single parallelized scan ($O(N \log N)$ sort), whereas recursive CTEs execute iteratively row-by-row, running 100x slower on massive clickstreams.


In [74]:
%%sql
CREATE TABLE clickstream (
    user_id INT,
    event_time TIMESTAMP,
    page VARCHAR
);
INSERT INTO clickstream VALUES
(1, TIMESTAMP '2024-01-01 10:00:00', 'Home'),
(1, TIMESTAMP '2024-01-01 10:15:00', 'Pricing'),
(1, TIMESTAMP '2024-01-01 11:00:00', 'Checkout'), -- 45m gap -> New Session 2
(1, TIMESTAMP '2024-01-01 11:05:00', 'Success'),
(2, TIMESTAMP '2024-01-01 09:00:00', 'Home');

WITH flagged_events AS (
    SELECT 
        user_id,
        event_time,
        page,
        CASE 
            WHEN event_time - LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time) > INTERVAL 30 MINUTE 
                 OR LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time) IS NULL 
            THEN 1 
            ELSE 0 
        END AS is_new_session
    FROM clickstream
)
SELECT 
    user_id,
    event_time,
    page,
    SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY event_time) AS session_id
FROM flagged_events
ORDER BY user_id, event_time;

user_id,event_time,page,session_id
1,2024-01-01 10:00:00,Home,1.0
1,2024-01-01 10:15:00,Pricing,1.0
1,2024-01-01 11:00:00,Checkout,2.0
1,2024-01-01 11:05:00,Success,2.0
2,2024-01-01 09:00:00,Home,1.0


### Q074: Merging Overlapping Time Intervals (Interval Collapse)

#### Verbal Answer:
* **The Problem:** Hotel bookings, insurance policies, or server outages often have overlapping intervals (e.g., Jan 1-5 and Jan 3-8). We need to merge them into a single continuous span (Jan 1-8).

* **Algorithm:**
  1. Calculate running maximum of `end_date` up to previous row: `MAX(end_date) OVER (PARTITION BY entity ORDER BY start_date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS prev_max_end`.
  2. If `start_date > prev_max_end`, this record starts a new disjoint interval group (`flag = 1`).
  3. `GROUP BY entity, SUM(flag) OVER (...)` taking `MIN(start_date)` and `MAX(end_date)`.

#### Tricky Follow-up:
* **Question:** What happens if a smaller interval is completely engulfed inside a larger preceding interval?

* **Answer:** Its `start_date <= prev_max_end`, so its flag is `0`. The outer `MAX(end_date)` preserves the overarching boundary correctly.


In [75]:
%%sql
CREATE TABLE room_reservations (
    room_id INT,
    start_date DATE,
    end_date DATE
);
INSERT INTO room_reservations VALUES
(101, DATE '2024-01-01', DATE '2024-01-05'),
(101, DATE '2024-01-03', DATE '2024-01-08'), -- Overlaps with 1st
(101, DATE '2024-01-10', DATE '2024-01-12'); -- Separate interval;

WITH marked_overlaps AS (
    SELECT 
        room_id,
        start_date,
        end_date,
        MAX(end_date) OVER (
            PARTITION BY room_id 
            ORDER BY start_date 
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ) AS max_prev_end
    FROM room_reservations
),
grouped_intervals AS (
    SELECT 
        room_id,
        start_date,
        end_date,
        CASE WHEN start_date <= max_prev_end THEN 0 ELSE 1 END AS is_new_group
    FROM marked_overlaps
),
cluster_ids AS (
    SELECT 
        room_id,
        start_date,
        end_date,
        SUM(is_new_group) OVER (PARTITION BY room_id ORDER BY start_date) AS cluster_id
    FROM grouped_intervals
)
SELECT 
    room_id,
    MIN(start_date) AS merged_start_date,
    MAX(end_date) AS merged_end_date
FROM cluster_ids
GROUP BY room_id, cluster_id
ORDER BY room_id, merged_start_date;

room_id,merged_start_date,merged_end_date
101,2024-01-01,2024-01-08
101,2024-01-10,2024-01-12


### Q075: Funnel Conversion Analysis and Drop-off Rates

#### Verbal Answer:
* **Multi-Step Funnel Tracking:** Measuring the progression and drop-off rate of users moving through ordered sequential stages: `Visit` -> `Add to Cart` -> `Checkout` -> `Purchase`.

* **SQL Pattern:**
  - Track the first timestamp each user completed each stage.
  - Require that each subsequent stage occurred *after* the preceding stage (`t_checkout > t_cart > t_visit`).
  - Aggregate conversion counts: `COUNT(visit_ts)`, `COUNT(cart_ts)`, etc., to compute step conversion rates.

#### Tricky Follow-up:
* **Question:** How do you enforce a 24-hour completion window across the entire funnel?

* **Answer:** Add the predicate `WHERE purchase_ts <= visit_ts + INTERVAL 24 HOUR`.


In [76]:
%%sql
CREATE TABLE funnel_events (
    user_id INT,
    event_name VARCHAR,
    ts TIMESTAMP
);
INSERT INTO funnel_events VALUES
(1, 'visit', TIMESTAMP '2024-01-01 10:00:00'),
(1, 'cart',  TIMESTAMP '2024-01-01 10:05:00'),
(1, 'buy',   TIMESTAMP '2024-01-01 10:10:00'),
(2, 'visit', TIMESTAMP '2024-01-01 11:00:00'),
(2, 'cart',  TIMESTAMP '2024-01-01 11:20:00'),
(3, 'visit', TIMESTAMP '2024-01-01 12:00:00');

WITH user_stages AS (
    SELECT 
        user_id,
        MIN(CASE WHEN event_name = 'visit' THEN ts END) AS visit_time,
        MIN(CASE WHEN event_name = 'cart' THEN ts END) AS cart_time,
        MIN(CASE WHEN event_name = 'buy' THEN ts END) AS buy_time
    FROM funnel_events
    GROUP BY user_id
)
SELECT 
    COUNT(visit_time) AS total_visitors,
    COUNT(CASE WHEN cart_time > visit_time THEN 1 END) AS reached_cart,
    COUNT(CASE WHEN buy_time > cart_time THEN 1 END) AS reached_buy,
    ROUND(COUNT(CASE WHEN cart_time > visit_time THEN 1 END) * 100.0 / COUNT(visit_time), 1) AS visit_to_cart_pct,
    ROUND(COUNT(CASE WHEN buy_time > cart_time THEN 1 END) * 100.0 / NULLIF(COUNT(CASE WHEN cart_time > visit_time THEN 1 END), 0), 1) AS cart_to_buy_pct
FROM user_stages;

total_visitors,reached_cart,reached_buy,visit_to_cart_pct,cart_to_buy_pct
3,2,1,66.7,50.0


### Q076: Time-Spent-on-Page / Event Duration Calculation

#### Verbal Answer:
* **Duration Calculation:** Given a stream of pageviews, calculate how many seconds the user spent on each page before navigating to the next page.

* **SQL Pattern:**
  - `LEAD(event_time) OVER (PARTITION BY user_id, session_id ORDER BY event_time) AS next_page_time`.
  - Duration = `DATEDIFF('second', event_time, next_page_time)`.
  - Cap extreme durations (e.g. `LEAST(duration, 1800)`) to prevent idle tabs from skewing averages.

#### Tricky Follow-up:
* **Question:** What is the duration for the final exit page in a user session?

* **Answer:** It evaluates to `NULL` because `LEAD()` returns `NULL`. In web analytics, exit page duration is either excluded or defaulted to a fixed baseline (e.g., 0s or average bounce duration).


In [77]:
%%sql
CREATE TABLE page_hits (
    user_id INT,
    page VARCHAR,
    hit_time TIMESTAMP
);
INSERT INTO page_hits VALUES
(101, 'Landing_Page', TIMESTAMP '2024-01-01 10:00:00'),
(101, 'Product_A',    TIMESTAMP '2024-01-01 10:02:30'),
(101, 'Checkout',     TIMESTAMP '2024-01-01 10:05:00');

SELECT 
    user_id,
    page,
    hit_time,
    LEAD(hit_time) OVER (PARTITION BY user_id ORDER BY hit_time) AS next_hit_time,
    DATEDIFF('second', hit_time, LEAD(hit_time) OVER (PARTITION BY user_id ORDER BY hit_time)) AS seconds_on_page
FROM page_hits
ORDER BY user_id, hit_time;

user_id,page,hit_time,next_hit_time,seconds_on_page
101,Landing_Page,2024-01-01 10:00:00,2024-01-01 10:02:30,150
101,Product_A,2024-01-01 10:02:30,2024-01-01 10:05:00,150
101,Checkout,2024-01-01 10:05:00,NaT,<NA>


### Q077: Finding Repeat Purchases Within N Days

#### Verbal Answer:
* **Retention Metric:** Measuring the percentage of customers who make a repeat purchase within 30 days of their initial order.

* **SQL Pattern:** Use `LEAD(order_date, 1) OVER (PARTITION BY user_id ORDER BY order_date)` or self-join on `DATEDIFF('day', o1.order_date, o2.order_date) BETWEEN 1 AND 30`.

#### Tricky Follow-up:
* **Question:** Why is the window function `LEAD` approach faster than a self-join for repeat purchase detection?

* **Answer:** Window functions evaluate in a single linear pass $O(N \log N)$ over the table, whereas self-joins generate an $O(N^2)$ join state across multiple orders per customer.


In [78]:
%%sql
CREATE TABLE customer_orders (
    order_id INT,
    user_id INT,
    order_date DATE,
    amount DOUBLE
);
INSERT INTO customer_orders VALUES
(1, 101, DATE '2024-01-01', 50.0),
(2, 101, DATE '2024-01-15', 75.0), -- 14 days later (<= 30 days)
(3, 102, DATE '2024-01-01', 100.0),
(4, 102, DATE '2024-03-01', 120.0); -- 60 days later (> 30 days);

WITH order_lags AS (
    SELECT 
        order_id,
        user_id,
        order_date,
        LEAD(order_date) OVER (PARTITION BY user_id ORDER BY order_date) AS next_order_date,
        DATEDIFF('day', order_date, LEAD(order_date) OVER (PARTITION BY user_id ORDER BY order_date)) AS days_to_next_order
    FROM customer_orders
)
SELECT 
    user_id,
    order_id,
    order_date,
    next_order_date,
    days_to_next_order,
    CASE WHEN days_to_next_order <= 30 THEN 1 ELSE 0 END AS is_30d_repeat_buyer
FROM order_lags;

user_id,order_id,order_date,next_order_date,days_to_next_order,is_30d_repeat_buyer
102,3,2024-01-01,2024-03-01,60,0
102,4,2024-03-01,NaT,<NA>,0
101,1,2024-01-01,2024-01-15,14,1
101,2,2024-01-15,NaT,<NA>,0


### Q078: Cohort Retention Matrix Analysis (M0, M1, M2, ... Retention)

#### Verbal Answer:
* **Cohort Definition:** Grouping users by their signup / first activity month (`cohort_month = DATE_TRUNC('month', first_active_date)`).

* **Activity Tracking:** For every subsequent transaction, calculate `month_number = DATEDIFF('month', cohort_month, DATE_TRUNC('month', activity_date))`.

* **Retention Matrix:** `COUNT(DISTINCT user_id)` grouped by `cohort_month` and `month_number`, pivoted into a classic triangular retention heatmap.

#### Tricky Follow-up:
* **Question:** What is Month 0 (M0) retention by definition?

* **Answer:** 100% (or the total size of the cohort), representing the baseline group of all users who signed up in that initial month.


In [79]:
%%sql
CREATE TABLE user_activity (
    user_id INT,
    activity_date DATE
);
INSERT INTO user_activity VALUES
(1, DATE '2024-01-10'), (1, DATE '2024-02-12'), (1, DATE '2024-03-05'),
(2, DATE '2024-01-20'), (2, DATE '2024-02-15'),
(3, DATE '2024-02-01'), (3, DATE '2024-03-10');

WITH user_cohort AS (
    SELECT 
        user_id,
        DATE_TRUNC('month', MIN(activity_date)) AS cohort_month
    FROM user_activity
    GROUP BY user_id
),
activity_periods AS (
    SELECT 
        a.user_id,
        c.cohort_month,
        DATEDIFF('month', c.cohort_month, DATE_TRUNC('month', a.activity_date)) AS period_month
    FROM user_activity a
    JOIN user_cohort c ON a.user_id = c.user_id
)
SELECT 
    CAST(cohort_month AS DATE) AS cohort,
    COUNT(DISTINCT CASE WHEN period_month = 0 THEN user_id END) AS m0_users,
    COUNT(DISTINCT CASE WHEN period_month = 1 THEN user_id END) AS m1_users,
    COUNT(DISTINCT CASE WHEN period_month = 2 THEN user_id END) AS m2_users
FROM activity_periods
GROUP BY cohort_month
ORDER BY cohort_month;

cohort,m0_users,m1_users,m2_users
2024-01-01,2,2,1
2024-02-01,1,1,0


### Q079: Churn and Reactivation Classification: Active, Churned, Reactivated

#### Verbal Answer:
* **State Classification:**
  - **New:** Active this month for the very first time.
  - **Retained:** Active this month and was active last month.
  - **Reactivated (Resurrected):** Active this month, but was *inactive* last month after having been active in earlier historical months.
  - **Churned:** Was active last month, but zero activity this month.

* **SQL Pattern:** Join monthly active users (`MAU_t`) with `MAU_{t-1}` on `user_id`.

#### Tricky Follow-up:
* **Question:** How do you detect churned users who generated zero rows in the current month?

* **Answer:** Perform a `FULL OUTER JOIN` between Month $T-1$ active users and Month $T$ active users. Rows where `user_id_{T-1} IS NOT NULL` and `user_id_T IS NULL` represent the churned population.


In [80]:
%%sql
CREATE TABLE monthly_activity (
    user_id INT,
    active_month DATE
);
INSERT INTO monthly_activity VALUES
(1, DATE '2024-01-01'), (1, DATE '2024-02-01'), -- Retained
(2, DATE '2024-01-01'),                         -- Churned in Feb
(3, DATE '2024-01-01'), (3, DATE '2024-03-01'), -- Reactivated in March
(4, DATE '2024-02-01');                         -- New in Feb;

WITH jan AS (
    SELECT user_id FROM monthly_activity WHERE active_month = DATE '2024-01-01'
),
feb AS (
    SELECT user_id FROM monthly_activity WHERE active_month = DATE '2024-02-01'
)
SELECT 
    COALESCE(j.user_id, f.user_id) AS user_id,
    CASE 
        WHEN j.user_id IS NOT NULL AND f.user_id IS NOT NULL THEN 'Retained'
        WHEN j.user_id IS NOT NULL AND f.user_id IS NULL THEN 'Churned'
        WHEN j.user_id IS NULL AND f.user_id IS NOT NULL THEN 'New / Reactivated'
    END AS user_status_in_feb
FROM jan j
FULL OUTER JOIN feb f ON j.user_id = f.user_id
ORDER BY user_id;

user_id,user_status_in_feb
1,Retained
2,Churned
3,Churned
4,New / Reactivated


### Q080: Market Basket Analysis: Finding Product Affinities and Co-Purchases

#### Verbal Answer:
* **Affinity Analysis:** Discovering which items are frequently purchased together in the same basket/order (e.g., Beer and Diapers).

* **SQL Pattern:**
  - Self-join the `order_items` table on `order_id`: `o1.order_id = o2.order_id AND o1.item < o2.item`.
  - Condition `o1.item < o2.item` prevents self-matching (`Beer - Beer`) and duplicate symmetric pairs (`(Beer, Diapers)` and `(Diapers, Beer)`).
  - `GROUP BY o1.item, o2.item` and `COUNT(*) AS co_purchase_count`.

#### Tricky Follow-up:
* **Question:** How do you calculate the Support and Confidence of pair (A -> B)?

* **Answer:** $\text{Support}(A, B) = \frac{\text{Count}(A \cap B)}{\text{Total Orders}}$. $\text{Confidence}(A \to B) = \frac{\text{Count}(A \cap B)}{\text{Count}(A)}$.


In [81]:
%%sql
CREATE TABLE order_items (
    order_id INT,
    item_name VARCHAR
);
INSERT INTO order_items VALUES
(1, 'Laptop'), (1, 'Mouse'), (1, 'Keyboard'),
(2, 'Laptop'), (2, 'Mouse'),
(3, 'Mouse'),  (3, 'Keyboard'),
(4, 'Laptop'), (4, 'Monitor');

SELECT 
    a.item_name AS item_a,
    b.item_name AS item_b,
    COUNT(*) AS times_bought_together
FROM order_items a
JOIN order_items b 
    ON a.order_id = b.order_id 
   AND a.item_name < b.item_name
GROUP BY a.item_name, b.item_name
ORDER BY times_bought_together DESC, item_a, item_b;

item_a,item_b,times_bought_together
Keyboard,Mouse,2
Laptop,Mouse,2
Keyboard,Laptop,1
Laptop,Monitor,1


### -------------------------------------------------------------------------
### PILLAR 9: SEMI-STRUCTURED DATA, JSON & NESTED ARRAYS/LISTS IN SQL
### -------------------------------------------------------------------------

### Q081: JSON Path Extraction: ->, ->>, and json_extract Mechanics

#### Verbal Answer:
* **JSON Operators in Modern SQL:**
  - `->`: Extracts a JSON object/sub-tree, returning type `JSON`.
  - `->>` (or `json_extract_string`): Extracts the field as a plain scalar `TEXT` / `VARCHAR` string with unescaped quotes.

* **JSON Path Syntax:** `json_extract(payload, '$.user.address.city')` traverses nested object keys.

#### Tricky Follow-up:
* **Question:** What is the difference between comparing `payload->'age' = 30` vs `payload->>'age' = '30'`?

* **Answer:** `->` preserves JSON formatting (returning JSON scalar integer or string with quotes), while `->>` returns pure text, ensuring deterministic casting to SQL data types.


In [82]:
%%sql
CREATE TABLE raw_payloads (
    id INT,
    payload JSON
);
INSERT INTO raw_payloads VALUES
(1, '{"user": {"name": "Alice", "age": 28, "city": "Seattle"}}'),
(2, '{"user": {"name": "Bob", "age": 35, "city": "Austin"}}');

SELECT 
    id,
    payload->'user'->>'name' AS user_name,
    CAST(payload->'user'->>'age' AS INT) AS user_age,
    payload->'user'->>'city' AS user_city
FROM raw_payloads;

id,user_name,user_age,user_city
1,Alice,28,Seattle
2,Bob,35,Austin


### Q082: UNNEST / EXPLODE: Flattening Arrays and Nested Lists into Rows

#### Verbal Answer:
* **Array Flattening:** Relational engines represent multi-valued attributes as native arrays (`VARCHAR[]` or `JSON`). `UNNEST(array_col)` transforms each element of the array into a separate individual row.

* **Engine Names:**
  - `UNNEST(arr)`: PostgreSQL, BigQuery, DuckDB, Presto/Trino.
  - `EXPLODE(arr)`: Databricks / Apache Spark SQL.
  - `FLATTEN(input => arr)`: Snowflake.

* **Join Semantics:** `CROSS JOIN UNNEST(arr)` drops rows where the array is empty or `NULL`. Use `LEFT JOIN UNNEST(arr) ON TRUE` to preserve parent rows with empty arrays.

#### Tricky Follow-up:
* **Question:** What happens to the row count when unnesting an array of 5 elements across 10 parent rows?

* **Answer:** It expands from 10 rows to $10 \times 5 = 50$ rows.


In [83]:
%%sql
CREATE TABLE user_skills (
    user_id INT,
    skills VARCHAR[]
);
INSERT INTO user_skills VALUES
(101, ['Python', 'SQL', 'DuckDB']),
(102, ['Java', 'C++']),
(103, ['SQL', 'Snowflake']);

SELECT 
    user_id,
    UNNEST(skills) AS skill_name
FROM user_skills
ORDER BY user_id;

user_id,skill_name
101,Python
101,SQL
101,DuckDB
102,Java
102,C++
103,SQL
103,Snowflake


### Q083: JSON Array Flattening and Dynamic Key-Value Pairs

#### Verbal Answer:
* **Parsing Arrays of JSON Objects:** Ingestion logs often contain nested lists of JSON objects (e.g., an order containing `[{"item": "A", "qty": 2}, {"item": "B", "qty": 1}]`).

* **Pipeline Steps:**
  1. Extract the JSON array from the root payload.
  2. `UNNEST` the array into individual JSON object rows.
  3. Extract scalar fields (`->> 'item'`, `->> 'qty'`) from each expanded object.

#### Tricky Follow-up:
* **Question:** How can you aggregate unnested item quantities back to the order level?

* **Answer:** Apply standard `SUM(CAST(item->>'qty' AS INT))` grouped by `order_id` after unnesting.


In [84]:
%%sql
CREATE TABLE raw_json_orders (
    order_id INT,
    items_json JSON
);
INSERT INTO raw_json_orders VALUES
(1, '[{"sku": "SKU_A", "price": 10.0}, {"sku": "SKU_B", "price": 25.0}]'),
(2, '[{"sku": "SKU_C", "price": 50.0}]');

WITH exploded AS (
    SELECT 
        order_id,
        UNNEST(CAST(items_json AS JSON[])) AS single_item
    FROM raw_json_orders
)
SELECT 
    order_id,
    single_item->>'sku' AS sku,
    CAST(single_item->>'price' AS DOUBLE) AS item_price
FROM exploded;

order_id,sku,item_price
1,SKU_A,10.0
1,SKU_B,25.0
2,SKU_C,50.0


### Q084: Array Aggregations, Slicing, and Set Operations on Arrays

#### Verbal Answer:
* **Constructing Arrays (`ARRAY_AGG`):** Collapses multiple rows into an in-memory array type.

* **Array Slicing:** Accessing specific indices: `arr[1]` (1-indexed in SQL) or slicing `arr[1:3]`.

* **Array Functions:**
  - `ARRAY_LENGTH(arr)`: Counts number of elements.
  - `ARRAY_CONTAINS(arr, val)`: Checks membership without unnesting.
  - `ARRAY_DISTINCT(arr)`: Deduplicates elements within the array.

#### Tricky Follow-up:
* **Question:** Why is storing tags as `VARCHAR[]` superior to comma-delimited strings (`'tag1,tag2'`)?

* **Answer:** Arrays provide type safety, support constant-time element lookups (`ARRAY_CONTAINS`), and eliminate fragile string matching bugs (e.g., matching `'tag'` inside `'tag_2'`).


In [85]:
%%sql
CREATE TABLE article_tags (
    article_id INT,
    tag_list VARCHAR[]
);
INSERT INTO article_tags VALUES
(1, ['ai', 'python', 'sql', 'python']),
(2, ['web', 'javascript']);

SELECT 
    article_id,
    tag_list,
    tag_list[1] AS first_tag,
    len(tag_list) AS total_tags,
    list_distinct(tag_list) AS deduped_tags,
    list_contains(tag_list, 'sql') AS has_sql_tag
FROM article_tags;

article_id,tag_list,first_tag,total_tags,deduped_tags,has_sql_tag
1,"[ai, python, sql, python]",ai,4,"[python, ai, sql]",True
2,"[web, javascript]",web,2,"[web, javascript]",False


### Q085: JSON Schema Validation and Error Handling: JSON_VALID

#### Verbal Answer:
* **Validating JSON Integrity:** Raw logs from distributed microservices occasionally contain truncated or corrupted JSON strings.

* **`JSON_VALID(str)`:** Evaluates whether a string is a syntactically valid JSON document, returning `TRUE` or `FALSE`.

* **Dead-Letter Queue Pattern:** Filter out invalid rows before calling parsing functions: `WHERE JSON_VALID(raw_payload) = FALSE` routes malformed payloads directly into an error auditing table.

#### Tricky Follow-up:
* **Question:** What happens if you run `json_extract()` on a string where `JSON_VALID()` is false?

* **Answer:** In strict engines it terminates the query with a parsing exception; in lenient engines it returns `NULL` and logs a warning.


In [86]:
%%sql
CREATE TABLE raw_inbound_stream (stream_id INT, raw_text VARCHAR);
INSERT INTO raw_inbound_stream VALUES
(1, '{"status": "ok", "code": 200}'),
(2, 'corrupted_json_string{'),
(3, '{"status": "error", "code": 500}');

SELECT 
    stream_id,
    raw_text,
    json_valid(raw_text) AS is_valid_json,
    CASE 
        WHEN json_valid(raw_text) THEN json_extract_string(raw_text, '$.status')
        ELSE 'PARSE_ERROR'
    END AS parsed_status
FROM raw_inbound_stream;

stream_id,raw_text,is_valid_json,parsed_status
1,"{""status"": ""ok"", ""code"": 200}",True,ok
2,corrupted_json_string{,False,PARSE_ERROR
3,"{""status"": ""error"", ""code"": 500}",True,error


### Q086: STRUCT and Nested Objects: Composite Typed Columns

#### Verbal Answer:
* **STRUCT / RECORD Types:** Unlike schema-less `JSON`, a `STRUCT` has a strictly defined, typed schema (e.g., `STRUCT(city VARCHAR, zip INT)`).

* **Columnar Storage Advantage:** In Parquet, DuckDB, and BigQuery, `STRUCT` fields are stored in separate, compressed physical columns. Querying `address.city` reads only the city column from disk, whereas JSON forces reading the entire blob!

* **Dot Notation Access:** Access fields directly via dot notation: `user_struct.city`.

#### Tricky Follow-up:
* **Question:** Why is `STRUCT` 10x faster to query in BigQuery/DuckDB than `JSON`?

* **Answer:** Because `STRUCT` allows columnar projection and vectorized SIMD execution without runtime JSON text parsing.


In [87]:
%%sql
CREATE TABLE users_struct (
    user_id INT,
    address STRUCT(street VARCHAR, city VARCHAR, zip INT)
);
INSERT INTO users_struct VALUES
(1, {'street': '123 Main St', 'city': 'Seattle', 'zip': 98101}),
(2, {'street': '456 Oak Rd', 'city': 'Austin', 'zip': 73301});

SELECT 
    user_id,
    address.city AS city,
    address.zip AS zip_code
FROM users_struct
WHERE address.city = 'Seattle';

user_id,city,zip_code
1,Seattle,98101


### Q087: LIST Comprehensions and Higher-Order Lambda Functions in SQL

#### Verbal Answer:
* **Modern Higher-Order SQL Functions:** Modern engines (DuckDB, Spark SQL, Snowflake, Trino) support functional lambda transforms directly on arrays without unnesting.

* **Key Higher-Order Functions:**
  - `list_transform(arr, x -> expr)` / `transform()`: Map over array elements.
  - `list_filter(arr, x -> condition)` / `filter()`: Filter elements meeting predicate.
  - `list_reduce(arr, (acc, x) -> expr)` / `aggregate()`: Fold array into a scalar.

#### Tricky Follow-up:
* **Question:** Why use `list_transform` instead of `UNNEST` + `GROUP BY`?

* **Answer:** It operates in-place in memory per row without generating intermediate row explosions and expensive regrouping hash tables.


In [88]:
%%sql
CREATE TABLE prices_table (
    order_id INT,
    item_prices DOUBLE[]
);
INSERT INTO prices_table VALUES
(1, [10.0, 20.0, 30.0]),
(2, [5.0, 15.0, 50.0]);

SELECT 
    order_id,
    item_prices,
    -- Map: Add 10% tax to each item
    list_transform(item_prices, x -> ROUND(x * 1.10, 2)) AS taxed_prices,
    -- Filter: Keep only items > 15
    list_filter(item_prices, x -> x > 15.0) AS high_ticket_items
FROM prices_table;

order_id,item_prices,taxed_prices,high_ticket_items
1,"[10.0, 20.0, 30.0]","[11.0, 22.0, 33.0]","[20.0, 30.0]"
2,"[5.0, 15.0, 50.0]","[5.5, 16.5, 55.0]",[50.0]


### Q088: Constructing JSON Objects: JSON_OBJECT and JSON_ARRAY

#### Verbal Answer:
* **Serialization to JSON:** Transforming relational relational tables into structured JSON payloads for external APIs or message queues.

* **Functions:**
  - `JSON_OBJECT('key1', val1, 'key2', val2)`: Assembles key-value pairs into a JSON object.
  - `JSON_ARRAY(val1, val2, ...)`: Assembles scalar values into a JSON array.
  - `JSON_GROUP_ARRAY()` / `TO_JSON()`: Serializes entire records or groups.

#### Tricky Follow-up:
* **Question:** How do you package an entire row into a single JSON object in DuckDB/PostgreSQL?

* **Answer:** Use `to_json(row_tuple)` or `row_to_json(t.*)`.


In [89]:
%%sql
CREATE TABLE api_export (
    client_id INT,
    client_name VARCHAR,
    tier VARCHAR
);
INSERT INTO api_export VALUES
(101, 'Acme Corp', 'Enterprise'),
(102, 'StartupHQ', 'Free');

SELECT 
    client_id,
    json_object(
        'id', client_id,
        'name', client_name,
        'subscription', json_object('plan', tier, 'active', TRUE)
    ) AS formatted_api_payload
FROM api_export;

client_id,formatted_api_payload
101,"{""id"":101,""name"":""Acme Corp"",""subscription"":{""plan"":""Enterprise"",""active"":true}}"
102,"{""id"":102,""name"":""StartupHQ"",""subscription"":{""plan"":""Free"",""active"":true}}"


### Q089: Map / Dictionary Data Types in Modern SQL

#### Verbal Answer:
* **MAP Type (`MAP(key_type, value_type)`):** Stores arbitrary key-value pairs where keys are dynamically defined at runtime without schema alterations.

* **Accessing Map Keys:** `map_column['target_key']` returns the associated value in $O(1)$ lookup time.

* **Map Functions:**
  - `MAP_KEYS(m)`: Returns array of all keys.
  - `MAP_VALUES(m)`: Returns array of all values.
  - `MAP_ENTRIES(m)`: Returns array of key-value structs.

#### Tricky Follow-up:
* **Question:** What is the key structural difference between a `MAP` and a `STRUCT`?

* **Answer:** A `STRUCT` requires all field names and types to be fixed in advance at DDL table creation time; a `MAP` accepts arbitrary dynamic runtime keys of homogeneous types.


In [90]:
%%sql
CREATE TABLE app_configs (
    app_id VARCHAR,
    settings MAP(VARCHAR, VARCHAR)
);
INSERT INTO app_configs VALUES
('App_1', map(['theme', 'env', 'version'], ['dark', 'production', 'v2.1'])),
('App_2', map(['theme', 'env'], ['light', 'staging']));

SELECT 
    app_id,
    settings['theme'] AS selected_theme,
    settings['env'] AS deployment_env,
    map_keys(settings) AS all_defined_keys
FROM app_configs;

app_id,selected_theme,deployment_env,all_defined_keys
App_1,dark,production,"[theme, env, version]"
App_2,light,staging,"[theme, env]"


### Q090: GIN Indexes and Inverted Indexing for Semi-Structured Search

#### Verbal Answer:
* **The Full-Scan Bottleneck:** Searching inside a JSON or Array column (`WHERE tags @> ARRAY['python']`) normally forces a full sequential scan of every row in the table.

* **GIN (Generalized Inverted Index):** Builds an inverted index mapping each individual key/element to the list of heap row pointers containing it.

* **Containment Operators:**
  - `@>` (Contains): Checks if left JSON/Array contains right JSON/Array.
  - `?` (Key Exists): Checks if a top-level key exists.

#### Tricky Follow-up:
* **Question:** Why are GIN indexes slower on `INSERT` / `UPDATE` operations than standard B-Tree indexes?

* **Answer:** Because a single JSON document with 50 keys requires updating 50 separate index entries in the GIN structure.


In [91]:
%%sql
CREATE TABLE document_store (
    doc_id INT,
    tags VARCHAR[]
);
INSERT INTO document_store VALUES
(1, ['security', 'finance', 'audit']),
(2, ['marketing', 'social']),
(3, ['security', 'cloud']);

SELECT 
    doc_id,
    tags
FROM document_store
WHERE list_contains(tags, 'security');

doc_id,tags
1,"[security, finance, audit]"
3,"[security, cloud]"


### -------------------------------------------------------------------------
### PILLAR 10: DEDUPLICATION, UPSERT, MERGE & CHANGE DATA CAPTURE (CDC)
### -------------------------------------------------------------------------

### Q091: Row Deduplication: ROW_NUMBER() vs DELETE with CTAS

#### Verbal Answer:
* **The Deduplication Challenge:** A pipeline accidentally ingested duplicate records with the same `(user_id, updated_at)` timestamp.

* **Pattern 1: CTE with `QUALIFY` / `WHERE`:** `SELECT * FROM tbl QUALIFY ROW_NUMBER() OVER (PARTITION BY business_key ORDER BY updated_at DESC) = 1`.

* **Pattern 2: CTAS (Create Table As Select):** In analytical warehouses where `DELETE` is slow or unsupported, write the deduplicated output into a new table and atomic swap: `CREATE TABLE clean_tbl AS SELECT ... QUALIFY rn = 1; ALTER TABLE clean_tbl SWAP WITH raw_tbl;`.

#### Tricky Follow-up:
* **Question:** Why is `DELETE FROM tbl WHERE id IN (SELECT id FROM ...)` dangerous on massive 100M-row tables?

* **Answer:** Row-by-row `DELETE` generates massive transaction undo/WAL logs and causes heavy table locking. Rebuilding via CTAS is a fast sequential streaming append operation.


In [92]:
%%sql
CREATE TABLE raw_ingest (
    order_id INT,
    customer VARCHAR,
    amount DOUBLE,
    received_at TIMESTAMP
);
INSERT INTO raw_ingest VALUES
(101, 'Alice', 99.0, TIMESTAMP '2024-01-01 10:00:00'),
(101, 'Alice', 99.0, TIMESTAMP '2024-01-01 10:00:05'), -- Duplicate retry
(102, 'Bob',   50.0, TIMESTAMP '2024-01-01 10:01:00');

SELECT 
    order_id,
    customer,
    amount,
    received_at
FROM raw_ingest
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY order_id 
    ORDER BY received_at DESC
) = 1
ORDER BY order_id;

order_id,customer,amount,received_at
101,Alice,99.0,2024-01-01 10:00:05
102,Bob,50.0,2024-01-01 10:01:00


### Q092: MERGE INTO / UPSERT: Architecture and Atomic Synchronization

#### Verbal Answer:
* **`MERGE INTO target USING source ON target.id = source.id`:** ANSI standard atomic operation that reconciles two tables:
  - `WHEN MATCHED THEN UPDATE SET ...`
  - `WHEN NOT MATCHED THEN INSERT ...`
  - `WHEN NOT MATCHED BY SOURCE THEN DELETE` (optional).

* **Atomicity:** Guarantees that either all updates/inserts succeed or the entire batch rolls back, preventing partial sync states.

#### Tricky Follow-up:
* **Question:** What happens if the source table contains duplicate join keys matching a single target row in `MERGE INTO`?

* **Answer:** The engine throws a fatal runtime error: `'Target row cannot match multiple source rows in MERGE'` to prevent non-deterministic updates.


In [93]:
%%sql
CREATE TABLE target_inventory (
    sku VARCHAR PRIMARY KEY,
    qty INT,
    price DOUBLE
);
CREATE TABLE daily_updates (
    sku VARCHAR,
    qty INT,
    price DOUBLE
);

INSERT INTO target_inventory VALUES
('SKU_1', 100, 10.0),
('SKU_2', 50,  20.0);

INSERT INTO daily_updates VALUES
('SKU_2', 40,  22.0), -- Price/Qty updated
('SKU_3', 200, 15.0); -- Brand new item;

INSERT OR REPLACE INTO target_inventory 
SELECT * FROM daily_updates;

SELECT * FROM target_inventory ORDER BY sku;

sku,qty,price
SKU_1,100,10.0
SKU_2,40,22.0
SKU_3,200,15.0


### Q093: Slowly Changing Dimensions (SCD Type 1 vs Type 2 vs Type 3)

#### Verbal Answer:
* **SCD Type 1 (Overwrite):** Updates existing record in place. Historical data is lost forever.

* **SCD Type 2 (Historical Tracking via Validity Windows):** Retains full history by closing out the old row (`valid_to = current_ts`, `is_current = FALSE`) and inserting a new row (`valid_from = current_ts`, `valid_to = NULL`, `is_current = TRUE`).

* **SCD Type 3 (Previous Column):** Adds an extra column (e.g., `previous_region`) to preserve only the immediately preceding state.

#### Tricky Follow-up:
* **Question:** Why is a surrogate key (e.g. `dim_customer_sk`) mandatory for SCD Type 2 tables instead of relying on `customer_id`?

* **Answer:** Because a single `customer_id` will now have multiple rows in the dimension table (one for each historical revision), violating primary key uniqueness.


In [94]:
%%sql
CREATE TABLE dim_customer_scd2 (
    cust_sk INT,
    cust_id INT,
    tier VARCHAR,
    valid_from DATE,
    valid_to DATE,
    is_current BOOLEAN
);
INSERT INTO dim_customer_scd2 VALUES
(1, 101, 'Silver', DATE '2023-01-01', DATE '2023-06-30', FALSE),
(2, 101, 'Gold',   DATE '2023-07-01', NULL,             TRUE);

SELECT 
    cust_id,
    tier,
    'Point-in-Time: March 2023' AS query_type
FROM dim_customer_scd2
WHERE cust_id = 101 AND DATE '2023-03-15' >= valid_from AND (DATE '2023-03-15' < valid_to OR valid_to IS NULL)

UNION ALL

SELECT 
    cust_id,
    tier,
    'Current State' AS query_type
FROM dim_customer_scd2
WHERE cust_id = 101 AND is_current = TRUE;

cust_id,tier,query_type
101,Silver,Point-in-Time: March 2023
101,Gold,Current State


### Q094: Change Data Capture (CDC): Debezium Streams and Operation Codes (I, U, D)

#### Verbal Answer:
* **CDC Architecture:** Debezium or AWS DMS reads database transaction logs (WAL/Binlog) and streams events as JSON messages with operation flags:
  - `c` / `I` = INSERT
  - `u` / `U` = UPDATE
  - `d` / `D` = DELETE

* **Materializing Current Snapshot:** Filter out deletes and select the most recent update per primary key using `ROW_NUMBER() OVER (PARTITION BY pk ORDER BY cdc_timestamp DESC)`.

#### Tricky Follow-up:
* **Question:** How do you accurately model soft-deleted records in downstream data marts from a CDC feed?

* **Answer:** When the latest CDC operation is `D`, insert a tombstone record with `is_deleted = TRUE` and filter out deleted records in analytical reporting views.


In [95]:
%%sql
CREATE TABLE cdc_stream (
    event_id INT,
    user_id INT,
    email VARCHAR,
    op_code VARCHAR, -- 'I', 'U', 'D'
    cdc_ts TIMESTAMP
);
INSERT INTO cdc_stream VALUES
(1, 101, 'alice@old.com', 'I', TIMESTAMP '2024-01-01 10:00:00'),
(2, 101, 'alice@new.com', 'U', TIMESTAMP '2024-01-01 10:05:00'),
(3, 102, 'bob@co.com',    'I', TIMESTAMP '2024-01-01 10:01:00'),
(4, 102, 'bob@co.com',    'D', TIMESTAMP '2024-01-01 10:20:00'); -- Bob was deleted;

WITH latest_cdc AS (
    SELECT 
        user_id,
        email,
        op_code,
        cdc_ts,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY cdc_ts DESC) AS rn
    FROM cdc_stream
)
SELECT 
    user_id,
    email,
    op_code AS last_op
FROM latest_cdc
WHERE rn = 1 AND op_code != 'D'; -- Filter out deleted entities;

Query Executed Successfully.


### Q095: Idempotent Pipeline Design: Preventing Duplicate State on Retries

#### Verbal Answer:
* **Idempotency Definition:** An ETL execution is **idempotent** if running it multiple times with the same input produces the exact same final system state ($f(f(x)) = f(x)$).

* **Anti-Pattern (Non-Idempotent):** Direct `INSERT INTO target SELECT ... FROM raw_today;`. If the airflflow task fails midway and retries, duplicate rows are created.

* **Idempotent Standards:**
  - `DELETE FROM target WHERE partition_date = '2024-01-01'` followed by `INSERT INTO`.
  - `INSERT OVERWRITE target PARTITION (dt = '2024-01-01') SELECT ...`.
  - `MERGE INTO` keyed on primary business keys.

#### Tricky Follow-up:
* **Question:** Why is `INSERT OVERWRITE` preferred over `DELETE` + `INSERT` in BigQuery/Snowflake?

* **Answer:** `INSERT OVERWRITE` is executed as an atomic metadata partition swap. If the incoming query fails during calculation, the old partition remains intact without data loss or downtime.


In [96]:
%%sql
CREATE TABLE fact_daily_sales (
    sale_date DATE,
    total_amount DOUBLE
);
INSERT INTO fact_daily_sales VALUES (DATE '2024-01-01', 5000.0);

DELETE FROM fact_daily_sales WHERE sale_date = DATE '2024-01-01';
INSERT INTO fact_daily_sales VALUES (DATE '2024-01-01', 5200.0); -- Revised total;

SELECT * FROM fact_daily_sales;

sale_date,total_amount
2024-01-01,5200.0


### Q096: Surrogate Keys vs Natural Keys: MD5 / SHA-256 Hashing Strategies

#### Verbal Answer:
* **Natural Key:** Business-assigned identifier (e.g. email, passport number). Prone to format changes, casing issues, or privacy compliance (GDPR deletion).

* **Surrogate Key:** System-assigned synthetic primary key (e.g. integer sequence or deterministic hash).

* **Hash Surrogate Keys (dbt / Modern DWH Standard):** Computing `MD5(CONCAT_WS('||', tenant_id, user_id, start_date))` creates a deterministic 32-character hexadecimal key across distributed parallel nodes without coordinating global autoincrement locks.

#### Tricky Follow-up:
* **Question:** What is the probability of MD5 hash collision when generating surrogate keys?

* **Answer:** With 128-bit hashes, collision probability is virtually zero ($< 10^{-15}$) for billions of rows according to the birthday paradox.


In [97]:
%%sql
CREATE TABLE raw_leads (
    org_code VARCHAR,
    lead_email VARCHAR,
    event_dt DATE
);
INSERT INTO raw_leads VALUES
('ORG_US', 'ALICE@co.com', DATE '2024-01-01'),
('ORG_EU', 'bob@co.de',     DATE '2024-01-01');

SELECT 
    -- Normalized deterministic surrogate key
    md5(LOWER(org_code) || '||' || LOWER(lead_email) || '||' || CAST(event_dt AS VARCHAR)) AS lead_surrogate_key,
    org_code,
    lead_email,
    event_dt
FROM raw_leads;

lead_surrogate_key,org_code,lead_email,event_dt
b34b979eb4476ba8b3335b54e719b68c,ORG_US,ALICE@co.com,2024-01-01
5cfe1d22a4e22bf4c9d8e19c3d578fe4,ORG_EU,bob@co.de,2024-01-01


### Q097: Tombstones and Soft Delete Recovery Patterns

#### Verbal Answer:
* **Soft Delete:** Instead of executing hard `DELETE FROM users WHERE id = 5`, set `deleted_at = CURRENT_TIMESTAMP` and `is_deleted = TRUE`.

* **Benefits:** Preserves referential integrity for historic reporting, allows compliance audits, and enables instantaneous accidental-deletion recovery.

* **Reporting Layer Best Practice:** Build a standard secure view `v_active_users AS SELECT * FROM users WHERE is_deleted = FALSE` to prevent downstream analysts from accidentally querying soft-deleted rows.

#### Tricky Follow-up:
* **Question:** How can you enforce unique constraints (e.g., unique active email) when soft-deleted rows exist?

* **Answer:** Create a Partial Unique Index: `CREATE UNIQUE INDEX unq_active_email ON users(email) WHERE is_deleted = FALSE`.


In [98]:
%%sql
CREATE TABLE users_table (
    user_id INT,
    email VARCHAR,
    deleted_at TIMESTAMP
);
INSERT INTO users_table VALUES
(1, 'active@co.com', NULL),
(2, 'deleted@co.com', TIMESTAMP '2024-01-10 12:00:00');

SELECT 
    user_id,
    email,
    CASE WHEN deleted_at IS NOT NULL THEN TRUE ELSE FALSE END AS is_deleted
FROM users_table
WHERE deleted_at IS NULL;

user_id,email,is_deleted
1,active@co.com,False


### Q098: Reconciling Source vs Target Tables: Full Outer Join Audits

#### Verbal Answer:
* **Data Quality Reconciliation:** Auditing whether a target data mart matches the upstream transactional source system exactly.

* **Reconciliation Pattern:**
  - `FULL OUTER JOIN source s ON s.id = t.id`.
  - Check for missing rows on left (`s.id IS NULL -> Ghost record in target`).
  - Check for missing rows on right (`t.id IS NULL -> Dropped record in target`).
  - Check for value discrepancies (`s.amount != t.amount -> Corrupted transformation`).

#### Tricky Follow-up:
* **Question:** What summary metric immediately tells an executive whether source and target are 100% in sync?

* **Answer:** `COUNT(CASE WHEN s.id IS NULL OR t.id IS NULL OR s.hash_diff != t.hash_diff THEN 1 END) AS discrepancy_count` (which must equal 0).


In [99]:
%%sql
CREATE TABLE oltp_source (id INT, amount DOUBLE);
CREATE TABLE dwh_target  (id INT, amount DOUBLE);

INSERT INTO oltp_source VALUES (1, 100.0), (2, 200.0), (3, 300.0);
INSERT INTO dwh_target  VALUES (1, 100.0), (2, 199.0), (4, 400.0); -- id 2 corrupted, id 3 missing, id 4 orphan;

SELECT 
    COALESCE(s.id, t.id) AS entity_id,
    s.amount AS src_amount,
    t.amount AS tgt_amount,
    CASE 
        WHEN s.id IS NULL THEN 'ORPHAN_IN_TARGET'
        WHEN t.id IS NULL THEN 'MISSING_FROM_TARGET'
        WHEN s.amount != t.amount THEN 'VALUE_MISMATCH'
        ELSE 'MATCH'
    END AS audit_status
FROM oltp_source s
FULL OUTER JOIN dwh_target t ON s.id = t.id
ORDER BY entity_id;

entity_id,src_amount,tgt_amount,audit_status
1,100.0,100.0,MATCH
2,200.0,199.0,VALUE_MISMATCH
3,300.0,NaN,MISSING_FROM_TARGET
4,NaN,400.0,ORPHAN_IN_TARGET


### Q099: Watermarking and High-Water Mark Incremental Loading

#### Verbal Answer:
* **Incremental Ingestion:** Instead of re-reading a 500GB table daily, query only records created or updated since the last pipeline run timestamp (the High-Water Mark).

* **Query Pattern:** `SELECT * FROM source WHERE updated_at > :last_successful_pipeline_hwm;`.

* **Clock Skew Protection:** Because distributed servers may have slight clock drift, always overlap the window slightly (`updated_at >= :hwm - INTERVAL 5 MINUTE`) and deduplicate downstream.

#### Tricky Follow-up:
* **Question:** What happens if a source transaction commits at `10:00:01` with an `updated_at` of `09:59:58`?

* **Answer:** It would be skipped by a naive watermark. This is known as the 'delayed commit anomaly'. Overlapping intervals or transactional sequence IDs prevent this data loss.


In [100]:
%%sql
CREATE TABLE raw_events_stream (
    event_id INT,
    event_name VARCHAR,
    updated_at TIMESTAMP
);
INSERT INTO raw_events_stream VALUES
(1, 'Signup',  TIMESTAMP '2024-01-01 10:00:00'),
(2, 'Login',   TIMESTAMP '2024-01-01 10:05:00'),
(3, 'Upgrade', TIMESTAMP '2024-01-01 10:15:00');

SELECT 
    event_id,
    event_name,
    updated_at
FROM raw_events_stream
WHERE updated_at > TIMESTAMP '2024-01-01 10:02:00'
ORDER BY updated_at;

event_id,event_name,updated_at
2,Login,2024-01-01 10:05:00
3,Upgrade,2024-01-01 10:15:00


### Q100: Late-Arriving Dimensions and Inferred Member Handling

#### Verbal Answer:
* **The Late-Arriving Dimension Problem:** A fact table transaction arrives today with `customer_id = 999`, but the customer dimension record has not synced yet from CRM due to an API delay.

* **The Anti-Pattern:** Dropping the fact record or failing the ETL pipeline.

* **The Production Solution (Inferred Members):**
  - Insert a placeholder row into `dim_customer` with `customer_id = 999`, `name = 'Inferred / Unknown'`, and `is_inferred = TRUE`.
  - Successfully load the fact record referencing this inferred surrogate key.
  - When the CRM dimension record finally arrives tomorrow, update the placeholder record with actual customer attributes and toggle `is_inferred = FALSE`.

#### Tricky Follow-up:
* **Question:** Why is preserving the fact record with an inferred dimension crucial for financial accounting?

* **Answer:** Because daily financial revenue totals must remain 100% balanced and complete, even if specific customer demographic attributes are temporarily delayed.


In [101]:
%%sql
CREATE TABLE dim_cust (
    cust_id INT PRIMARY KEY,
    cust_name VARCHAR,
    is_inferred BOOLEAN
);
CREATE TABLE fact_orders (
    order_id INT,
    cust_id INT,
    amount DOUBLE
);

-- Dimension only has customer 1
INSERT INTO dim_cust VALUES (1, 'Alice Corp', FALSE);

-- Order arrives for customer 2 (late arriving)
INSERT INTO fact_orders VALUES (101, 1, 500.0), (102, 2, 350.0);

INSERT INTO dim_cust 
SELECT DISTINCT f.cust_id, 'UNKNOWN (INFERRED)', TRUE 
FROM fact_orders f 
WHERE NOT EXISTS (SELECT 1 FROM dim_cust d WHERE d.cust_id = f.cust_id);

SELECT 
    f.order_id,
    f.amount,
    d.cust_id,
    d.cust_name,
    d.is_inferred
FROM fact_orders f
JOIN dim_cust d ON f.cust_id = d.cust_id
ORDER BY f.order_id;

order_id,amount,cust_id,cust_name,is_inferred
101,500.0,1,Alice Corp,False
102,350.0,2,UNKNOWN (INFERRED),True


## =========================================================================
## SENIORITY LEVEL: STAFF / LEAD DATA SCIENTIST / QUANT / DATA ARCHITECT
## =========================================================================

### -------------------------------------------------------------------------
### PILLAR 11: QUERY OPTIMIZATION, EXPLAIN PLANS & INDEXING MECHANICS
### -------------------------------------------------------------------------

### Q101: EXPLAIN ANALYZE: Physical Operators, Cost Estimation, and Actual Runtimes

#### Verbal Answer:
* **EXPLAIN vs EXPLAIN ANALYZE:**
  - `EXPLAIN`: Shows the optimizer's estimated execution plan, cost units, and predicted row cardinalities based on table statistics. Does *not* execute the query.
  - `EXPLAIN ANALYZE`: Actually executes the query, recording true wall-clock time, actual memory usage, and exact row counts per physical operator.

* **Key Plan Metrics:**
  - **Estimated vs Actual Rows:** Large discrepancies (e.g. optimizer predicted 10 rows but got 10M rows) signal stale statistics, leading to disastrous join algorithm choices.
  - **Execution Time Bottlenecks:** Pinpoints the exact node (e.g. Hash Join vs Seq Scan) consuming the majority of total query duration.

#### Tricky Follow-up:
* **Question:** Why is running `EXPLAIN ANALYZE` on a `DELETE` or `UPDATE` statement dangerous in production?

* **Answer:** Because `EXPLAIN ANALYZE` actually executes the statement! It will modify or delete production data unless wrapped in a transaction block with an explicit `ROLLBACK`.


In [102]:
%%sql
CREATE TABLE orders_opt AS 
SELECT 
    i AS order_id, 
    (i % 50) AS customer_id, 
    (random() * 1000)::DOUBLE AS amount 
FROM generate_series(1, 10000) t(i);

EXPLAIN ANALYZE 
SELECT customer_id, SUM(amount) AS total 
FROM orders_opt 
WHERE amount > 500 
GROUP BY customer_id;

explain_key,explain_value
analyzed_plan,"┌─────────────────────────────────────┐\n│┌───────────────────────────────────┐│\n││ Query Profiling Information ││\n│└───────────────────────────────────┘│\n└─────────────────────────────────────┘\nEXPLAIN ANALYZE SELECT customer_id, SUM(amount) AS total FROM orders_opt WHERE amount > 500 GROUP BY customer_id\n┌────────────────────────────────────────────────┐\n│┌──────────────────────────────────────────────┐│\n││ Total Time: 0.0022s ││\n│└──────────────────────────────────────────────┘│\n└────────────────────────────────────────────────┘\n┌───────────────────────────┐\n│ QUERY │\n└─────────────┬─────────────┘\n┌─────────────┴─────────────┐\n│ EXPLAIN_ANALYZE │\n│ ──────────────────── │\n│ │\n│ 0 rows │\n│ 0.00s │\n└─────────────┬─────────────┘\n┌─────────────┴─────────────┐\n│ PROJECTION │\n│ ──────────────────── │\n│__internal_decompress_integ│\n│ ral_bigint(#0, 0) │\n│ #1 │\n│ │\n│ │\n│ │\n│ 50 rows │\n│ 0.00s │\n└─────────────┬─────────────┘\n┌─────────────┴─────────────┐\n│ PERFECT_HASH_GROUP_BY │\n│ ──────────────────── │\n│ Groups: #0 │\n│ Aggregates: sum(#1) │\n│ │\n│ │\n│ │\n│ 50 rows │\n│ 0.00s │\n└─────────────┬─────────────┘\n┌─────────────┴─────────────┐\n│ PROJECTION │\n│ ──────────────────── │\n│ customer_id │\n│ amount │\n│ │\n│ │\n│ │\n│ 4,937 rows │\n│ 0.00s │\n└─────────────┬─────────────┘\n┌─────────────┴─────────────┐\n│ PROJECTION │\n│ ──────────────────── │\n│ #0 │\n│__internal_compress_integra│\n│ l_utinyint(#1, 0) │\n│ │\n│ │\n│ │\n│ 4,937 rows │\n│ 0.00s │\n└─────────────┬─────────────┘\n┌─────────────┴─────────────┐\n│ TABLE_SCAN │\n│ ──────────────────── │\n│ Table: │\n│ memory.main.orders_opt │\n│ │\n│ Type: Sequential Scan │\n│ │\n│ Projections: │\n│ amount │\n│ customer_id │\n│ │\n│ Filters: amount>500.0 │\n│ │\n│ │\n│ │\n│ 4,937 rows │\n│ 0.00s │\n└───────────────────────────┘\n"


### Q102: Join Algorithms: Nested Loop, Hash Join, and Sort-Merge Join

#### Verbal Answer:
* **Nested Loop Join:**
  - *Mechanism:* For every row in the outer table, scans the inner table (or searches an inner index).
  - *Complexity:* $O(N \times M)$ (or $O(N \log M)$ with index).
  - *Best for:* Small outer table (< 100 rows) joining to an indexed inner table, or non-equi joins.

* **Hash Join:**
  - *Mechanism:* Builds an in-memory hash table on the smaller build relation, then streams and probes the larger probe relation.
  - *Complexity:* $O(N + M)$ time, $O(N)$ memory.
  - *Best for:* Large unsorted tables with equi-join predicates (`=`).

* **Sort-Merge Join:**
  - *Mechanism:* Sorts both relations on join keys (or leverages existing ordered indexes), then merges them sequentially in lock-step.
  - *Complexity:* $O(N \log N + M \log M)$ sort time, $O(N + M)$ merge time.
  - *Best for:* Very large tables already sorted on disk, or when memory is too small to hold a hash table.

#### Tricky Follow-up:
* **Question:** What causes a Hash Join to degrade dramatically into Grace Hash Join (disk spilling)?

* **Answer:** When the build table's hash map exceeds available working memory (`work_mem`), forcing the engine to partition and spill hash buckets to disk, incurring severe I/O penalties.


In [103]:
%%sql
CREATE TABLE t_small (id INT, val VARCHAR);
CREATE TABLE t_large (id INT, score DOUBLE);

INSERT INTO t_small SELECT i, 'name_' || i FROM generate_series(1, 10) t(i);
INSERT INTO t_large SELECT (i % 10), random() FROM generate_series(1, 10000) t(i);

EXPLAIN 
SELECT s.val, AVG(l.score) 
FROM t_small s 
JOIN t_large l ON s.id = l.id 
GROUP BY s.val;

explain_key,explain_value
physical_plan,"┌───────────────────────────┐\n│ PROJECTION │\n│ ──────────────────── │\n│__internal_decompress_strin│\n│ g(#0) │\n│ #1 │\n│ │\n│ ~10 rows │\n└─────────────┬─────────────┘\n┌─────────────┴─────────────┐\n│ HASH_GROUP_BY │\n│ ──────────────────── │\n│ Groups: #0 │\n│ Aggregates: avg(#1) │\n│ │\n│ ~10 rows │\n└─────────────┬─────────────┘\n┌─────────────┴─────────────┐\n│ PROJECTION │\n│ ──────────────────── │\n│ val │\n│ score │\n│ │\n│ ~9,090 rows │\n└─────────────┬─────────────┘\n┌─────────────┴─────────────┐\n│ PROJECTION │\n│ ──────────────────── │\n│ #0 │\n│__internal_compress_string_│\n│ ubigint(#1) │\n│ │\n│ ~9,090 rows │\n└─────────────┬─────────────┘\n┌─────────────┴─────────────┐\n│ HASH_JOIN │\n│ ──────────────────── │\n│ Join Type: INNER │\n│ Conditions: id = id ├──────────────┐\n│ │ │\n│ ~9,090 rows │ │\n└─────────────┬─────────────┘ │\n┌─────────────┴─────────────┐┌─────────────┴─────────────┐\n│ SEQ_SCAN ││ SEQ_SCAN │\n│ ──────────────────── ││ ──────────────────── │\n│ Table: ││ Table: │\n│ memory.main.t_large ││ memory.main.t_small │\n│ ││ │\n│ Type: Sequential Scan ││ Type: Sequential Scan │\n│ ││ │\n│ Projections: ││ Projections: │\n│ id ││ id │\n│ score ││ val │\n│ ││ │\n│ Filters: id>=1 ││ Filters: id<=9 │\n│ ││ │\n│ ~10,000 rows ││ ~10 rows │\n└───────────────────────────┘└───────────────────────────┘\n"


### Q103: B-Tree vs Hash vs Bitmap vs Clustered Indexes

#### Verbal Answer:
* **B-Tree Index:** Self-balancing search tree supporting exact lookups ($O(\log N)$), prefix searches (`LIKE 'abc%'`), range queries (`BETWEEN 10 AND 50`), and sorted ordering (`ORDER BY`).

* **Hash Index:** Provides $O(1)$ exact equality lookups (`WHERE id = 42`). Cannot support range queries or ordering.

* **Bitmap Index:** Represents distinct values as binary bit arrays. Extremely fast for low-cardinality columns (e.g. `Gender`, `State`) and combining multiple boolean filters (`AND` / `OR` via bitwise operations) in OLAP engines.

* **Clustered Index:** Physically reorders the actual rows on disk to match the index order (e.g., Primary Key in MySQL InnoDB). Only one clustered index can exist per table.

#### Tricky Follow-up:
* **Question:** Why are Bitmap indexes avoided in high-concurrency OLTP databases with frequent INSERT/UPDATE operations?

* **Answer:** Updating a single row requires locking the entire bitmap vector for that category, which locks thousands of other rows and creates severe concurrency bottlenecks.


In [104]:
%%sql
CREATE TABLE index_demo (
    id INT PRIMARY KEY,
    category VARCHAR,
    created_at TIMESTAMP
);
INSERT INTO index_demo VALUES
(1, 'Tech', TIMESTAMP '2024-01-01 10:00:00'),
(2, 'Finance', TIMESTAMP '2024-01-02 11:00:00');

PRAGMA table_info('index_demo');

cid,name,type,notnull,dflt_value,pk
0,id,INTEGER,True,None,True
1,category,VARCHAR,False,None,False
2,created_at,TIMESTAMP,False,None,False


### Q104: Composite Indexes: Column Ordering and the Leftmost Prefix Rule

#### Verbal Answer:
* **Composite Index:** An index spanning multiple columns: `CREATE INDEX idx_abc ON tbl(A, B, C)`.

* **Leftmost Prefix Rule:** The B-Tree is ordered primarily by `A`, then by `B` within `A`, then by `C` within `B`.
  - Can accelerate queries filtering on: `(A)`, `(A, B)`, `(A, B, C)`.
  - **Cannot** accelerate queries filtering solely on `(B)` or `(C)` without `A`.

* **Range Predicate Rule:** Once a column in the composite index is used in a range comparison (e.g. `A = 1 AND B > 10`), subsequent columns (`C`) cannot be used for index seeks.

#### Tricky Follow-up:
* **Question:** Given query `WHERE status = 'OPEN' ORDER BY created_at DESC`, how should a composite index be ordered?

* **Answer:** `CREATE INDEX idx ON orders (status, created_at DESC)`. Equality columns must come first, followed by sorting/range columns to satisfy both filtering and sorting in a single index seek without filesort.


In [105]:
%%sql
CREATE TABLE audit_logs (
    tenant_id INT,
    user_id INT,
    action_type VARCHAR,
    logged_at TIMESTAMP
);
INSERT INTO audit_logs VALUES
(1, 10, 'LOGIN', TIMESTAMP '2024-01-01 10:00:00'),
(1, 10, 'PURCHASE', TIMESTAMP '2024-01-01 10:05:00'),
(2, 20, 'LOGIN', TIMESTAMP '2024-01-01 10:00:00');

SELECT * 
FROM audit_logs 
WHERE tenant_id = 1 AND user_id = 10 
ORDER BY logged_at;

tenant_id,user_id,action_type,logged_at
1,10,LOGIN,2024-01-01 10:00:00
1,10,PURCHASE,2024-01-01 10:05:00


### Q105: Covering Indexes and Index-Only Scans

#### Verbal Answer:
* **Index-Only Scan:** Occurs when **every single column** requested by the query (`SELECT`, `WHERE`, `JOIN`, `ORDER BY`) is contained directly inside the index structure.

* **Performance Advantage:** The engine reads only the compact B-Tree leaf pages in memory and completely skips accessing the main table heap pages on disk (zero random heap I/O).

* **`INCLUDE` Clause (Non-Key Columns):** Modern SQL (Postgres, SQL Server) allows `CREATE INDEX idx ON tbl (search_col) INCLUDE (payload_col)`. This stores `payload_col` in the leaf nodes without adding it to the B-Tree sorting hierarchy.

#### Tricky Follow-up:
* **Question:** Why is `SELECT *` the enemy of Covering Indexes?

* **Answer:** `SELECT *` requests all columns, almost certainly including attributes not present in the index, forcing the engine to perform expensive random heap lookups for every matching row.


In [106]:
%%sql
CREATE TABLE covering_demo (
    emp_id INT,
    dept_id INT,
    salary DOUBLE,
    biography VARCHAR -- Large text column
);
INSERT INTO covering_demo VALUES
(1, 10, 95000, 'Bio text 1...'),
(2, 10, 110000, 'Bio text 2...'),
(3, 20, 80000, 'Bio text 3...');

SELECT dept_id, AVG(salary) 
FROM covering_demo 
WHERE dept_id = 10 
GROUP BY dept_id;

dept_id,avg(salary)
10,102500.0


### Q106: Sargability: Why Functions on Indexed Columns Break Index Seeks

#### Verbal Answer:
* **Sargable (Search Argument Able):** A predicate is sargable if the engine can leverage a B-Tree index seek to locate matching rows directly in $O(\log N)$ time.

* **Non-Sargable Anti-Patterns (Index Invalidation):**
  - Wrapping indexed column in a function: `WHERE YEAR(created_at) = 2024` (forces full scan of all rows).
  - Arithmetic expressions on column: `WHERE salary * 1.1 > 100000`.
  - Leading wildcards: `WHERE name LIKE '%smith'`.

* **Sargable Rewrites:**
  - Rewrite to range: `WHERE created_at >= '2024-01-01' AND created_at < '2025-01-01'`.
  - Move arithmetic to constant side: `WHERE salary > 100000 / 1.1`.

#### Tricky Follow-up:
* **Question:** How can you make a function query sargable if the function is strictly unavoidable (e.g. `LOWER(email)`)?

* **Answer:** Create an **Expression / Function-Based Index**: `CREATE INDEX idx_lower_email ON users (LOWER(email))`.


In [107]:
%%sql
CREATE TABLE sargable_demo (
    tx_id INT,
    created_at TIMESTAMP,
    amount DOUBLE
);
INSERT INTO sargable_demo VALUES
(1, TIMESTAMP '2024-01-15 10:00:00', 500.0),
(2, TIMESTAMP '2023-12-31 23:59:59', 300.0),
(3, TIMESTAMP '2024-06-01 12:00:00', 700.0);

SELECT * 
FROM sargable_demo 
WHERE created_at >= TIMESTAMP '2024-01-01 00:00:00' 
  AND created_at < TIMESTAMP '2025-01-01 00:00:00';

tx_id,created_at,amount
1,2024-01-15 10:00:00,500.0
3,2024-06-01 12:00:00,700.0


### Q107: Predicate Pushdown and Projection Pruning in Columnar Engines

#### Verbal Answer:
* **Projection Pruning:** Columnar engines (DuckDB, Parquet, Snowflake) read only the exact physical byte columns specified in the `SELECT` list. Unreferenced columns are never read from disk into RAM.

* **Predicate Pushdown:** Pushes `WHERE` filters down into the physical storage reader before scanning or materializing rows.

* **Zone Maps / Parquet Statistics:** Parquet files and columnar micro-partitions store `min` and `max` values per block. If `WHERE age > 65` and a block's max is `40`, the entire file block is skipped without reading a single byte.

#### Tricky Follow-up:
* **Question:** Why does `SELECT *` completely disable projection pruning benefits in columnar storage?

* **Answer:** Because `SELECT *` forces the storage engine to decompress, decode, and load all 100+ columns from storage into memory, destroying I/O throughput.


In [108]:
%%sql
CREATE TABLE columnar_test AS 
SELECT 
    i AS id, 
    i * 10 AS metric_a, 
    'heavy_text_payload_' || i AS payload 
FROM generate_series(1, 10000) t(i);

EXPLAIN 
SELECT metric_a 
FROM columnar_test 
WHERE metric_a > 50000;

explain_key,explain_value
physical_plan,"┌───────────────────────────┐\n│ SEQ_SCAN │\n│ ──────────────────── │\n│ Table: │\n│ memory.main.columnar_test │\n│ │\n│ Type: Sequential Scan │\n│ Projections: metric_a │\n│ │\n│ Filters: │\n│ metric_a>50000 │\n│ │\n│ ~2,000 rows │\n└───────────────────────────┘\n"


### Q108: Cost-Based Optimizer (CBO) Mechanics: Histograms and Cardinality Estimation

#### Verbal Answer:
* **Cost-Based Optimization:** The CBO generates dozens of candidate execution trees, estimating the CPU and I/O cost of each, and picks the lowest-cost plan.

* **Statistics & Equi-Width / Equi-Depth Histograms:** The engine maintains statistical distribution summaries of column values (MCVs: Most Common Values, distinct count approximations, and bucketed value histograms).

* **Stale Statistics Failure:** If a table grows from 1,000 to 50,000,000 rows without running `ANALYZE` / `VACUUM ANALYZE`, the optimizer mistakenly assumes the table is tiny and chooses a disastrous Nested Loop Join over millions of rows.

#### Tricky Follow-up:
* **Question:** How do database engines handle correlation between columns (e.g. `WHERE make = 'Audi' AND model = 'A4'`)?

* **Answer:** By default, engines assume attribute independence, multiplying probabilities ($P(A) \times P(B)$), which severely underestimates cardinality. Modern engines allow creating **Extended Multi-Column Statistics** (`CREATE STATISTICS ... ON make, model`).


In [109]:
%%sql
CREATE TABLE stats_demo AS 
SELECT 
    (i % 5) AS category_id, 
    random() AS val 
FROM generate_series(1, 5000) t(i);

SUMMARIZE stats_demo;

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
category_id,BIGINT,0,4,5,2.0,1.4143550049460718,1,2,3,5000,0.0
val,DOUBLE,0.0001584325062908226,0.9997899517767063,6003,0.508328665967308,0.28725659863658604,0.26701533276905043,0.5147317938786025,0.7558443999729354,5000,0.0


### Q109: Partial and Filtered Indexes for Skewed and Hot-Data Workloads

#### Verbal Answer:
* **Partial Index Definition:** An index built over a subset of rows meeting a specific predicate: `CREATE INDEX idx_unprocessed ON queue(created_at) WHERE status = 'PENDING'`.

* **Storage and Write Efficiency:**
  - 99% of queue records are `'PROCESSED'` and are **not** indexed.
  - The index is tiny (e.g. 50 KB instead of 5 GB), fitting 100% in L1/L2 CPU cache.
  - Inserting or updating `'PROCESSED'` rows incurs zero index maintenance overhead.

#### Tricky Follow-up:
* **Question:** Will a partial index with `WHERE is_active = TRUE` be used by a query with `WHERE is_active = FALSE`?

* **Answer:** No. The optimizer can only utilize a partial index if the query's `WHERE` clause mathematically guarantees that candidate rows fall strictly within the index's filter predicate.


In [110]:
%%sql
CREATE TABLE job_queue (
    job_id INT PRIMARY KEY,
    status VARCHAR,
    priority INT
);
INSERT INTO job_queue VALUES
(1, 'COMPLETED', 1), (2, 'COMPLETED', 2), (3, 'PENDING', 10), (4, 'PENDING', 5);

SELECT job_id, priority 
FROM job_queue 
WHERE status = 'PENDING' 
ORDER BY priority DESC;

job_id,priority
3,10
4,5


### Q110: Optimizer Hints and Plan Forcing: When and Why to Avoid

#### Verbal Answer:
* **Optimizer Hints:** Directives embedded in SQL comments (e.g. `/*+ INDEX(t idx_name) */` or `/*+ BROADCAST(t) */`) that force the query planner to use a specific index, join order, or join algorithm.

* **The Production Hazard:** Hardcoded hints bypass the Cost-Based Optimizer. When data volumes grow 100x or table distributions shift in 6 months, a forced Nested Loop hint will bring down production databases.

* **Rule of Thumb:** Hints are a temporary emergency patch. The root fix is updating table statistics, rewriting non-sargable queries, or adjusting schema indexing.

#### Tricky Follow-up:
* **Question:** What is the modern alternative to hints in cloud warehouses like Snowflake and BigQuery?

* **Answer:** Snowflake and BigQuery deliberately omit optimizer hints, relying instead on automatic clustering, dynamic partition pruning, and automated adaptive query execution.


In [111]:
%%sql
CREATE TABLE orders_tbl (id INT, cust_id INT);
CREATE TABLE cust_tbl   (cust_id INT, name VARCHAR);

INSERT INTO orders_tbl VALUES (1, 101), (2, 102);
INSERT INTO cust_tbl   VALUES (101, 'Alice'), (102, 'Bob');

SELECT c.name, COUNT(o.id) AS order_cnt
FROM cust_tbl c
JOIN orders_tbl o ON c.cust_id = o.cust_id
GROUP BY c.name;

name,order_cnt
Alice,1
Bob,1


### -------------------------------------------------------------------------
### PILLAR 12: ANALYTICAL DATA MODELING, KIMBALL VS ONE BIG TABLE (OBT) & DATA MARTS
### -------------------------------------------------------------------------

### Q111: Kimball Dimensional Modeling: Facts, Dimensions, and Grain

#### Verbal Answer:
* **The 4 Steps of Kimball Dimensional Design:**
  1. Select the Business Process (e.g., Retail Sales, Invoicing).
  2. Declare the Grain (e.g., One row per individual item line on a customer receipt).
  3. Identify the Dimensions (e.g., Date, Customer, Store, Product, Promotion).
  4. Identify the Numeric Facts (e.g., Quantity Sold, Extended Dollar Price, Discount Amount).

* **Grain Consistency:** Mixing facts of different grains (e.g., daily store targets with individual second-by-second transactions) in the same fact table causes massive double-counting and aggregation errors.

#### Tricky Follow-up:
* **Question:** What is the difference between an Additive, Semi-Additive, and Non-Additive Fact?

* **Answer:** Additive facts (`sales_amount`) can be summed across all dimensions. Semi-additive facts (`account_balance`) can be summed across customers/branches but NOT across time (must use point-in-time snapshot). Non-additive facts (`unit_price`, `ratio`) can never be summed.


In [112]:
%%sql
CREATE TABLE dim_date (date_key INT PRIMARY KEY, full_date DATE, day_of_week VARCHAR);
CREATE TABLE dim_product (prod_key INT PRIMARY KEY, sku VARCHAR, category VARCHAR);
CREATE TABLE fact_sales (
    date_key INT,
    prod_key INT,
    units_sold INT,
    gross_revenue DOUBLE
);

INSERT INTO dim_date VALUES (20240101, DATE '2024-01-01', 'Monday');
INSERT INTO dim_product VALUES (1, 'SKU_LAPTOP', 'Electronics');
INSERT INTO fact_sales VALUES (20240101, 1, 2, 2400.0);

SELECT 
    d.day_of_week,
    p.category,
    SUM(f.units_sold) AS total_units,
    SUM(f.gross_revenue) AS total_revenue
FROM fact_sales f
JOIN dim_date d ON f.date_key = d.date_key
JOIN dim_product p ON f.prod_key = p.prod_key
GROUP BY d.day_of_week, p.category;

day_of_week,category,total_units,total_revenue
Monday,Electronics,2.0,2400.0


### Q112: Star Schema vs Snowflake Schema vs 3NF (Third Normal Form)

#### Verbal Answer:
* **Star Schema:** Fact table at center connected directly to denormalized dimension tables. Optimized for OLAP query speed and simple BI joins.

* **Snowflake Schema:** Dimension tables are normalized into sub-dimension hierarchies (e.g. `Product` -> `SubCategory` -> `Category`). Minimizes dimension data redundancy, but requires more joins during queries.

* **3NF (Third Normal Form):** Highly normalized OLTP transactional design. Eliminates update anomalies, but requires 15+ joins for analytical queries, making OLAP reporting unacceptably slow.

#### Tricky Follow-up:
* **Question:** Why do modern data warehouses prefer Star Schemas over Snowflake Schemas?

* **Answer:** Storage is cheap, and modern columnar vectorized engines execute queries dramatically faster when dimension attributes are pre-joined (denormalized), eliminating runtime join shuffle overhead.


In [113]:
%%sql
CREATE TABLE dim_product_star (
    prod_id INT,
    prod_name VARCHAR,
    subcategory_name VARCHAR,
    category_name VARCHAR
);
INSERT INTO dim_product_star VALUES (1, 'MacBook Pro', 'Laptops', 'Computers');

SELECT category_name, subcategory_name, prod_name 
FROM dim_product_star;

category_name,subcategory_name,prod_name
Computers,Laptops,MacBook Pro


### Q113: One Big Table (OBT) Architecture vs Normalized Star Schemas in Modern DWH

#### Verbal Answer:
* **One Big Table (OBT):** A single, massive, completely flat denormalized table combining all fact metrics and dimension attributes into 100+ columns.

* **Why OBT Wins in Cloud Columnar DWH (Snowflake, BigQuery, ClickHouse):**
  - **Zero Joins:** Eliminates all network join shuffles and hash-table construction.
  - **Columnar Compression:** Repeated dimension values compress extremely efficiently via Run-Length Encoding (RLE) and dictionary encoding.
  - **Micro-Partition Pruning:** Queries filter directly on flattened partition metadata.

* **Trade-Off:** Increased storage footprint and re-ingestion complexity when dimension attributes update.

#### Tricky Follow-up:
* **Question:** When does Star Schema still beat OBT?

* **Answer:** When dimension attributes change constantly (SCD Type 2), or when multiple distinct business processes (e.g. Sales, Returns, Inventory) share conformed dimensions.


In [114]:
%%sql
CREATE TABLE obt_sales_flat (
    tx_id INT,
    tx_date DATE,
    cust_id INT,
    cust_name VARCHAR,
    cust_city VARCHAR,
    cust_segment VARCHAR,
    prod_sku VARCHAR,
    prod_category VARCHAR,
    amount DOUBLE,
    qty INT
);
INSERT INTO obt_sales_flat VALUES
(101, DATE '2024-01-01', 1, 'Alice', 'Seattle', 'Enterprise', 'SKU_1', 'Tech', 1200.0, 1),
(102, DATE '2024-01-01', 2, 'Bob',   'Austin',  'SMB',        'SKU_2', 'Office', 45.0, 3);

SELECT 
    cust_segment,
    prod_category,
    SUM(amount) AS total_revenue
FROM obt_sales_flat
GROUP BY cust_segment, prod_category;

cust_segment,prod_category,total_revenue
Enterprise,Tech,1200.0
SMB,Office,45.0


### Q114: Conformed Dimensions and the Enterprise Data Warehouse Bus Matrix

#### Verbal Answer:
* **Conformed Dimension:** A single, standardized, consistently governed dimension table (e.g., `Dim_Customer`, `Dim_Date`) shared across multiple different business process fact tables (e.g., `Fact_Sales`, `Fact_Support_Tickets`, `Fact_Marketing_Campaigns`).

* **Bus Matrix:** A matrix mapping business processes (rows) to conformed dimensions (columns).

* **Core Value:** Guarantees consistent definitions across the enterprise (e.g., "Customer Churn" is calculated identically in Sales, Finance, and Product).

#### Tricky Follow-up:
* **Question:** What query technique allows cross-drilling across two separate fact tables sharing a conformed dimension?

* **Answer:** Drill-across queries: aggregate each fact table to the conformed dimension level in separate CTEs, then `FULL OUTER JOIN` the CTEs on the conformed dimension key.


In [115]:
%%sql
CREATE TABLE conformed_dim_customer (cust_id INT PRIMARY KEY, name VARCHAR);
CREATE TABLE fact_orders_proc (cust_id INT, order_spend DOUBLE);
CREATE TABLE fact_returns_proc (cust_id INT, return_spend DOUBLE);

INSERT INTO conformed_dim_customer VALUES (1, 'Alice'), (2, 'Bob');
INSERT INTO fact_orders_proc VALUES (1, 500.0), (1, 300.0), (2, 200.0);
INSERT INTO fact_returns_proc VALUES (1, 50.0);

WITH orders_agg AS (
    SELECT cust_id, SUM(order_spend) AS total_ordered FROM fact_orders_proc GROUP BY cust_id
),
returns_agg AS (
    SELECT cust_id, SUM(return_spend) AS total_returned FROM fact_returns_proc GROUP BY cust_id
)
SELECT 
    c.name,
    COALESCE(o.total_ordered, 0.0) AS total_ordered,
    COALESCE(r.total_returned, 0.0) AS total_returned,
    COALESCE(o.total_ordered, 0.0) - COALESCE(r.total_returned, 0.0) AS net_revenue
FROM conformed_dim_customer c
LEFT JOIN orders_agg o ON c.cust_id = o.cust_id
LEFT JOIN returns_agg r ON c.cust_id = r.cust_id;

name,total_ordered,total_returned,net_revenue
Alice,800.0,50.0,750.0
Bob,200.0,0.0,200.0


### Q115: Factless Fact Tables: Event Tracking and Coverage Matrices

#### Verbal Answer:
* **Factless Fact Table:** A fact table containing **zero numeric measurement columns**, consisting solely of foreign surrogate keys to dimension tables.

* **Two Main Types:**
  1. **Event Tracking:** Records that an event occurred (e.g., student attendance in class, web page visit log).
  2. **Coverage Matrix:** Records what did *not* happen (e.g., products on promotion that had zero sales).

* **Aggregations:** Analyzed via `COUNT(*)` to count event occurrences.

#### Tricky Follow-up:
* **Question:** How do you identify which students were absent on a given day using a factless attendance table?

* **Answer:** `CROSS JOIN` all students with the date, then `LEFT JOIN` the factless attendance table `WHERE attendance.student_id IS NULL`.


In [116]:
%%sql
CREATE TABLE factless_attendance (
    date_key INT,
    student_id INT,
    course_id INT
);
INSERT INTO factless_attendance VALUES
(20240101, 101, 501),
(20240101, 102, 501),
(20240102, 101, 501);

SELECT 
    course_id,
    date_key,
    COUNT(student_id) AS attendees_count
FROM factless_attendance
GROUP BY course_id, date_key
ORDER BY date_key;

course_id,date_key,attendees_count
501,20240101,2
501,20240102,1


### Q116: Junk Dimensions: Consolidating Low-Cardinality Flags and Indicators

#### Verbal Answer:
* **The Problem:** A fact table has 8 boolean/categorical flag columns (`is_gift`, `is_express_delivery`, `payment_method`, `credit_check_passed`). Creating 8 separate dimension tables requires 8 joins; leaving them in the fact table bloats row width.

* **Junk Dimension Solution:** Combine all low-cardinality flags into a single consolidated dimension table (`Dim_Order_Attributes`) containing every unique combination of flags.

* **Fact Table Efficiency:** Replaces 8 separate flag columns with a single `junk_dim_id` foreign key.

#### Tricky Follow-up:
* **Question:** If you have 4 boolean flags, what is the maximum number of rows in the junk dimension?

* **Answer:** $2^4 = 16$ rows total.


In [117]:
%%sql
CREATE TABLE dim_order_junk (
    junk_key INT PRIMARY KEY,
    is_gift VARCHAR,
    payment_type VARCHAR,
    is_rush_ship VARCHAR
);
INSERT INTO dim_order_junk VALUES
(1, 'YES', 'CARD', 'YES'),
(2, 'NO',  'CARD', 'NO'),
(3, 'NO',  'PAYPAL', 'NO');

CREATE TABLE fact_orders_junk (
    order_id INT,
    junk_key INT,
    amount DOUBLE
);
INSERT INTO fact_orders_junk VALUES (101, 1, 150.0), (102, 2, 80.0);

SELECT 
    j.is_gift,
    j.payment_type,
    SUM(f.amount) AS total_sales
FROM fact_orders_junk f
JOIN dim_order_junk j ON f.junk_key = j.junk_key
GROUP BY j.is_gift, j.payment_type;

is_gift,payment_type,total_sales
YES,CARD,150.0
NO,CARD,80.0


### Q117: Periodic Snapshot vs Accumulating Snapshot vs Transaction Fact Tables

#### Verbal Answer:
* **Transaction Fact Table:** One row per point-in-time discrete transaction (e.g. individual credit card swipe). Most granular; infinite growth.

* **Periodic Snapshot Fact Table:** One row summarizing a specific entity over a fixed interval (e.g. daily account closing balance, monthly warehouse inventory level).

* **Accumulating Snapshot Fact Table:** One row per business pipeline lifecycle with multiple milestone date keys updated as the entity progresses (e.g. Order Placed -> Shipped -> Delivered -> Paid -> Returned).

#### Tricky Follow-up:
* **Question:** Which fact table type requires updating existing historical rows as milestones are reached?

* **Answer:** Accumulating Snapshot fact tables. As an order moves from 'Shipped' to 'Delivered', the `delivered_date_key` column is updated in place.


In [118]:
%%sql
CREATE TABLE fact_order_accumulating (
    order_id INT,
    placed_dt DATE,
    shipped_dt DATE,
    delivered_dt DATE,
    amount DOUBLE
);
INSERT INTO fact_order_accumulating VALUES
(1, DATE '2024-01-01', DATE '2024-01-02', DATE '2024-01-05', 250.0),
(2, DATE '2024-01-03', DATE '2024-01-04', NULL,             120.0); -- In transit;

SELECT 
    order_id,
    placed_dt,
    delivered_dt,
    DATEDIFF('day', placed_dt, delivered_dt) AS days_to_fulfill,
    CASE WHEN delivered_dt IS NULL THEN 'IN_TRANSIT' ELSE 'FULFILLED' END AS fulfillment_status
FROM fact_order_accumulating;

order_id,placed_dt,delivered_dt,days_to_fulfill,fulfillment_status
1,2024-01-01,2024-01-05,4,FULFILLED
2,2024-01-03,NaT,<NA>,IN_TRANSIT


### Q118: Data Vault 2.0: Hubs, Links, and Satellites Architecture

#### Verbal Answer:
* **Hub (Core Business Concept):** Stores unique business keys (e.g., `hub_customer` containing `customer_id`, MD5 hash key, load timestamp).

* **Link (Unit of Work / Relationship):** Stores associations and transactions between hubs (e.g., `link_customer_order` containing MD5 hash of customer + order).

* **Satellite (Descriptive Context over Time):** Stores mutable descriptive attributes and history (e.g., `sat_customer_details` containing `name`, `address`, `phone`, `load_end_date`).

* **Enterprise Value:** Infinite scalability, fully decoupled parallel ETL loading, and 100% auditability.

#### Tricky Follow-up:
* **Question:** Why can Hubs and Links be loaded in parallel without waiting for foreign key dependencies in Data Vault?

* **Answer:** Because surrogate keys are generated as deterministic cryptographic hashes (MD5/SHA256) of business keys directly from source data without global sequence counters.


In [119]:
%%sql
CREATE TABLE hub_customer (
    h_customer_hash VARCHAR PRIMARY KEY,
    customer_id INT,
    load_dts TIMESTAMP,
    rec_src VARCHAR
);
CREATE TABLE sat_customer (
    h_customer_hash VARCHAR,
    cust_name VARCHAR,
    cust_city VARCHAR,
    load_dts TIMESTAMP
);

INSERT INTO hub_customer VALUES (md5('101'), 101, CURRENT_TIMESTAMP, 'CRM_APP');
INSERT INTO sat_customer VALUES (md5('101'), 'Alice Enterprise', 'Seattle', CURRENT_TIMESTAMP);

SELECT 
    h.customer_id,
    s.cust_name,
    s.cust_city,
    h.rec_src
FROM hub_customer h
JOIN sat_customer s ON h.h_customer_hash = s.h_customer_hash;

customer_id,cust_name,cust_city,rec_src
101,Alice Enterprise,Seattle,CRM_APP


### Q119: Activity Schema Architecture: A Single Time-Series Event Log Paradigm

#### Verbal Answer:
* **The Activity Schema Paradigm:** Replaces complex star schemas with a single standard table of customer events: `(customer, activity, timestamp, anonymous_id, feature_json)`.

* **Core Premise:** All human behavior consists of an entity performing an action at a specific time.

* **Advantages:** Extremely simple ELT, zero multi-table data warehouse maintenance, and temporal queries constructed via standardized time-series CTE joins.

#### Tricky Follow-up:
* **Question:** How does Activity Schema link an anonymous website visitor to their post-signup customer profile?

* **Answer:** Via identity resolution stitching: an activity record associates `anonymous_id` with `customer_id` upon login, updating the unified identity mapping.


In [120]:
%%sql
CREATE TABLE activity_stream (
    customer_id VARCHAR,
    activity VARCHAR,
    ts TIMESTAMP,
    revenue DOUBLE
);
INSERT INTO activity_stream VALUES
('user_1', 'visited_page', TIMESTAMP '2024-01-01 10:00:00', NULL),
('user_1', 'subscribed',   TIMESTAMP '2024-01-01 10:15:00', 99.0),
('user_2', 'visited_page', TIMESTAMP '2024-01-01 11:00:00', NULL);

SELECT 
    COUNT(DISTINCT CASE WHEN activity = 'visited_page' THEN customer_id END) AS visitors,
    COUNT(DISTINCT CASE WHEN activity = 'subscribed' THEN customer_id END) AS conversions,
    ROUND(COUNT(DISTINCT CASE WHEN activity = 'subscribed' THEN customer_id END) * 100.0 / 
          COUNT(DISTINCT CASE WHEN activity = 'visited_page' THEN customer_id END), 2) AS conv_rate_pct
FROM activity_stream;

visitors,conversions,conv_rate_pct
2,1,50.0


### Q120: Medallion Architecture: Bronze (Raw) vs Silver (Cleaned) vs Gold (Aggregated)

#### Verbal Answer:
* **Bronze Layer (Raw Ingestion):** Append-only raw data dump directly from Kafka/APIs/CDC. Preserves exact source schema and raw history.

* **Silver Layer (Enterprise Cleaned & Conformed):** Deduplicated, validated, typed, and enriched data models. Filtered for quality anomalies and soft deletes.

* **Gold Layer (Business Aggregates & Data Marts):** Star schemas, OBTs, and specialized KPI rollups modeled for executive reporting, BI dashboards, and ML feature tables.

#### Tricky Follow-up:
* **Question:** Why is the Bronze layer strictly append-only?

* **Answer:** To provide a resilient, permanent audit trail that allows replaying all upstream business logic from scratch if a bug is discovered in Silver or Gold transformation pipelines.


In [121]:
%%sql
CREATE TABLE bronze_events (raw_payload JSON, ingested_at TIMESTAMP);
INSERT INTO bronze_events VALUES ('{"uid": 1, "amt": "100.5"}', CURRENT_TIMESTAMP);

CREATE TABLE silver_events AS
SELECT 
    CAST(raw_payload->>'uid' AS INT) AS user_id,
    CAST(raw_payload->>'amt' AS DOUBLE) AS amount,
    ingested_at
FROM bronze_events;

SELECT 
    user_id,
    SUM(amount) AS total_lifetime_spend
FROM silver_events
GROUP BY user_id;

user_id,total_lifetime_spend
1,100.5


### -------------------------------------------------------------------------
### PILLAR 13: PARTITIONING, SHARDING, SKEW MITIGATION & MPP ARCHITECTURE
### -------------------------------------------------------------------------

### Q121: Partition Pruning Mechanics: Static vs Dynamic Partition Pruning

#### Verbal Answer:
* **Partition Pruning:** The query engine skips reading physical directories or partition files based on query predicates (`WHERE date = '2024-01-01'`).

* **Static Partition Pruning (SPP):** Evaluated at query compile time when predicates contain literal constants or deterministic expressions (`WHERE dt = CURRENT_DATE`).

* **Dynamic Partition Pruning (DPP):** Evaluated at query runtime when filtering on the result of a join with another table (`WHERE fact.date_key IN (SELECT date_key FROM dim_date WHERE is_holiday = TRUE)`). The engine builds a bloom filter on the dimension join keys and dynamically prunes fact partitions during the scan.

#### Tricky Follow-up:
* **Question:** What happens if a query has `WHERE CAST(partition_col AS VARCHAR) = '2024-01-01'`?

* **Answer:** Partition pruning is completely disabled! The engine cannot evaluate the partition expression statically and is forced to scan every partition in the entire table.


In [122]:
%%sql
CREATE TABLE partitioned_sales AS 
SELECT 
    DATE '2024-01-01' + (i % 5) * INTERVAL 1 DAY AS sale_date,
    i AS sale_id,
    (i * 15.0) AS amount
FROM generate_series(1, 1000) t(i);

SELECT 
    sale_date,
    COUNT(*) AS total_sales,
    SUM(amount) AS total_revenue
FROM partitioned_sales
WHERE sale_date = DATE '2024-01-01'
GROUP BY sale_date;

sale_date,total_sales,total_revenue
2024-01-01,200,1507500.0


### Q122: Data Skew in Distributed Joins: The Straggler Problem and Salting Keys

#### Verbal Answer:
* **The Straggler Bottleneck:** In distributed MPP engines (Spark, BigQuery, Snowflake), rows are hashed and shuffled to worker nodes based on the join key (`user_id`). If 50% of transactions belong to a single guest user (`user_id = 0` or `NULL`), that single worker node receives 50% of the entire cluster's data, causing Out-Of-Memory (OOM) failures while other workers sit idle.

* **Salting Mitigation Strategy:**
  - Append a random integer salt ($1$ to $K$) to the skewed key on Table A: `CONCAT(user_id, '_', FLOOR(RANDOM() * 10))`.
  - Explode/Replicate Table B keys by $K$ rows with matching salt values.
  - Join on the salted key, distributing the skewed partition evenly across $K$ worker nodes.

#### Tricky Follow-up:
* **Question:** What is the simplest fix if the skewed value is `NULL`?

* **Answer:** Assign a unique random negative number to each `NULL` row during join: `COALESCE(user_id, -1 * ROW_NUMBER() OVER ())`, scattering `NULL`s across all nodes.


In [123]:
%%sql
CREATE TABLE skewed_transactions (
    user_id INT,
    amount DOUBLE
);
-- 80% of rows have user_id = 0 (Guest Checkout)
INSERT INTO skewed_transactions 
SELECT CASE WHEN i <= 80 THEN 0 ELSE i END, (i * 10.0) 
FROM generate_series(1, 100) t(i);

SELECT 
    user_id,
    -- Salted key splits user 0 into 5 distinct buckets: 0_0, 0_1, ..., 0_4
    CONCAT(CAST(user_id AS VARCHAR), '_', CAST(FLOOR(random() * 5) AS VARCHAR)) AS salted_join_key,
    amount
FROM skewed_transactions
LIMIT 5;

user_id,salted_join_key,amount
0,0_1.0,10.0
0,0_0.0,20.0
0,0_4.0,30.0
0,0_0.0,40.0
0,0_3.0,50.0


### Q123: Broadcast Join vs Shuffle Hash Join in MPP Engines

#### Verbal Answer:
* **Shuffle Hash Join:** Both large tables are re-partitioned across the entire network by hashing their join keys. Requires massive network I/O data transfer across all cluster nodes.

* **Broadcast (Map-Side) Join:** When joining a massive fact table (10 TB) with a tiny dimension table (5 MB), the master node broadcasts a full copy of the tiny dimension to the local memory of every single worker node.

* **Performance Gain:** Completely eliminates network shuffling of the 10 TB fact table, converting the join into a fast local scan.

#### Tricky Follow-up:
* **Question:** What is the primary risk of forcing a Broadcast Join on a growing table?

* **Answer:** If the broadcast table grows beyond worker memory (e.g. exceeds 8 GB), every worker node crashes simultaneously with an `OutOfMemoryError` (OOM).


In [124]:
%%sql
CREATE TABLE fact_massive (id INT, cat_id INT, val DOUBLE);
CREATE TABLE dim_tiny (cat_id INT, cat_name VARCHAR);

INSERT INTO dim_tiny VALUES (1, 'Tech'), (2, 'Bio');
INSERT INTO fact_massive SELECT i, (i % 2) + 1, random() FROM generate_series(1, 1000) t(i);

SELECT d.cat_name, AVG(f.val) AS avg_metric
FROM fact_massive f
JOIN dim_tiny d ON f.cat_id = d.cat_id
GROUP BY d.cat_name;

cat_name,avg_metric
Bio,0.508788
Tech,0.517723


### Q124: Z-Ordering and Multi-Dimensional Clustering (MDC)

#### Verbal Answer:
* **The 1D Partitioning Limitation:** Traditional partitioning only sorts data along a single dimension (e.g., `date`). Filtering on `(region, product_category)` still scans the entire partition.

* **Z-Ordering (Space-Filling Curve):** Maps multi-dimensional data `(date, region, product)` into a 1-dimensional space-filling Hilbert or Z-curve while preserving spatial locality.

* **Clustering Benefits:** Allows queries filtering on **any combination** of the clustered attributes to achieve dramatic micro-partition skipping in Delta Lake, Snowflake, and Apache Iceberg.

#### Tricky Follow-up:
* **Question:** How many columns should you typically include in a Z-Order / Cluster Key?

* **Answer:** Between 1 and 4 columns. Adding more than 4 columns dilutes spatial locality, degrading skipping efficiency across all dimensions (curse of dimensionality).


In [125]:
%%sql
CREATE TABLE clustered_events (
    event_dt DATE,
    country VARCHAR,
    user_id INT,
    spend DOUBLE
);
INSERT INTO clustered_events VALUES
(DATE '2024-01-01', 'US', 101, 500.0),
(DATE '2024-01-01', 'UK', 102, 300.0),
(DATE '2024-01-02', 'US', 103, 700.0);

SELECT country, SUM(spend) AS country_spend
FROM clustered_events
WHERE event_dt >= DATE '2024-01-01' AND country = 'US'
GROUP BY country;

country,country_spend
US,1200.0


### Q125: Consistent Hashing and Database Sharding Topologies

#### Verbal Answer:
* **Horizontal Sharding:** Splitting a single logical table across multiple independent physical database servers based on a shard key (`user_id`).

* **Modulo Hashing Flaw:** `shard = user_id % N`. When scaling from $N$ to $N+1$ nodes, almost 100% of all keys must be re-shuffled and migrated across the network.

* **Consistent Hashing:** Maps nodes and keys onto a circular hash ring ($[0, 2^{32}-1]$). Adding or removing a server node only requires migrating $K/N$ keys from its immediate neighbor, leaving all other shards untouched.

#### Tricky Follow-up:
* **Question:** What is the 'Cross-Shard Join' problem?

* **Answer:** A query that joins data residing on different physical shard servers (e.g. joining users on Shard 1 with transactions on Shard 4), which requires expensive 2-phase network coordination and degrades performance.


In [126]:
%%sql
CREATE TABLE user_shards (user_id INT, email VARCHAR);
INSERT INTO user_shards VALUES (101, 'a@co.com'), (102, 'b@co.com'), (103, 'c@co.com');

SELECT 
    user_id,
    email,
    -- Route to one of 4 shard nodes deterministically
    abs(hash(user_id)) % 4 AS assigned_shard_id
FROM user_shards;

user_id,email,assigned_shard_id
101,a@co.com,2
102,b@co.com,0
103,c@co.com,2


### Q126: Bloom Filters: Accelerated Set Membership Checks in Join Pipelines

#### Verbal Answer:
* **Bloom Filter Definition:** Space-efficient probabilistic data structure used to test whether an element is definitely **NOT** in a set or **MIGHT** be in a set (zero false negatives, tunable false positive rate).

* **Runtime Join Optimization:** When performing a large join, the engine builds a compact Bloom Filter on the small dimension table keys in memory and transmits it to fact table scanners.

* **I/O Savings:** Fact scanner nodes discard 95%+ of non-matching fact rows before they are ever decompressed or shuffled across the network.

#### Tricky Follow-up:
* **Question:** Can a Bloom Filter return a false negative (saying an element is NOT present when it actually is)?

* **Answer:** No. By mathematical design, a Bloom filter never produces false negatives. If it returns 'false', the row is 100% guaranteed to not match the join.


In [127]:
%%sql
CREATE TABLE dim_targeted_customers (cust_id INT PRIMARY KEY);
INSERT INTO dim_targeted_customers VALUES (101), (105);

CREATE TABLE fact_stream (cust_id INT, tx_id INT, amount DOUBLE);
INSERT INTO fact_stream VALUES 
(101, 1, 50.0), (102, 2, 80.0), (103, 3, 120.0), (105, 4, 300.0);

SELECT f.* 
FROM fact_stream f
WHERE EXISTS (
    SELECT 1 FROM dim_targeted_customers d WHERE d.cust_id = f.cust_id
);

cust_id,tx_id,amount
101,1,50.0
105,4,300.0


### Q127: Range Partitioning vs List Partitioning vs Hash Partitioning

#### Verbal Answer:
* **Range Partitioning:** Partitions data by continuous numeric or temporal boundaries (e.g., `PARTITION BY RANGE (created_at)` for daily/monthly partitions).

* **List Partitioning:** Partitions data by explicit enumerated lists of discrete values (e.g., `PARTITION BY LIST (country_code)` for `'US'`, `'EU'`, `'APAC'`).

* **Hash Partitioning:** Distributes rows across $N$ partitions using a cryptographic hash modulus (`hash(user_id) % N`). Ideal for preventing hotspots when no obvious natural range exists.

#### Tricky Follow-up:
* **Question:** What happens in Range Partitioning if a row arrives with a date beyond all defined future partition boundaries?

* **Answer:** In engines without automated partition creation (PostgreSQL), the `INSERT` fails with an exception unless a `DEFAULT` partition is declared.


In [128]:
%%sql
CREATE TABLE raw_range_data (
    region VARCHAR,
    sale_year INT,
    amount DOUBLE
);
INSERT INTO raw_range_data VALUES
('AMER', 2023, 1000),
('AMER', 2024, 1500),
('EMEA', 2024, 2000);

SELECT 
    region,
    sale_year,
    SUM(amount) AS annual_region_sales
FROM raw_range_data
WHERE region = 'AMER' AND sale_year = 2024
GROUP BY region, sale_year;

region,sale_year,annual_region_sales
AMER,2024,1500.0


### Q128: The Small File Problem in Distributed Object Storage (S3 / GCS / ADLS)

#### Verbal Answer:
* **The Problem:** Streaming pipelines (Kafka, Flink) committing microscopic Parquet files every 10 seconds produce millions of 100 KB files in S3/GCS.

* **Impact on SQL Engines:** Query engines spend 90% of execution time making HTTP GET metadata requests, establishing TCP connections, and reading file footers rather than processing data.

* **Compaction Solutions:** Running periodic compaction / bin-packing jobs (`OPTIMIZE tbl COMPACT` in Delta Lake / Iceberg) merges thousands of small files into optimal 128 MB - 512 MB file chunks.

#### Tricky Follow-up:
* **Question:** Why is 128 MB to 512 MB considered the gold standard file size for Parquet?

* **Answer:** It strikes the perfect balance: large enough to maximize columnar compression and amortize I/O metadata overhead, yet small enough to enable granular parallel worker distribution and partition pruning.


In [129]:
%%sql
CREATE TABLE file_stats (
    file_name VARCHAR,
    file_size_kb INT,
    row_count INT
);
INSERT INTO file_stats VALUES
('part_001.parquet', 150, 2000),
('part_002.parquet', 180, 2500),
('compacted_target.parquet', 131072, 1500000);

SELECT 
    file_name,
    file_size_kb,
    CASE 
        WHEN file_size_kb < 1024 THEN 'NEEDS_COMPACTION (Small File)'
        ELSE 'OPTIMAL_SIZE'
    END AS storage_health
FROM file_stats;

file_name,file_size_kb,storage_health
part_001.parquet,150,NEEDS_COMPACTION (Small File)
part_002.parquet,180,NEEDS_COMPACTION (Small File)
compacted_target.parquet,131072,OPTIMAL_SIZE


### Q129: Read Replicas, Connection Pooling, and CQRS Pattern

#### Verbal Answer:
* **Read-Write Splitting:** Transactional OLTP writes (`INSERT`, `UPDATE`) target the Primary Master database, while analytical reads (`SELECT`) are routed to asynchronously replicated Read Replicas.

* **Replication Lag Hazard:** If an analyst updates their profile and immediately queries a read replica, they may see stale data if replication lag is 500ms (eventual consistency).

* **CQRS (Command Query Responsibility Segregation):** Structurally separating the write model (OLTP) from the read query model (OLAP / Elasticsearch / Star Schema) via event streams.

#### Tricky Follow-up:
* **Question:** How does Connection Pooling (e.g. PgBouncer) prevent database crashes during traffic spikes?

* **Answer:** Forking a new database process per connection consumes 5-10 MB of RAM and incurs heavy CPU context switching. PgBouncer multiplexes 10,000 client requests across a fixed pool of 50 persistent server connections.


In [130]:
%%sql
CREATE TABLE replica_status (
    node_id VARCHAR,
    role VARCHAR,
    replication_lag_ms INT
);
INSERT INTO replica_status VALUES
('db_primary', 'PRIMARY_WRITER', 0),
('db_replica_1', 'READ_REPLICA', 15),
('db_replica_2', 'READ_REPLICA', 450);

SELECT 
    node_id,
    role,
    replication_lag_ms,
    CASE 
        WHEN replication_lag_ms > 100 THEN 'DEGRADED_SYNC_RISK'
        ELSE 'HEALTHY'
    END AS status
FROM replica_status;

node_id,role,replication_lag_ms,status
db_primary,PRIMARY_WRITER,0,HEALTHY
db_replica_1,READ_REPLICA,15,HEALTHY
db_replica_2,READ_REPLICA,450,DEGRADED_SYNC_RISK


### Q130: Materialized Views: Refresh Strategies (Eager vs Incremental vs Scheduled)

#### Verbal Answer:
* **Materialized View (MV):** Physically precomputes and persists the result of a complex query to disk for instantaneous read latency.

* **Refresh Strategies:**
  - **Complete Refresh:** Recomputes the entire query from scratch (expensive, locked during refresh).
  - **Scheduled / On-Demand:** Triggered via cron or Airflow DAG (e.g. hourly refresh).
  - **Fast / Incremental Materialized View (IVM):** Database intercepts Delta logs from base tables and applies only mathematical differentials to the MV state without recalculating the full table.

#### Tricky Follow-up:
* **Question:** Why is an Incremental Materialized View easy for `SUM` and `COUNT` but mathematically difficult for `MEDIAN` or `AVG`?

* **Answer:** `SUM` and `COUNT` are distributive algebraic aggregates ($S_{new} = S_{old} + \Delta$). `MEDIAN` is holistic and requires knowing the entire sorted distribution of all values to recompute upon row deletions.


In [131]:
%%sql
CREATE TABLE stream_orders (tx_id INT, category VARCHAR, price DOUBLE);
INSERT INTO stream_orders VALUES (1, 'A', 100), (2, 'B', 200), (3, 'A', 150);

CREATE TABLE mv_category_revenue AS
SELECT category, SUM(price) AS total_revenue, COUNT(*) AS tx_count
FROM stream_orders
GROUP BY category;

SELECT * FROM mv_category_revenue ORDER BY category;

category,total_revenue,tx_count
A,250.0,2
B,200.0,1


### -------------------------------------------------------------------------
### PILLAR 14: ACID, ISOLATION LEVELS, CONCURRENCY & LOCKING VS MVCC
### -------------------------------------------------------------------------

### Q131: ACID Properties: Atomic, Consistent, Isolated, Durable

#### Verbal Answer:
* **Atomicity:** All statements within a transaction boundary (`BEGIN ... COMMIT`) succeed together, or all changes are completely rolled back upon error (`ROLLBACK`). No half-completed writes exist.

* **Consistency:** A transaction can only transition the database from one valid state to another, strictly enforcing all declared constraints, check rules, and referential foreign keys.

* **Isolation:** Determines how concurrent transactions interact and whether intermediate uncommitted states are visible to other sessions.

* **Durability:** Once a transaction commits, its modifications are permanently recorded in non-volatile storage (via Write-Ahead Logging - WAL) and survive system crashes or power failures.

#### Tricky Follow-up:
* **Question:** What is the Write-Ahead Log (WAL) and how does it guarantee Durability?

* **Answer:** Before any data page is modified in RAM or written to disk, the change is written sequentially to the append-only WAL on disk. Upon crash restart, the engine replays the WAL to recover committed state.


In [132]:
%%sql
CREATE TABLE bank_accounts (account_id INT PRIMARY KEY, balance DOUBLE);
INSERT INTO bank_accounts VALUES (1, 1000.0), (2, 500.0);

BEGIN TRANSACTION;
UPDATE bank_accounts SET balance = balance - 200.0 WHERE account_id = 1;
UPDATE bank_accounts SET balance = balance + 200.0 WHERE account_id = 2;
COMMIT;

SELECT * FROM bank_accounts ORDER BY account_id;

account_id,balance
1,800.0
2,700.0


### Q132: The 4 ANSI SQL Isolation Levels and Transaction Concurrency Anomalies

#### Verbal Answer:
* **Isolation Levels & Anomalies Permitted:**
  1. **Read Uncommitted:** Allows Dirty Reads (reading uncommitted changes from another transaction that might later roll back).
  2. **Read Committed (Postgres/Oracle/Snowflake default):** Prevents Dirty Reads. Allows Non-Repeatable Reads (reading row X twice yields different values if another tx commits an update).
  3. **Repeatable Read (MySQL default):** Prevents Non-Repeatable Reads. May allow Phantom Reads (new rows inserted by concurrent tx matching a range filter).
  4. **Serializable:** Highest level. Strict mathematical equivalence to executing transactions one-by-one serially. Prevents all anomalies including Write Skew.

#### Tricky Follow-up:
* **Question:** What is the performance trade-off of running all transactions at `SERIALIZABLE`?

* **Answer:** Dramatically increased transaction abort rates, high lock contention, severe serialization failure retries, and lower throughput.


In [133]:
%%sql
SELECT 'Executed' AS status;

status
Executed


### Q133: Multi-Version Concurrency Control (MVCC): Readers Never Block Writers

#### Verbal Answer:
* **MVCC Core Architecture:** Instead of acquiring read locks that block write operations, the engine maintains multiple physical versions of each row.

* **Row Header Metadata:** Each tuple stores `xmin` (the transaction ID that created the row) and `xmax` (the transaction ID that deleted/superseded the row).

* **Snapshot Isolation:** When a query begins, it captures a snapshot of active transaction IDs. The query sees only row versions that were committed prior to its snapshot, completely ignoring in-flight or subsequent modifications.

* **Golden Rule:** "Readers never block writers, and writers never block readers."

#### Tricky Follow-up:
* **Question:** What database process is responsible for cleaning up old invisible row versions in MVCC?

* **Answer:** The **Vacuum** process (e.g. `VACUUM` in PostgreSQL). It scans table pages, reclaims dead tuples where `xmax` is older than all active snapshots, and prevents table bloat.


In [134]:
%%sql
CREATE TABLE mvcc_test (id INT, val VARCHAR);
INSERT INTO mvcc_test VALUES (1, 'Initial');

UPDATE mvcc_test SET val = 'Updated' WHERE id = 1;

SELECT * FROM mvcc_test;

id,val
1,Updated


### Q134: Pessimistic Locking vs Optimistic Concurrency Control (OCC)

#### Verbal Answer:
* **Pessimistic Locking (`SELECT ... FOR UPDATE`):**
  - Assumes conflicts will occur.
  - Explicitly locks selected rows against concurrent modifications until the current transaction commits.
  - Ideal for high-contention inventory reservation or banking operations.

* **Optimistic Concurrency Control (OCC):**
  - Assumes conflicts are rare.
  - Reads data without locks alongside a `version_number`.
  - Upon commit, executes: `UPDATE tbl SET val = :new_val, version = version + 1 WHERE id = :id AND version = :read_version`.
  - If 0 rows updated, a conflict occurred; the application rolls back and retries.

#### Tricky Follow-up:
* **Question:** Why is Optimistic Concurrency Control preferred in web applications and microservices?

* **Answer:** Because holding database locks across HTTP request-response roundtrips destroys server connection pools and causes massive locking bottlenecks.


In [135]:
%%sql
CREATE TABLE inventory_occ (
    sku VARCHAR PRIMARY KEY,
    stock_qty INT,
    version_id INT
);
INSERT INTO inventory_occ VALUES ('ITEM_A', 10, 1);

UPDATE inventory_occ 
SET stock_qty = stock_qty - 1, version_id = version_id + 1 
WHERE sku = 'ITEM_A' AND version_id = 1;

SELECT * FROM inventory_occ;

sku,stock_qty,version_id
ITEM_A,9,2


### Q135: Deadlocks: Detection, Prevention, and Lock Hierarchy Standards

#### Verbal Answer:
* **Deadlock Definition:** Occurs when Transaction 1 holds Lock A and waits for Lock B, while Transaction 2 holds Lock B and waits for Lock A. Neither transaction can proceed.

* **Engine Detection:** The database engine maintains a directed "Wait-For Graph" of locks. When a cycle is detected, the engine abruptly aborts one transaction (the "deadlock victim") with an error to unblock the other.

* **Deadlock Prevention Golden Rule:** Always acquire locks on multiple resources in the **exact same deterministic order** across all application code (e.g. `ORDER BY account_id`).

#### Tricky Follow-up:
* **Question:** If Tx 1 updates Account 5 then Account 10, and Tx 2 updates Account 10 then Account 5, what happens under high load?

* **Answer:** A classic deadlock cycle occurs. Ordering both queries to lock `MIN(acc_id)` first followed by `MAX(acc_id)` completely eliminates the possibility of deadlock.


In [136]:
%%sql
CREATE TABLE accounts_locked (acc_id INT PRIMARY KEY, balance DOUBLE);
INSERT INTO accounts_locked VALUES (5, 100.0), (10, 200.0);

SELECT * 
FROM accounts_locked 
WHERE acc_id IN (5, 10) 
ORDER BY acc_id;

acc_id,balance
5,100.0
10,200.0


### Q136: SELECT FOR UPDATE vs SELECT FOR SHARE vs SKIP LOCKED

#### Verbal Answer:
* **`SELECT FOR UPDATE`:** Acquires an exclusive write lock on matching rows, blocking concurrent readers requesting `FOR UPDATE` and all writers.

* **`SELECT FOR SHARE`:** Acquires a shared read lock, preventing other transactions from updating/deleting the rows while allowing other concurrent shared readers.

* **`SELECT ... FOR UPDATE SKIP LOCKED`:** Bypasses any rows currently locked by other concurrent sessions and returns only unlocked candidate rows.

* **Message Queue Golden Standard:** `SKIP LOCKED` powers ultra-fast, distributed, rock-solid job queues in PostgreSQL/MySQL without worker contention.

#### Tricky Follow-up:
* **Question:** How does `SKIP LOCKED` prevent workers from competing over the same job queue task?

* **Answer:** When 10 worker threads run `SELECT * FROM queue WHERE status = 'PENDING' LIMIT 1 FOR UPDATE SKIP LOCKED`, each worker instantly grabs a unique unlocked row without waiting or blocking.


In [137]:
%%sql
CREATE TABLE worker_queue (
    task_id INT PRIMARY KEY,
    payload VARCHAR,
    is_locked BOOLEAN
);
INSERT INTO worker_queue VALUES
(1, 'Send_Email', TRUE),  -- Currently being worked by Worker 1
(2, 'Process_Video', FALSE), -- Available
(3, 'Generate_PDF', FALSE);  -- Available;

SELECT * 
FROM worker_queue 
WHERE is_locked = FALSE 
ORDER BY task_id 
LIMIT 1;

task_id,payload,is_locked
2,Process_Video,False


### Q137: Write Skew Anomaly in Snapshot Isolation

#### Verbal Answer:
* **Write Skew Definition:** An anomaly that occurs under Snapshot Isolation (Repeatable Read) where two concurrent transactions read overlapping data sets, make decisions based on what they read, and perform disjoint writes that violate an overall invariant.

* **Classic Medical On-Call Example:**
  - Rule: At least 1 doctor must remain on call at all times.
  - Doctors Alice and Bob are both currently on call.
  - Tx 1 checks if count >= 2, sees 2, and takes Alice off call.
  - Concurrent Tx 2 checks if count >= 2, sees 2 (under snapshot isolation), and takes Bob off call.
  - Both transactions commit successfully, leaving **zero doctors on call**!

* **Resolution:** Must use `SERIALIZABLE` isolation or explicit pessimistic row locking (`SELECT FOR UPDATE`).

#### Tricky Follow-up:
* **Question:** Why didn't standard row locking prevent write skew in the doctor example?

* **Answer:** Because Tx 1 modified Alice's row and Tx 2 modified Bob's row. Neither transaction modified the same physical row, so no row-level write lock conflict occurred.


In [138]:
%%sql
CREATE TABLE on_call_schedule (
    doctor_name VARCHAR PRIMARY KEY,
    is_on_call BOOLEAN
);
INSERT INTO on_call_schedule VALUES ('Alice', TRUE), ('Bob', TRUE);

SELECT 
    COUNT(*) AS total_active_doctors
FROM on_call_schedule
WHERE is_on_call = TRUE;

total_active_doctors
2


### Q138: Lock Granularity Escalation: Row Locks vs Page Locks vs Table Locks

#### Verbal Answer:
* **Lock Hierarchy:**
  - **Row Lock:** Granular; locks a single tuple. High concurrency, high lock manager memory overhead.
  - **Page Lock:** Locks a 4 KB - 8 KB disk block containing dozens of rows.
  - **Table Lock:** Locks the entire relation. Lowest concurrency, zero memory overhead.

* **Lock Escalation (SQL Server / Traditional RDBMS):** When a single query updates > 5,000 individual row locks, the engine automatically escalates the thousands of row locks into a single Exclusive Table Lock, instantly blocking all concurrent users.

#### Tricky Follow-up:
* **Question:** How can you prevent lock escalation during massive 1,000,000-row batch updates?

* **Answer:** Break the update into smaller micro-batches (e.g. 1,000 rows per transaction in a loop with small sleeps), committing each batch independently.


In [139]:
%%sql
CREATE TABLE batch_updates (id INT PRIMARY KEY, processed BOOLEAN);
INSERT INTO batch_updates SELECT i, FALSE FROM generate_series(1, 10) t(i);

UPDATE batch_updates SET processed = TRUE WHERE id <= 5;

SELECT * FROM batch_updates ORDER BY id;

id,processed
1,True
2,True
3,True
4,True
5,True
6,False
7,False
8,False
9,False
10,False


### Q139: Two-Phase Commit (2PC) Protocol in Distributed Transactions

#### Verbal Answer:
* **The Distributed Transaction Challenge:** Atomically committing a transaction that spans multiple independent physical databases (e.g., deducting inventory in Postgres and charging a card in Stripe).

* **Two Phases of 2PC:**
  1. **Prepare Phase:** The Coordinator node asks all participant nodes to execute the transaction up to the commit point and write undo/redo logs. Each participant votes `READY` or `ABORT`.
  2. **Commit Phase:** If all participants voted `READY`, Coordinator broadcasts `GLOBAL COMMIT`. If any participant voted `ABORT`, Coordinator broadcasts `GLOBAL ROLLBACK`.

* **Vulnerability:** If the Coordinator crashes permanently after Phase 1, participant nodes are left in doubt holding locks indefinitely (blocking protocol).

#### Tricky Follow-up:
* **Question:** What modern architectural pattern replaces 2PC in microservice architectures?

* **Answer:** The **Saga Pattern** using asynchronous event-driven choreographies and compensating transactions (e.g. if Payment fails, trigger a `Compensate_Cancel_Order` event).


In [140]:
%%sql
CREATE TABLE saga_events (
    tx_id INT,
    service VARCHAR,
    status VARCHAR
);
INSERT INTO saga_events VALUES
(101, 'Inventory_Service', 'RESERVED'),
(101, 'Payment_Service',   'CHARGED'),
(101, 'Notification_Svc',  'SENT');

SELECT 
    tx_id,
    COUNT(CASE WHEN status IN ('RESERVED', 'CHARGED', 'SENT') THEN 1 END) AS successful_steps
FROM saga_events
GROUP BY tx_id;

tx_id,successful_steps
101,3


### Q140: Non-Repeatable Reads vs Phantom Reads: The Exact Distinction

#### Verbal Answer:
* **Non-Repeatable Read (Updates/Deletes):** Transaction 1 queries Row 42. Transaction 2 updates Row 42 and commits. Transaction 1 re-reads Row 42 and discovers its **existing column values have changed**.

* **Phantom Read (Inserts):** Transaction 1 queries `WHERE salary > 50000`, receiving 10 rows. Transaction 2 inserts a **brand new employee** with salary 60,000 and commits. Transaction 1 re-runs the exact same query and receives 11 rows (the new "phantom" row appeared).

* **Prevention Mechanism:** Non-repeatable reads are prevented by locking existing rows. Phantom reads require **Predicate / Gap Locking** to lock the non-existent space between index keys.

#### Tricky Follow-up:
* **Question:** Why does Gap Locking exist in MySQL InnoDB at Repeatable Read isolation level?

* **Answer:** Gap locks lock the index range gaps between existing records (e.g. between 10 and 20), preventing other sessions from inserting new records into that range and eliminating phantom reads.


In [141]:
%%sql
CREATE TABLE salaries_phantom (emp_id INT, salary DOUBLE);
INSERT INTO salaries_phantom VALUES (1, 60000), (2, 75000);

SELECT COUNT(*) AS high_earners_count 
FROM salaries_phantom 
WHERE salary > 50000;

high_earners_count
2


### -------------------------------------------------------------------------
### PILLAR 15: MODERN CLOUD DATA WAREHOUSES (DUCKDB, SNOWFLAKE, BIGQUERY, SPARK SQL)
### -------------------------------------------------------------------------

### Q141: DuckDB Vectorized In-Memory Execution: Vector Volcano and Columnar SIMD

#### Verbal Answer:
* **The Traditional Volcano Iterator Bottleneck:** Traditional row engines (Postgres, SQLite) evaluate queries one row at a time via `next()` virtual function calls, incurring massive CPU branch mispredictions and L1 cache evictions.

* **Vectorized Execution (Vector Volcano / DuckDB / Snowflake):** Evaluates operations in chunks of vectors (typically 2,048 tuples at a time).

* **Hardware Acceleration (SIMD):** Columnar data vectors are kept contiguous in CPU L1/L2 caches, allowing hardware SIMD (Single Instruction, Multiple Data) CPU registers (AVX-512) to compute filters and math over 8-16 values in a single CPU cycle.

#### Tricky Follow-up:
* **Question:** Why can DuckDB query 100M rows on a local laptop faster than a distributed Spark cluster?

* **Answer:** Because DuckDB has zero JVM overhead, zero network shuffle serialization, and evaluates vectorized C++ SIMD kernels directly on raw in-memory column vectors.


In [142]:
%%sql
CREATE TABLE vec_benchmark AS 
SELECT i AS id, (i * 2.5) AS val 
FROM generate_series(1, 100000) t(i);

SELECT 
    COUNT(*) AS total_rows,
    SUM(val) AS sum_val,
    AVG(val) AS avg_val
FROM vec_benchmark
WHERE val > 50000;

total_rows,sum_val,avg_val
80000,1.200010e+10,150001.25


### Q142: Direct Parquet and S3 / Remote Data Querying in SQL without Ingestion

#### Verbal Answer:
* **Zero-Copy Ingestion:** Modern analytical engines (DuckDB, Snowflake, Trino) query Parquet, CSV, and JSON files residing directly in Amazon S3, Azure Blob, or local disks via `read_parquet('s3://...')` without loading them into database tables.

* **Parquet Footer Reading:** The engine issues HTTP range requests to fetch only the Parquet metadata footer, identifies relevant column chunk offsets and row groups, and streams only the filtered byte offsets over the network.

#### Tricky Follow-up:
* **Question:** What happens if a remote Parquet file contains 1,000,000 rows but the query has `WHERE id = 5` and row groups have min/max zone maps?

* **Answer:** The engine reads only the small metadata footer and downloads just the single 64 KB column chunk containing id 5, skipping 99% of remote network bandwidth.


In [143]:
%%sql
COPY (
    SELECT i AS id, 'item_' || i AS name 
    FROM generate_series(1, 1000) t(i)
) TO 'scratch/sample_export.parquet' (FORMAT PARQUET);

SELECT * 
FROM read_parquet('scratch/sample_export.parquet') 
WHERE id <= 5;

id,name
1,item_1
2,item_2
3,item_3
4,item_4
5,item_5


### Q143: Snowflake Micro-Partitions and Automated Columnar Metadata

#### Verbal Answer:
* **Micro-Partitions:** Snowflake automatically partitions all tables into contiguous, immutable columnar units (50 MB to 500 MB uncompressed).

* **Zero DDL Partitioning:** Unlike legacy databases where engineers must declare `PARTITION BY (date)`, Snowflake handles all partitioning automatically.

* **Metadata Pruning:** Snowflake maintains rich metadata for every micro-partition (min/max values of every column, null counts, distinct counts). The Cloud Services coordination layer prunes out 95%+ of micro-partitions before spinning up virtual warehouse compute.

#### Tricky Follow-up:
* **Question:** What causes clustering degradation in Snowflake over time?

* **Answer:** Frequent random single-row `INSERT` and `UPDATE` operations create overlapping min/max ranges across micro-partitions. Running `AUTOMATIC CLUSTERING` reorganizes micro-partitions to restore pruning efficiency.


In [144]:
%%sql
CREATE TABLE mock_partitions (
    partition_id INT,
    min_date DATE,
    max_date DATE,
    row_count INT
);
INSERT INTO mock_partitions VALUES
(1, DATE '2024-01-01', DATE '2024-01-10', 100000),
(2, DATE '2024-01-11', DATE '2024-01-20', 100000),
(3, DATE '2024-01-21', DATE '2024-01-31', 100000);

SELECT partition_id, min_date, max_date
FROM mock_partitions
WHERE DATE '2024-01-15' BETWEEN min_date AND max_date;

partition_id,min_date,max_date
2,2024-01-11,2024-01-20


### Q144: Time Travel and Zero-Copy Cloning in Modern Cloud Warehouses

#### Verbal Answer:
* **Time Travel (`AT` / `BEFORE`):** Access historic snapshots of a table at an exact timestamp or query ID (`SELECT * FROM tbl AT(TIMESTAMP => '2024-01-01 12:00:00')`). Enabled by immutable micro-partitions and snapshot metadata.

* **Zero-Copy Cloning (`CREATE TABLE clone_tbl CLONE prod_tbl`):**
  - Creates a new metadata pointer referencing the existing immutable micro-partitions of the source table.
  - Takes 2 seconds and consumes **zero bytes of storage**.
  - As new writes occur on the clone, new micro-partitions are written (Copy-on-Write).

#### Tricky Follow-up:
* **Question:** How can Time Travel instantly restore a table dropped by an accidental `DROP TABLE` command?

* **Answer:** Execute `UNDROP TABLE table_name;` within the retention period (e.g. 1 to 90 days), which immediately restores the table metadata pointer from the catalog.


In [145]:
%%sql
CREATE TABLE table_snapshots (
    snapshot_id INT,
    snapshot_ts TIMESTAMP,
    table_state_desc VARCHAR
);
INSERT INTO table_snapshots VALUES
(1, TIMESTAMP '2024-01-01 10:00:00', 'Pre-Migration Baseline'),
(2, TIMESTAMP '2024-01-01 11:00:00', 'Post-ETL Sync');

SELECT * FROM table_snapshots;

snapshot_id,snapshot_ts,table_state_desc
1,2024-01-01 10:00:00,Pre-Migration Baseline
2,2024-01-01 11:00:00,Post-ETL Sync


### Q145: BigQuery Slots, Shuffling Architecture, and Cost-Per-Query Optimization

#### Verbal Answer:
* **BigQuery Slots:** Virtual CPU units dedicated to executing SQL tasks. BigQuery dynamically allocates and scales slots across queries.

* **Dremel Execution Engine:** Separates compute from Colossus distributed storage via Jupiter multi-terabit network fabric.

* **On-Demand Pricing Optimization Rules:**
  - BigQuery charges strictly per **bytes scanned** ($6.25 per TB).
  - Never run `SELECT *`.
  - Always partition on date/timestamp and filter with partition predicates.
  - Cluster frequently filtered columns to maximize block pruning.

#### Tricky Follow-up:
* **Question:** Why does `LIMIT 10` on an unpartitioned 500 GB table still bill for the entire 500 GB in BigQuery?

* **Answer:** Because BigQuery scans entire columns across all distributed storage blocks to assemble tuples before applying the `LIMIT` slice in memory.


In [146]:
%%sql
CREATE TABLE query_audit (
    query_text VARCHAR,
    columns_scanned INT,
    total_bytes_mb DOUBLE
);
INSERT INTO query_audit VALUES
('SELECT * FROM tbl', 50, 50000.0),
('SELECT id, metric FROM tbl WHERE dt = today', 2, 250.0);

SELECT * FROM query_audit;

query_text,columns_scanned,total_bytes_mb
SELECT * FROM tbl,50,50000.0
"SELECT id, metric FROM tbl WHERE dt = today",2,250.0


### Q146: Open Table Formats: Apache Iceberg vs Delta Lake vs Apache Hudi

#### Verbal Answer:
* **The Open Lakehouse Standard:** Decouples the storage engine and data catalog from proprietary cloud vendors, storing data as open Parquet files on S3/GCS.

* **Key Architectural Capabilities:**
  - Full ACID transaction guarantees on object storage.
  - Schema Evolution (add/rename/drop columns safely without rewriting data).
  - Hidden Partitioning (partition evolution without breaking downstream queries).
  - Time travel and snapshot isolation.

* **Metadata Layer:** Maintains hierarchical snapshot manifests tracking exactly which Parquet files constitute the current table version.

#### Tricky Follow-up:
* **Question:** What is 'Hidden Partitioning' in Apache Iceberg?

* **Answer:** Users query `WHERE event_time >= '2024-01-01'` directly without needing to know or specify the underlying physical partition transform (e.g. `month(event_time)`), eliminating query bugs.


In [147]:
%%sql
CREATE TABLE iceberg_manifest (
    snapshot_id BIGINT,
    data_file VARCHAR,
    partition_spec VARCHAR,
    record_count BIGINT
);
INSERT INTO iceberg_manifest VALUES
(1001, 's3://lake/data_001.parquet', 'event_date=2024-01-01', 50000),
(1002, 's3://lake/data_002.parquet', 'event_date=2024-01-02', 45000);

SELECT * FROM iceberg_manifest;

snapshot_id,data_file,partition_spec,record_count
1001,s3://lake/data_001.parquet,event_date=2024-01-01,50000
1002,s3://lake/data_002.parquet,event_date=2024-01-02,45000


### Q147: dbt (data build tool): Modular SQL Pipelines, Ref Functions, and Lineage DAGs

#### Verbal Answer:
* **The dbt Philosophy:** "Transform data in the warehouse using modular SQL `SELECT` statements and software engineering best practices."

* **The `{{ ref('model_name') }}` Function:**
  - Dynamically interpolates the correct environment database schema (e.g. `analytics_dev` vs `analytics_prod`).
  - Automatically infers dependencies and constructs the full Directed Acyclic Graph (DAG) for parallel execution.

* **Materializations:** Models declare how they compile: `view`, `table`, `incremental`, or `ephemeral`.

#### Tricky Follow-up:
* **Question:** How does dbt implement incremental models without full table rewrites?

* **Answer:** Via the `is_incremental()` macro: `{% if is_incremental() %} WHERE updated_at > (select max(updated_at) from {{ this }}) {% endif %}`, which compiles into an atomic `MERGE INTO` statement.


In [148]:
%%sql
CREATE TABLE stg_orders AS SELECT 1 AS order_id, 101 AS user_id, 150.0 AS amount;
CREATE TABLE dim_users  AS SELECT 101 AS user_id, 'Enterprise' AS tier;

SELECT 
    o.order_id,
    o.amount,
    u.tier
FROM stg_orders o
JOIN dim_users u ON o.user_id = u.user_id;

order_id,amount,tier
1,150.0,Enterprise


### Q148: Dynamic Data Masking, Row-Level Security (RLS), and RBAC Governance

#### Verbal Answer:
* **Role-Based Access Control (RBAC):** Users inherit privileges through assigned roles (`ANALYST`, `DATA_ENGINEER`, `C_SUITE`).

* **Dynamic Data Masking:** Masks sensitive columns on-the-fly based on the querying user's role (e.g., `ANALYST` sees `'XXXX-XXXX-XXXX-1234'`, `FINANCE` sees `'4111-2222-3333-1234'`) without duplicating data.

* **Row-Level Security (RLS):** Automatically injects a filter predicate into every query based on user identity (e.g., a regional sales manager can only see rows `WHERE region = CURRENT_USER_REGION()`).

#### Tricky Follow-up:
* **Question:** Why is dynamic data masking superior to creating separate sanitized tables for analysts?

* **Answer:** It eliminates redundant ETL pipelines, eliminates duplicate storage costs, and prevents data leakage when source tables update.


In [149]:
%%sql
CREATE TABLE customer_pii (
    cust_id INT,
    email VARCHAR,
    ssn VARCHAR
);
INSERT INTO customer_pii VALUES
(1, 'alice@co.com', '123-45-6789'),
(2, 'bob@co.com',   '987-65-4321');

SELECT 
    cust_id,
    -- Dynamic email masking
    CONCAT(LEFT(email, 2), '***@***', RIGHT(email, 4)) AS masked_email,
    -- Dynamic SSN masking
    CONCAT('XXX-XX-', RIGHT(ssn, 4)) AS masked_ssn
FROM customer_pii;

cust_id,masked_email,masked_ssn
1,al***@***.com,XXX-XX-6789
2,bo***@***.com,XXX-XX-4321


### Q149: Continuous Ingestion: Snowflake Snowpipe, Streaming Tables, and Micro-Batching

#### Verbal Answer:
* **Snowpipe:** Serverless event-driven continuous data ingestion service. When new files land in S3/GCS, cloud storage notifications (SNS/SQS) trigger Snowpipe to automatically load files within seconds.

* **Micro-Batching:** Ingestion files are batched in 1-minute intervals, providing near-real-time streaming latency with columnar batch write throughput.

* **Streaming Tables (Databricks Delta Live Tables):** Declarative SQL pipelines that continuously process streaming change feeds with automatic checkpointing and state tracking.

#### Tricky Follow-up:
* **Question:** How does Snowpipe ensure that an S3 file is never ingested twice even if the notification fires multiple times?

* **Answer:** Snowpipe maintains 14 days of ingestion history metadata (file path and checksum). Duplicate notifications for already-loaded files are automatically skipped.


In [150]:
%%sql
CREATE TABLE snowpipe_ingest_log (
    file_name VARCHAR PRIMARY KEY,
    ingested_at TIMESTAMP,
    status VARCHAR
);
INSERT INTO snowpipe_ingest_log VALUES
('s3://bucket/20240101_1000.json.gz', CURRENT_TIMESTAMP, 'LOADED'),
('s3://bucket/20240101_1001.json.gz', CURRENT_TIMESTAMP, 'LOADED');

INSERT OR IGNORE INTO snowpipe_ingest_log VALUES
('s3://bucket/20240101_1000.json.gz', CURRENT_TIMESTAMP, 'DUPLICATE_IGNORED');

SELECT * FROM snowpipe_ingest_log;

file_name,ingested_at,status
s3://bucket/20240101_1000.json.gz,2026-10-02 15:44:58.949918,LOADED
s3://bucket/20240101_1001.json.gz,2026-10-02 15:44:58.949918,LOADED


### Q150: Semantic Layers, Metric Trees, and SQL as the Universal Analytics Engine

#### Verbal Answer:
* **The Metric Drift Problem:** Finance defines "Active Customer" as placing an order in 30 days; Marketing defines it as opening an email in 60 days. Queries across teams return contradictory numbers.

* **The Semantic Layer (Cube.js, dbt Semantic Layer, Looker LookML):** A centralized abstraction layer on top of the data warehouse that centrally defines dimensions, metrics, metric trees, and join paths.

* **SQL as the Universal API:** Business Intelligence tools (Tableau, PowerBI), Python notebooks, and AI LLM agents query the semantic layer using standard SQL, guaranteeing consistent, single-source-of-truth metrics across the entire enterprise.

#### Tricky Follow-up:
* **Question:** How does a Semantic Layer handle fan-out join issues when calculating metrics across mixed grains?

* **Answer:** The semantic compiler automatically splits calculations into sub-queries aggregated at the correct dimensional grain before joining the results, eliminating Cartesian multiplication errors.


In [151]:
%%sql
CREATE TABLE semantic_orders (user_id INT, amount DOUBLE, is_active BOOLEAN);
INSERT INTO semantic_orders VALUES (1, 100.0, TRUE), (2, 250.0, TRUE), (3, 50.0, FALSE);

SELECT 
    -- Metric: Governed Active User ARR
    SUM(amount) FILTER (WHERE is_active = TRUE) AS governed_active_arr,
    -- Metric: Total Count of Active Accounts
    COUNT(*) FILTER (WHERE is_active = TRUE) AS active_accounts_count,
    -- Metric: ARPU (Average Revenue Per Active User)
    ROUND(SUM(amount) FILTER (WHERE is_active = TRUE) / 
          NULLIF(COUNT(*) FILTER (WHERE is_active = TRUE), 0), 2) AS arpu
FROM semantic_orders;

governed_active_arr,active_accounts_count,arpu
350.0,2,175.0
